# RiceGuardian AI v5 — Train → TFLite On-Device (Kaggle)

Pipeline (v5, 2026-10-09, tài khoản `anhnguynthvan`):
1. **YOLO11s** @640 — detector vẽ box quanh vùng bệnh, tên trên box = lớp của detector (`DET_DISPLAY_NAMES`)
2. **EfficientNetB3** @300 — classifier 8 lớp trên cả ảnh, augmentation `field-v1` → **TFLite INT8**

```
Train: Ảnh → YOLO crop-aware → EfficientNetB3 (GATE: acc ≥ 0.85, macro-F1 ≥ 0.80, recall BLB/Blast/Brown Spot ≥ 0.75)
App:   Camera → classifier (TTA) → 1 bệnh + % (output_rule top1_exclusive) + detector TFLite → box + tên bệnh
```

Quy tắc hiển thị (`pipeline_config.json` → `output_rule`): chỉ lớp cao nhất giữ % (trung bình softmax qua TTA), mọi lớp
khác = 0%. Lớp cao nhất là bệnh → Khỏe mạnh = 0%; lớp cao nhất là Khỏe mạnh → mọi bệnh = 0% và không vẽ box.
% là độ tin cậy của model về loại bệnh, **không** phải diện tích vùng bệnh.

## 3 lần chạy Kaggle (`TRAINING_GROUP`)

| Group | Việc | Upload dataset sau khi xong |
|-------|------|------------------------------|
| `'0A'` | Data audit + label audit + train YOLO11s | `anhnguynthvan/session-train-0a-v5` |
| `'B_B2'` | Classifier EffB3 + B2 keep-best | `anhnguynthvan/session-train-b-v5` |
| `'C'` | Gate + TFLite classifier + TFLite detector | `anhnguynthvan/rice-v3-mobile-release` |

**Mặc định Config:** `TRAINING_GROUP = '0A'`, `ALLOW_C_WITHOUT_GATE = False`. Chỉ chạy `'C'` khi B có `b_ready_for_c`.
Nguyên tắc: `docs/principles/RICE_CLASSIFIER_TRAINING_PRINCIPLES.md`.

**Chống mất output:** export zip → Save Version → download Output → upload Kaggle Dataset → mới đóng session.

### Lần 1 — `TRAINING_GROUP = '0A'`
Add Data: `anhnguynthvan/dataset-rice-leaf` + `yusufmurtaza01/rice-leaf-diseases` (Internet ON để tải `yolo11s.pt`)

### Lần 2 — `TRAINING_GROUP = 'B_B2'`
Add Data: `anhnguynthvan/dataset-rice-leaf` + `anhnguynthvan/dataset-train-rice-mobile` + `anhnguynthvan/session-train-0a-v5` (Internet ON)

### Lần 3 — `TRAINING_GROUP = 'C'`
Add Data: `anhnguynthvan/dataset-rice-leaf` + `anhnguynthvan/session-train-b-v5` + `yusufmurtaza01/rice-leaf-diseases` (Internet ON)


## 00. Runtime guard (import torch first)

Imports torch in this kernel before kagglehub / pip. The PID must change after stopping and restarting the session;
a failed torch import cannot be retried in the same kernel.


In [ ]:
import os
import sys
import traceback
from importlib.metadata import PackageNotFoundError, version as _pkg_version


def _installed(name):
    try:
        return _pkg_version(name)
    except PackageNotFoundError:
        return None


print('Kernel PID:', os.getpid())
print('Python:', sys.version.replace('\n', ' '))
print('sys.gettrace():', sys.gettrace())

_STOP_SESSION = (
    'Stop the session (power icon), start a new session, then Run All. '
    'Restart kernel is not enough if the PID above does not change.'
)
_torch_mod = sys.modules.get('torch')
if _torch_mod is not None and not hasattr(_torch_mod, '__version__'):
    raise RuntimeError(f'This kernel already holds a half-finished torch import. {_STOP_SESSION}')

try:
    import torch
except Exception as e:
    print('torch import FAILED in the kernel:', f'{type(e).__name__}: {e}')
    print('Installed versions:', {n: _installed(n) for n in ('torch', 'torchvision', 'numpy', 'typing_extensions')})
    print('sys.gettrace():', sys.gettrace())
    traceback.print_exc()
    raise RuntimeError(f'torch does not import in this kernel. {_STOP_SESSION} Send the full output of this cell.') from e

print(f'torch {torch.__version__} cuda {torch.cuda.is_available()} | file {torch.__file__}')


## 0. Download YOLO dataset (kagglehub)

`yusufmurtaza01/rice-leaf-diseases` (detector mAP50 / parity / INT8 calibration in 0A and C). Needs Settings → Internet ON.
Skipped when the dataset is already mounted via Add Data. Runs before Config because DET_ROOT is resolved there.


In [ ]:
import os

DOWNLOAD_DET_DATASET = True  # B_B2 does not need it; 0A and C do
DET_KAGGLEHUB_SLUG = 'yusufmurtaza01/rice-leaf-diseases'
DET_KAGGLEHUB_PATH = None
_mounted = [
    p for p in ('/kaggle/input/rice-leaf-diseases', '/kaggle/input/datasets/yusufmurtaza01/rice-leaf-diseases')
    if os.path.isdir(p)
]
if not DOWNLOAD_DET_DATASET:
    print('Skip kagglehub download (DOWNLOAD_DET_DATASET = False)')
elif _mounted:
    print('YOLO dataset already mounted:', _mounted)
else:
    try:
        import kagglehub
        DET_KAGGLEHUB_PATH = kagglehub.dataset_download(DET_KAGGLEHUB_SLUG)
        print('Path to dataset files:', DET_KAGGLEHUB_PATH)
    except Exception as e:
        print(f'WARNING: kagglehub download of {DET_KAGGLEHUB_SLUG} failed ({type(e).__name__}: {e}). '
              'Settings -> Internet ON, or Add Data the dataset.')


## 1. Config v3


In [ ]:
import os
from pathlib import Path

KAGGLE_USER = 'anhnguynthvan'
DATASET_RICE_TRAIN_CANDIDATES = [
    f'/kaggle/input/datasets/{KAGGLE_USER}/dataset-rice-leaf',
    '/kaggle/input/datasets/anhhoang3003/dataset-rice-train',
]
DATASET_RICE_TRAIN = next(
    (p for p in DATASET_RICE_TRAIN_CANDIDATES if os.path.isdir(p)), DATASET_RICE_TRAIN_CANDIDATES[0]
)

CLS_ROOT_CANDIDATES = [
    f'{DATASET_RICE_TRAIN}/Rice Leaf Bacterial and Fungal Disease Dataset/Rice leaf disease/Rice leaf disease',
    f'{DATASET_RICE_TRAIN}/Rice Leaf Bacterial and Fungal Disease Dataset/Rice leaf disease',
    f'{DATASET_RICE_TRAIN}/Rice Leaf Bacterial and Fungal Disease Dataset',
]

CLS_EXTRA_CANDIDATES = [
    f'{DATASET_RICE_TRAIN}/Rice leaf disease/Training data',
    f'{DATASET_RICE_TRAIN}/Rice leaf disease',
]
MERGE_EXTRA_INTO_TRAIN = True
MERGE_WEAK_CLASSES_ONLY = True
MERGE_WEAK_CLASSES = [
    'Brown Spot', 'Narrow Brown Leaf Spot',
    'Leaf scald', 'Leaf Blast', 'Bacterial Leaf Blight',
]
MERGED_TRAIN_DIR = '/kaggle/working/merged_training_data'

USE_EXTRA_V1 = True
CLS_EXTRA_V1_ROOTS = [
    f'/kaggle/input/datasets/{KAGGLE_USER}/dataset-train-rice-mobile',
    f'/kaggle/input/datasets/{KAGGLE_USER}/rice-extra-v1',
    '/kaggle/input/rice-extra-v1',
]
# Order = closest domain to Mehedi first, so dedup keeps the closer-domain copy.
EXTRA_V1_SOURCES = ['dhan_shomadhan', 'roboflow_rld', 'yusuf_crops', 'paddy_doctor']
WEAK_CLASS_TARGET = 600
LEAK_WARN_RATIO = 0.30
MIN_WEAK_CLASS_TRAIN = 400

OPTIONAL_IMAGE_ROOTS = [
    '/kaggle/input/notebooks/azizahaque/rice-leaf',
    '/kaggle/input/notebooks/sahilr05/rice-leaf-diseases-tensorflow',
    '/kaggle/input/notebooks/maimunulkjisan/rice-leaf-disease-classification-using-cnn',
    '/kaggle/input/notebooks/stpeteishii/leaf-rice-disease-classify-densenet201',
    '/kaggle/input/notebooks/ohagwucollinspatrick/rice-leaf-disease-detection-w-cnn-kera',
]
MERGE_OPTIONAL_NOTEBOOKS = False
BLOCKED_CLASS_KEYS = {'tungro', 'leafsmut', 'leaf smut', 'smut'}

DET_ROOT_CANDIDATES = [
    # Kaggle Add Data mount variants (slug-only OR datasets/user/slug)
    '/kaggle/input/rice-leaf-diseases/rice',
    '/kaggle/input/rice-leaf-diseases',
    '/kaggle/input/datasets/yusufmurtaza01/rice-leaf-diseases/rice',
    '/kaggle/input/datasets/yusufmurtaza01/rice-leaf-diseases',
    '/kaggle/input/yusufmurtaza01-rice-leaf-diseases/rice',
    '/kaggle/input/yusufmurtaza01-rice-leaf-diseases',
]
if globals().get('DET_KAGGLEHUB_PATH'):
    DET_ROOT_CANDIDATES += [os.path.join(DET_KAGGLEHUB_PATH, 'rice'), DET_KAGGLEHUB_PATH]

WORK_DIR = '/kaggle/working'

# '0A' | 'B_B2' | 'C'  — v5 (2026-10-09): 0A (YOLO11s) -> B_B2 (EffB3 @300, augment field-v1) -> C (TFLite).
# Run C only when B b_ready_for_c is true.
TRAINING_GROUP = 'C'
# Trial override: lets C export from a checkpoint that failed GATE. Must stay False for a release.
# v5 trial (2026-10-10): session-train-b-v5 = Run 5, GATE FAIL; upload only as rice-v3-mobile-trial.
ALLOW_C_WITHOUT_GATE = True
# Session C exports yolo_best.pt to TFLite. 'auto' ships int8 only when cell 16a prints "INT8 recommended: yes"
# (mAP50 drop and small-box recall vs .pt), else float32. Never float16.
EXPORT_DET_TFLITE = True
DET_TFLITE_SHIP = 'auto'
# Missing yusufmurtaza01/rice-leaf-diseases is an error: this run must report mAP50 / parity of the TFLite file.
DET_EVAL_REQUIRED = True
# Calibrating on all ~1730 val images OOMs Kaggle; INT8 calibrates on a seeded random subset of val.
EXPORT_DET_INT8 = True
DET_INT8_CALIB_IMAGES = 300
DET_INT8_MAX_MAP50_DROP = 0.02
# 'pr_f1': detector conf_thresh = max-F1 conf of the shipped file's PR curve; 'pipeline': cell 12.5 BEST_CONF.
DET_CONF_SOURCE = 'pr_f1'
DET_MIN_RECALL = 0.80
# Detector class -> classifier class. None = no classifier class: the box is still cropped and the classifier labels it.
DET_CLASS_MAP = {
    'Rice__BacterialLeafBlight': 'Bacterial Leaf Blight',
    'Rice__BrownSpot': 'Brown Spot',
    'Rice__Healthy': 'Healthy Rice Leaf',
    'Rice__Hispa': 'Rice Hispa',
    'Rice__LeafBlast': 'Leaf Blast',
    'Rice__LeafScald': 'Leaf scald',
    'Rice__LeafSmut': None,
    'Rice__NarrowBrownLeafSpot': 'Narrow Brown Leaf Spot',
    'Rice__NeckBlast': None,
}
# Label audit (cell 16a-0): a box covering more than DET_WHOLE_LEAF_AREA of the image counts as a whole-leaf box.
DET_WHOLE_LEAF_AREA = 0.6
DET_LESION_MEDIAN_MAX = 0.25
DET_WHOLE_LEAF_MAX_SHARE = 0.20
DET_PARITY_IMAGES = 100
DET_SMALL_BOX_AREA = 0.03
DET_NMS_IOU = 0.7
# Box label = the detector's own class; None = box not drawn.
DET_DISPLAY_NAMES = {
    'Rice__BacterialLeafBlight': 'Bacterial Leaf Blight',
    'Rice__BrownSpot': 'Brown Spot',
    'Rice__Healthy': None,
    'Rice__Hispa': 'Rice Hispa',
    'Rice__LeafBlast': 'Leaf Blast',
    'Rice__LeafScald': 'Leaf scald',
    'Rice__LeafSmut': 'Leaf Smut',
    'Rice__NarrowBrownLeafSpot': 'Narrow Brown Leaf Spot',
    'Rice__NeckBlast': 'Neck Blast',
}
DET_DEMO_CONF = 0.25
HEALTHY_CLASS = 'Healthy Rice Leaf'
# Only the top class keeps its probability (TTA-mean softmax); every other class, Healthy included, shows 0%.
OUTPUT_RULE = {
    'mode': 'top1_exclusive',
    'healthy_class': HEALTHY_CLASS,
    'percent_source': 'softmax_tta_mean',
    'hide_boxes_when_healthy': True,
}
TRAINING_SESSION = 'A' if TRAINING_GROUP == '0A' else 'B'  # legacy alias

SESSION_A_UPLOAD_SLUG = f'{KAGGLE_USER}/session-train-0a-v5'
SESSION_B_UPLOAD_SLUG = f'{KAGGLE_USER}/session-train-b-v5'
SESSION_A_INPUT_CANDIDATES = [
    f'/kaggle/input/datasets/{SESSION_A_UPLOAD_SLUG}',
    '/kaggle/input/session-train-0a-v5',
    # User upload slug after Group 0A
    f'/kaggle/input/datasets/{KAGGLE_USER}/session-train-0a',
    f'/kaggle/input/session-train-0a',
    # Actual upload from Group 0A (user dataset)
    f'/kaggle/input/datasets/{KAGGLE_USER}/output-session-0a',
    f'/kaggle/input/output-session-0a',
    f'/kaggle/input/datasets/{KAGGLE_USER}/rice-v3-session-a-checkpoint',
    f'/kaggle/input/datasets/{KAGGLE_USER}/session-a-checkpoint',
    '/kaggle/input/rice-v3-session-a-checkpoint',
    '/kaggle/input/session-a-checkpoint',
]
SESSION_B_INPUT_CANDIDATES = [
    f'/kaggle/input/datasets/{SESSION_B_UPLOAD_SLUG}',
    '/kaggle/input/session-train-b-v5',
    f'/kaggle/input/datasets/{KAGGLE_USER}/session-train-b-v4',
    f'/kaggle/input/session-train-b-v4',
    # Account anhnguynthvan / current Session C checkpoint
    f'/kaggle/input/datasets/{KAGGLE_USER}/session-train-b',
    f'/kaggle/input/session-train-b',
    f'/kaggle/input/datasets/{KAGGLE_USER}/session-train-b-v1',
    f'/kaggle/input/session-train-b-v1',
    f'/kaggle/input/datasets/{KAGGLE_USER}/rice-v3-session-b-checkpoint',
    '/kaggle/input/rice-v3-session-b-checkpoint',
]
EXTERNAL_VAL_CANDIDATES = [
    f'/kaggle/input/datasets/{KAGGLE_USER}/external-validation',
    f'/kaggle/input/datasets/{KAGGLE_USER}/rice-leaf-external-holdout',
    '/kaggle/input/external-validation',
]

SESSION_A_ZIP_NAME = 'session_a_checkpoint.zip'
SESSION_B_ZIP_NAME = 'session_b_checkpoint.zip'
MOBILE_ZIP_NAME = 'rice_pipeline_v3_mobile_trial.zip' if ALLOW_C_WITHOUT_GATE else 'rice_pipeline_v3_mobile.zip'
SESSION_A_EXPORT_DIR = os.path.join(WORK_DIR, 'session_a_export')
SESSION_B_EXPORT_DIR = os.path.join(WORK_DIR, 'session_b_export')
SESSION_A_ZIP_PATH = os.path.join(WORK_DIR, SESSION_A_ZIP_NAME)
SESSION_B_ZIP_PATH = os.path.join(WORK_DIR, SESSION_B_ZIP_NAME)
MOBILE_ZIP_PATH = os.path.join(WORK_DIR, MOBILE_ZIP_NAME)
DATA_AUDIT_PATH = os.path.join(WORK_DIR, 'data_audit.json')

def resolve_first_existing(candidates):
    for path in candidates:
        if os.path.isdir(path):
            return path
    return candidates[0]


def _looks_like_yolo_root(root):
    """Evidence-based: YOLO layout needs images/train + labels/train."""
    if not root or not os.path.isdir(root):
        return False
    return any(
        os.path.isdir(os.path.join(root, *img)) and os.path.isdir(os.path.join(root, *lbl))
        for img, lbl in ((('images', 'train'), ('labels', 'train')), (('train', 'images'), ('train', 'labels')))
    )


def discover_yolo_root(input_root='/kaggle/input', max_depth=6):
    """Scan mounted Kaggle inputs for YOLO dirs. Returns None if not found."""
    if not os.path.isdir(input_root):
        return None
    best = None
    # prefer complete train+val layout
    for dirpath, dirnames, _ in os.walk(input_root):
        rel = os.path.relpath(dirpath, input_root)
        depth = 0 if rel == '.' else rel.count(os.sep) + 1
        if depth > max_depth:
            dirnames[:] = []
            continue
        if not _looks_like_yolo_root(dirpath):
            continue
        has_val = any(
            os.path.isdir(os.path.join(dirpath, sub))
            for sub in ('images/val', 'images/valid', 'valid/images', 'val/images')
        )
        if has_val:
            return dirpath
        if best is None:
            best = dirpath
    return best


def list_kaggle_inputs(input_root='/kaggle/input'):
    """Return actual mounted input names (for error messages — not guessed)."""
    if not os.path.isdir(input_root):
        return []
    out = []
    try:
        for name in sorted(os.listdir(input_root)):
            p = os.path.join(input_root, name)
            out.append(name + ('/' if os.path.isdir(p) else ''))
            # one level under datasets/
            if name == 'datasets' and os.path.isdir(p):
                for u in sorted(os.listdir(p))[:30]:
                    up = os.path.join(p, u)
                    if os.path.isdir(up):
                        out.append(f'datasets/{u}/')
                        for d in sorted(os.listdir(up))[:20]:
                            out.append(f'datasets/{u}/{d}')
    except OSError as e:
        out.append(f'<listdir error: {e}>')
    return out


CLS_ROOT = resolve_first_existing(CLS_ROOT_CANDIDATES)
CLS_EXTRA = resolve_first_existing(CLS_EXTRA_CANDIDATES)
CLS_EXTRA_V1 = resolve_first_existing(CLS_EXTRA_V1_ROOTS)
EXTERNAL_VAL_ROOT = resolve_first_existing(EXTERNAL_VAL_CANDIDATES)

# DET: prefer candidate that has YOLO layout; else scan /kaggle/input
DET_ROOT = None
for cand in DET_ROOT_CANDIDATES:
    if _looks_like_yolo_root(cand):
        DET_ROOT = cand
        break
if DET_ROOT is None:
    for cand in DET_ROOT_CANDIDATES:
        if os.path.isdir(cand):
            # maybe rice/ nested one level
            nested = os.path.join(cand, 'rice')
            if _looks_like_yolo_root(nested):
                DET_ROOT = nested
                break
            DET_ROOT = cand  # exists but layout unknown — keep for audit message
            break
if DET_ROOT is None or not _looks_like_yolo_root(DET_ROOT):
    found = discover_yolo_root('/kaggle/input') or (
        discover_yolo_root(DET_KAGGLEHUB_PATH) if globals().get('DET_KAGGLEHUB_PATH') else None
    )
    if found:
        print('Auto-discovered YOLO DET_ROOT:', found)
        DET_ROOT = found
if DET_ROOT is None:
    DET_ROOT = DET_ROOT_CANDIDATES[0]

PIPELINE_DIR = os.path.join(WORK_DIR, 'rice_pipeline_v3')
REPORT_PATH = os.path.join(WORK_DIR, 'training_report.json')

CANONICAL_CLASSES = [
    'Bacterial Leaf Blight', 'Brown Spot', 'Healthy Rice Leaf', 'Leaf Blast',
    'Leaf scald', 'Narrow Brown Leaf Spot', 'Rice Hispa', 'Sheath Blight',
]

CLASS_NAME_MAP = {
    'bacterial leaf blight': 'Bacterial Leaf Blight',
    'bacterialleafblight': 'Bacterial Leaf Blight',
    'bacterial_leaf_blight': 'Bacterial Leaf Blight',
    'bacterialblight': 'Bacterial Leaf Blight',
    'bacterial blight': 'Bacterial Leaf Blight',
    'brown spot': 'Brown Spot',
    'brownspot': 'Brown Spot',
    'brown_spot': 'Brown Spot',
    'healthy rice leaf': 'Healthy Rice Leaf',
    'healthy': 'Healthy Rice Leaf',
    'healthy_rice_leaf': 'Healthy Rice Leaf',
    'leaf blast': 'Leaf Blast',
    'leafblast': 'Leaf Blast',
    'leaf_blast': 'Leaf Blast',
    'blast': 'Leaf Blast',
    'leaf scald': 'Leaf scald',
    'leafscald': 'Leaf scald',
    'leaf_scald': 'Leaf scald',
    'narrow brown leaf spot': 'Narrow Brown Leaf Spot',
    'narrowbrownleafspot': 'Narrow Brown Leaf Spot',
    'narrow_brown_leaf_spot': 'Narrow Brown Leaf Spot',
    'narrow brown spot': 'Narrow Brown Leaf Spot',
    'rice hispa': 'Rice Hispa',
    'ricehispa': 'Rice Hispa',
    'rice_hispa': 'Rice Hispa',
    'hispa': 'Rice Hispa',
    'sheath blight': 'Sheath Blight',
    'sheathblight': 'Sheath Blight',
    'sheath_blight': 'Sheath Blight',
}

WEAK_CLASSES_OVERSAMPLE = [
    'Brown Spot', 'Narrow Brown Leaf Spot', 'Bacterial Leaf Blight', 'Leaf Blast', 'Leaf scald',
]
WEAK_CLASS_FACTORS = {
    'Brown Spot': 6, 'Narrow Brown Leaf Spot': 6, 'Leaf Blast': 4,
    'Bacterial Leaf Blight': 4, 'Leaf scald': 5,
}
HARD_PAIR_CLASS_FACTORS = {
    'Brown Spot': 3, 'Narrow Brown Leaf Spot': 3, 'Leaf scald': 2,
    'Leaf Blast': 2, 'Bacterial Leaf Blight': 2,
}
OVERSAMPLE_FACTOR = 4
BALANCED_SAMPLER_MAX_FACTOR = 3
DEDUP_HAMMING = 4

USE_CROP_AWARE_TRAIN = True
CROP_AWARE_RATIO = 0.25
CROP_AWARE_CONF = 0.25
CROP_AWARE_PADDING = 0.15
CROP_AWARE_MIN_AREA = 0.15

YOLO_MODEL = 'yolo11s.pt'
DET_ARCH = Path(YOLO_MODEL).stem
DET_IMG_SIZE = 640
DET_EPOCHS = 120
DET_BATCH = 16
DET_PATIENCE = 25
DET_AUGMENT = {
    'mosaic': 1.0, 'close_mosaic': 10, 'fliplr': 0.5, 'flipud': 0.5, 'degrees': 10.0,
    'scale': 0.5, 'translate': 0.1, 'hsv_h': 0.015, 'hsv_s': 0.6, 'hsv_v': 0.4, 'mixup': 0.1,
}
YOLO_BEST_PATH = os.path.join(WORK_DIR, 'yolo_rice_detect', 'weights', 'best.pt')
# Resume 0A YOLO from a saved run (Add Data yolo-oa-partial). False = train from YOLO_MODEL.
DET_RESUME = False
DET_RESUME_SLUG = 'yolo-oa-partial'
DET_RESUME_CANDIDATES = [
    f'/kaggle/input/datasets/{KAGGLE_USER}/{DET_RESUME_SLUG}/yolo_0a_partial/weights/last.pt',
    f'/kaggle/input/{DET_RESUME_SLUG}/yolo_0a_partial/weights/last.pt',
]
DET_RESUME_GLOB = '/kaggle/input/**/yolo_0a_partial/weights/last.pt'

CLS_IMG_SIZE = (300, 300)
CLS_BATCH = 16
CLS_BACKBONES = ['efficientnetb3']
CLS_EPOCHS_P1 = 10
CLS_EPOCHS_P2 = 50
CLS_LR_P1 = 1e-3
CLS_LR_P2 = 1e-4
CLS_WARMUP_EPOCHS_P2 = 2
CLS_UNFREEZE_EFFNET = -100
CLS_DROPOUT = 0.5
CLS_LABEL_SMOOTHING = 0.1
CLS_WEIGHT_DECAY = 1e-4
CLS_EARLY_STOP = 12
CLS_CHECKPOINT_METRIC = 'val_macro_f1'
CLS_RESUME = False
USE_FOCAL_LOSS = True
FOCAL_GAMMA = 1.0
USE_CLASS_WEIGHT = False  # balanced sampler already compensates; do not stack
VAL_F1_EVERY_N_EPOCHS = 1
FREE_RAM_BEFORE_PHASE2 = True
QUICK_TRAIN = False
TARGET_ACCURACY = 0.85
TARGET_MACRO_F1 = 0.80
REJECT_THRESHOLD = 0.45
TEMPERATURE = 1.0
RUN_B2_FINETUNE = True
B2_EPOCHS = 20
B2_LR = CLS_LR_P2 / 10
USE_TTA_EVAL = True
TTA_VIEWS = ('orig', 'hflip', 'vflip')
GATE_USE_TTA = True
CLS_AUGMENT_VERSION = 'field-v1'
CLASSIFIER_BEST_PATH = os.path.join(WORK_DIR, 'classifier_best.keras')
TFLITE_PATH = os.path.join(WORK_DIR, 'rice_leaf_cls_int8.tflite')

THRESHOLD_GRID = {
    'conf': [0.10, 0.15, 0.20, 0.25],
    'padding': [0.10, 0.15, 0.20, 0.25],
}
MIN_CROP_SIZE = 32
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}

IS_0A = TRAINING_GROUP == '0A'
IS_BB2 = TRAINING_GROUP == 'B_B2'
IS_C = TRAINING_GROUP == 'C'
RUN_YOLO = IS_0A

training_report = {
    'data_audit': {}, 'detection': {}, 'classifier_leaderboard': [],
    'pipeline': {}, 'mobile': {}, 'training_group': TRAINING_GROUP,
}

os.makedirs(WORK_DIR, exist_ok=True)
os.makedirs(PIPELINE_DIR, exist_ok=True)
NOTEBOOK_PATCH_BASE = 'bb2-data-v1-img300-b3-c-trial-det-fix7'
NOTEBOOK_PATCH = 'v5-yolo11s-b3-aug'
NOTEBOOK_PATCH_RESUME = 'v5-yolo-resume'
NOTEBOOK_PATCH_BB2 = 'v5-bb2-start'
NOTEBOOK_PATCH_C = 'v5-trial-c'
# Kaggle kills a session at 12 h and the classifier cannot resume in another session: training epochs stop once
# CLS_TIME_BUDGET_H hours have passed since the session started, so GATE evaluation and export still run.
CLS_TIME_BUDGET_H = 10.5


def _session_t0():
    import time as _time
    now = _time.time()
    for pid, source in ((1, 'container_pid1'), (os.getpid(), 'kernel_process')):
        try:
            import psutil
            t0 = psutil.Process(pid).create_time()
        except Exception as e:
            print(f'WARNING: start time of {source} unavailable ({type(e).__name__}: {e})')
            continue
        if 0 <= now - t0 < 12 * 3600:
            return t0, source
    print('WARNING: session start unknown; the time budget counts from this Config cell')
    return now, 'config_cell'


def session_elapsed_h():
    import time as _time
    return (_time.time() - SESSION_T0) / 3600


SESSION_T0, SESSION_T0_SOURCE = _session_t0()
training_report['time_guard'] = {
    't0_source': SESSION_T0_SOURCE, 'budget_h': CLS_TIME_BUDGET_H, 'projections': {}, 'stops': [],
}
print(f'NOTEBOOK_PATCH_BB2: {NOTEBOOK_PATCH_BB2} | session elapsed {session_elapsed_h():.2f} h '
      f'(start from {SESSION_T0_SOURCE}) | CLS_TIME_BUDGET_H={CLS_TIME_BUDGET_H}')
print('TRAINING_GROUP:', TRAINING_GROUP)
print('NOTEBOOK_PATCH_C:', NOTEBOOK_PATCH_C, '| ALLOW_C_WITHOUT_GATE:', ALLOW_C_WITHOUT_GATE, '| MOBILE_ZIP_NAME:', MOBILE_ZIP_NAME)
print('NOTEBOOK_PATCH_RESUME:', NOTEBOOK_PATCH_RESUME, '| DET_RESUME:', DET_RESUME)
print('NOTEBOOK_PATCH:', NOTEBOOK_PATCH)
print(f'FLAGS: IS_0A={IS_0A} IS_BB2={IS_BB2} IS_C={IS_C} RUN_YOLO={RUN_YOLO}')
if IS_0A:
    print('*** GROUP 0A: se TRAIN YOLO (~3-5h). Muon Session B -> dat TRAINING_GROUP=\"B_B2\" ***')
elif IS_BB2:
    print(f'*** GROUP B_B2: SKIP YOLO — load session-train-0a, train {CLS_BACKBONES[0]} (+B2) ***')
elif IS_C:
    print('*** GROUP C: SKIP YOLO + SKIP classifier train — load session-train-b -> TFLite ***')
print('CLS_ROOT:', CLS_ROOT, 'exists=', os.path.isdir(CLS_ROOT))
print('CLS_EXTRA:', CLS_EXTRA, 'exists=', os.path.isdir(CLS_EXTRA))
print('CLS_EXTRA_V1:', CLS_EXTRA_V1, 'exists=', os.path.isdir(CLS_EXTRA_V1), 'use=', USE_EXTRA_V1)
print('DET_ROOT:', DET_ROOT, 'exists=', os.path.isdir(DET_ROOT), 'yolo_layout=', _looks_like_yolo_root(DET_ROOT))
if not _looks_like_yolo_root(DET_ROOT):
    print('WARNING: YOLO layout not found yet. Mounted /kaggle/input:', list_kaggle_inputs())
print('EXTERNAL_VAL_ROOT:', EXTERNAL_VAL_ROOT, 'exists=', os.path.isdir(EXTERNAL_VAL_ROOT))

if TRAINING_GROUP not in ('0A', 'B_B2', 'C'):
    raise ValueError("TRAINING_GROUP phải là '0A' | 'B_B2' | 'C'")

if IS_0A:
    print(f'=== 0A: Add Data {KAGGLE_USER}/dataset-rice-leaf + yusufmurtaza01/rice-leaf-diseases + Internet ON ===')
    print(f'0A detector: {YOLO_MODEL} imgsz={DET_IMG_SIZE} epochs={DET_EPOCHS} augment={DET_AUGMENT}')
if IS_BB2:
    print(f'=== B_B2: Add Data {SESSION_A_UPLOAD_SLUG} + {KAGGLE_USER}/dataset-rice-leaf + '
          f'{KAGGLE_USER}/dataset-train-rice-mobile + Internet ON ===')
    print(f'B_B2 classifier augment: {CLS_AUGMENT_VERSION}')
    print(f'B_B2 hyperparams: CROP_AWARE_RATIO={CROP_AWARE_RATIO} HARD_PAIR_CLASS_FACTORS={HARD_PAIR_CLASS_FACTORS}')
    print(f'B_B2 backbone={CLS_BACKBONES} USE_CLASS_WEIGHT={USE_CLASS_WEIGHT} RUN_B2_FINETUNE={RUN_B2_FINETUNE}')
    print(
        f'B_B2 v3: sampler sqrt max {BALANCED_SAMPLER_MAX_FACTOR}x FOCAL_GAMMA={FOCAL_GAMMA} '
        f'LR_P2={CLS_LR_P2} unfreeze={CLS_UNFREEZE_EFFNET} (BN frozen) '
        f'CROP_AWARE_CONF={CROP_AWARE_CONF} MIN_AREA={CROP_AWARE_MIN_AREA} DEDUP_HAMMING={DEDUP_HAMMING}'
    )
    print(
        f'B_B2 data v1: sources={EXTRA_V1_SOURCES} fill weak classes to {WEAK_CLASS_TARGET} '
        f'| TTA eval={USE_TTA_EVAL} views={TTA_VIEWS} gate_use_tta={GATE_USE_TTA}'
    )
if IS_C:
    print(
        f'=== C: Add Data {SESSION_B_UPLOAD_SLUG} + Mehedi ({KAGGLE_USER}/dataset-rice-leaf) + Internet ON; '
        '+ yusufmurtaza01/rice-leaf-diseases (detector mAP50/parity; required when DET_EVAL_REQUIRED) '
        f'| ALLOW_C_WITHOUT_GATE={ALLOW_C_WITHOUT_GATE} ==='
    )


## 2. Setup — dependencies & GPU

In [ ]:
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version as _pkg_version

# Kaggle's torch / numpy / TF / OpenCV builds are compiled against each other. If pip replaces one of them,
# `import torch` fails with "SystemError: ... bad call flags", so every pip install in this notebook uses these pins.
PIP_BASE_PACKAGES = (
    'torch', 'torchvision', 'torchaudio', 'numpy', 'scipy', 'pandas', 'tensorflow', 'keras', 'tf_keras',
    'opencv-python', 'opencv-python-headless', 'opencv-contrib-python',
)
PIP_CONSTRAINTS = '/kaggle/working/pip_constraints.txt'


def _installed(name):
    try:
        return _pkg_version(name)
    except PackageNotFoundError:
        return None


base_before = {n: _installed(n) for n in PIP_BASE_PACKAGES}
with open(PIP_CONSTRAINTS, 'w', encoding='utf-8') as f:
    f.writelines(f'{n}=={v}\n' for n, v in base_before.items() if v)
print('Pinned base packages:', {n: v for n, v in base_before.items() if v})

# ultralytics>=8.4.83 turns format="tflite" into LiteRT (other names / layout); cell 16a needs the onnx2tf exporter.
_pip = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-c', PIP_CONSTRAINTS,
     'ultralytics>=8.3,<8.4.83', 'albumentations>=2.0,<3', 'imagehash'],
    capture_output=True, text=True,
)
if _pip.returncode != 0:
    print(_pip.stdout[-3000:], _pip.stderr[-3000:])
    raise RuntimeError(f'pip install failed with the base packages pinned ({PIP_CONSTRAINTS}); output above')

base_changed = {
    n: (v, _installed(n)) for n, v in base_before.items() if _installed(n) != v
}
if base_changed:
    raise RuntimeError(
        f'pip changed base packages {base_changed}. Stop the session (power icon), start a new one, then Run All.'
    )

_smoke = subprocess.run(
    [sys.executable, '-c',
     'import torch, torchvision, numpy, cv2, ultralytics, albumentations; '
     'print("torch", torch.__version__, "cuda", torch.cuda.is_available(), "| torchvision", torchvision.__version__, '
     '"| numpy", numpy.__version__, "| cv2", cv2.__version__, "| ultralytics", ultralytics.__version__, '
     '"| albumentations", albumentations.__version__)'],
    capture_output=True, text=True,
)
print(_smoke.stdout.strip())
if _smoke.returncode != 0:
    print(_smoke.stderr[-4000:])
    raise RuntimeError(
        'torch / ultralytics / albumentations do not import in a fresh Python process (error above). '
        'pip changes from an earlier run stay until the session stops: stop the session (power icon), '
        'start a new one, then Run All. Do not re-run cells in the broken session.'
    )


In [ ]:
import json
import math
import random
import shutil
import time
from collections import Counter
from pathlib import Path

import torch
import albumentations as A
import cv2
import matplotlib.patches as patches
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sn
import tensorflow as tf
import yaml
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.applications import DenseNet201, EfficientNetB0, EfficientNetB3, MobileNetV2
from tensorflow.keras.applications.densenet import preprocess_input as densenet_preprocess_input
from tensorflow.keras.applications.efficientnet import preprocess_input as efficientnet_preprocess_input
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mobilenet_preprocess_input
from tensorflow.keras.callbacks import CSVLogger, EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import Sequence
from ultralytics import YOLO

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

gpus = tf.config.list_physical_devices('GPU')
if not gpus:
    raise RuntimeError('Không có GPU. Kaggle: Settings → Accelerator → GPU')
print('GPU:', [g.name for g in gpus])
print('TensorFlow:', tf.__version__)


## 2.6 VERIFY_AUGMENT

In [ ]:
def verify_albumentations(transform):
    import warnings
    dummy = np.random.randint(0, 255, (224, 224, 3), dtype=np.uint8)
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter('always')
        _ = transform(image=dummy)['image']
        bad = [str(w.message) for w in caught if 'not valid' in str(w.message).lower()]
    assert not bad, f'Albumentations invalid params: {bad}'
    print(f'Albumentations OK (v{A.__version__}) — no invalid-param warnings')


# build_albumentations() defined in cell 8; quick smoke test after import
def _smoke_test_albumentations_api():
    _ = A.ImageCompression(quality_range=(60, 95), p=1.0)
    _ = A.CoarseDropout(
        num_holes_range=(4, 8),
        hole_height_range=(0.04, 0.10),
        hole_width_range=(0.04, 0.10),
        p=1.0,
    )
    print('Albumentations API smoke test passed')


_smoke_test_albumentations_api()


## 2.5 DATA_AUDIT

In [ ]:
def normalize_class_name(name):
    key = name.strip().lower().replace('-', ' ').replace('_', ' ')
    key = ''.join(key.split()) if key.replace(' ', '') in CLASS_NAME_MAP else key
    if key in CLASS_NAME_MAP:
        return CLASS_NAME_MAP[key]
    key_compact = key.replace(' ', '')
    if key_compact in CLASS_NAME_MAP:
        return CLASS_NAME_MAP[key_compact]
    for canonical in CANONICAL_CLASSES:
        if canonical.lower() == name.strip().lower():
            return canonical
    return name.strip()


def count_split(split_dir):
    counts = {}
    if not os.path.isdir(split_dir):
        return counts
    for cls in os.listdir(split_dir):
        cp = os.path.join(split_dir, cls)
        if os.path.isdir(cp):
            n = len([f for f in os.listdir(cp) if Path(f).suffix.lower() in IMAGE_EXTENSIONS])
            if n:
                counts[normalize_class_name(cls)] = counts.get(normalize_class_name(cls), 0) + n
    return counts


def resolve_mehedi_splits(root):
    layouts = [
        ('Training data', 'Validation data', 'Testing data'),
        ('Training Data', 'Validation Data', 'Testing Data'),
        ('train', 'valid', 'test'),
        ('train', 'val', 'test'),
    ]
    roots_to_try = [root]
    nested = os.path.join(root, 'Rice leaf disease', 'Rice leaf disease')
    if nested not in roots_to_try:
        roots_to_try.append(nested)

    for try_root in roots_to_try:
        if not os.path.isdir(try_root):
            continue
        for tr, va, te in layouts:
            paths = [os.path.join(try_root, x) for x in (tr, va, te)]
            if all(os.path.isdir(p) for p in paths):
                return paths[0], paths[1], paths[2], try_root

    for dirpath, dirnames, _ in os.walk(root):
        for tr, va, te in layouts:
            paths = [os.path.join(dirpath, x) for x in (tr, va, te)]
            if all(os.path.isdir(p) for p in paths):
                return paths[0], paths[1], paths[2], dirpath

    raise FileNotFoundError(f'Không tìm thấy train/val/test trong {root}')


def resolve_extra_train_root(extra_root):
    if not extra_root or not os.path.isdir(extra_root):
        return extra_root
    split_names = {'Training data', 'Validation data', 'Testing data',
                   'Training Data', 'Validation Data', 'Testing Data'}
    subdirs = [d for d in os.listdir(extra_root) if os.path.isdir(os.path.join(extra_root, d))]
    if any(d in split_names for d in subdirs):
        train_path = os.path.join(extra_root, 'Training data')
        if os.path.isdir(train_path):
            print(f'  Auto-resolve CLS_EXTRA → {train_path}')
            return train_path
        train_path = os.path.join(extra_root, 'Training Data')
        if os.path.isdir(train_path):
            print(f'  Auto-resolve CLS_EXTRA → {train_path}')
            return train_path
    return extra_root


def audit_det_layout(root):
    checks = {}
    for sub in ('images/train', 'images/val', 'labels/train', 'labels/val'):
        checks[sub] = os.path.isdir(os.path.join(root, sub))
    return checks


CLS_EXTRA = resolve_extra_train_root(CLS_EXTRA)

audit = {
    'cls_root_exists': os.path.isdir(CLS_ROOT),
    'cls_extra_exists': os.path.isdir(CLS_EXTRA),
    'det_root_exists': os.path.isdir(DET_ROOT),
}
print('=== DATA AUDIT ===')
print(f'  CLS_ROOT: {CLS_ROOT}')
print(f'  CLS_EXTRA: {CLS_EXTRA}')
print(f'  DET_ROOT: {DET_ROOT}')
for k, v in audit.items():
    print(f'  {k}: {v}')
assert audit['cls_root_exists'], f'Missing Mehedi dataset. Tried: {CLS_ROOT_CANDIDATES}'

# Re-resolve DET on audit (in case Config ran before Add Data finished mounting)
if (IS_0A or (IS_C and EXPORT_DET_TFLITE)) and not _looks_like_yolo_root(DET_ROOT):
    found = discover_yolo_root('/kaggle/input') or (
        discover_yolo_root(DET_KAGGLEHUB_PATH) if globals().get('DET_KAGGLEHUB_PATH') else None
    )
    if found:
        DET_ROOT = found
        audit['det_root_exists'] = True
        print('  Re-resolved DET_ROOT ->', DET_ROOT)

audit['det_root_exists'] = _looks_like_yolo_root(DET_ROOT) or os.path.isdir(DET_ROOT)
audit['det_yolo_layout'] = _looks_like_yolo_root(DET_ROOT)
print(f'  DET_ROOT (final): {DET_ROOT}')
print(f'  det_yolo_layout: {audit["det_yolo_layout"]}')

if IS_0A:
    if not audit['det_yolo_layout']:
        mounted = list_kaggle_inputs()
        print('  Mounted /kaggle/input entries (actual):')
        for m in mounted:
            print('   ', m)
        raise AssertionError(
            'Missing YOLO dataset with images/train + labels/train.\n'
            'Action: Kaggle right panel → Add Data → search "yusufmurtaza01/rice-leaf-diseases" → Add.\n'
            f'Tried candidates: {DET_ROOT_CANDIDATES}\n'
            f'Actual /kaggle/input mounts: {mounted}'
        )
elif IS_C and EXPORT_DET_TFLITE and DET_EVAL_REQUIRED and not audit['det_yolo_layout']:
    mounted = list_kaggle_inputs()
    print('  Mounted /kaggle/input entries (actual):')
    for m in mounted:
        print('   ', m)
    raise AssertionError(
        'Group C needs the YOLO dataset for detector mAP50 / parity / INT8 calibration (DET_EVAL_REQUIRED = True), '
        'but no images/train + labels/train (or train/images + train/labels) is mounted.\n'
        'Action: Kaggle right panel → Add Data → "yusufmurtaza01/rice-leaf-diseases" → Add, wait for the mount, '
        'then re-run from the Config cell. Float32-only export without eval: set DET_EVAL_REQUIRED = False.\n'
        f'kagglehub {globals().get("DET_KAGGLEHUB_SLUG")}: path={globals().get("DET_KAGGLEHUB_PATH")} '
        '(None = skipped or failed: Settings -> Internet ON, then re-run from cell 0).\n'
        f'Tried candidates: {DET_ROOT_CANDIDATES}\n'
        f'Actual /kaggle/input mounts: {mounted}'
    )
elif not audit['det_yolo_layout']:
    print('  NOTE: DET_ROOT missing — OK for B_B2/C if YOLO loaded from checkpoint')

mehedi_train, mehedi_val, mehedi_test, cls_split_root = resolve_mehedi_splits(CLS_ROOT)
print(f'  cls_split_root: {cls_split_root}')

if IS_0A or audit.get('det_yolo_layout') or audit['det_root_exists']:
    det_layout = audit_det_layout(DET_ROOT)
    print('\nYOLO layout:')
    for sub, ok in det_layout.items():
        print(f'  {sub}: {ok}')
    if not all(det_layout.values()):
        print('  WARNING: một số thư mục YOLO thiếu — kiểm tra DET_ROOT')
else:
    det_layout = {}
    print('\nYOLO layout: skipped (using Session A checkpoint)')

if audit['cls_extra_exists']:
    extra_classes = [d for d in os.listdir(CLS_EXTRA) if os.path.isdir(os.path.join(CLS_EXTRA, d))]
    print(f'\nCLS_EXTRA: {len(extra_classes)} class folders')
else:
    print('\nCLS_EXTRA: not found (merge skipped)')

for name, path in [('Train', mehedi_train), ('Val', mehedi_val), ('Test', mehedi_test)]:
    c = count_split(path)
    print(f'\n{name}: {sum(c.values())} images, {len(c)} classes')
    for cls in CANONICAL_CLASSES:
        print(f'  {cls}: {c.get(cls, 0)}')
    missing = [cls for cls in CANONICAL_CLASSES if cls not in c]
    if missing:
        print(f'  WARNING missing classes: {missing}')

training_report['data_audit'] = audit
training_report['data_audit']['resolved_paths'] = {
    'CLS_ROOT': CLS_ROOT, 'CLS_EXTRA': CLS_EXTRA, 'DET_ROOT': DET_ROOT,
}
training_report['data_audit']['cls_split_root'] = cls_split_root
training_report['data_audit']['det_layout'] = det_layout
training_report['data_audit']['mehedi_splits'] = {
    'train': mehedi_train, 'val': mehedi_val, 'test': mehedi_test,
}


## 2.7 SESSION 0 — Data audit (Group 0A)

Ghi `data_audit.json`. Không map tungro/smut.


In [ ]:
if not IS_0A:
    print('Skip Session 0 — TRAINING_GROUP != 0A')
else:
    def count_images_in_class_dirs(root):
        counts = {}
        if not os.path.isdir(root):
            return counts
        bases = [root]
        for sub in ('Training data', 'train', 'Train', 'Testing data', 'Validation data'):
            p = os.path.join(root, sub)
            if os.path.isdir(p):
                bases.append(p)
        for base in bases:
            try:
                names = os.listdir(base)
            except OSError:
                continue
            for name in names:
                d = os.path.join(base, name)
                if not os.path.isdir(d):
                    continue
                key = name.strip().lower().replace('-', ' ').replace('_', ' ')
                compact = ''.join(key.split())
                if compact in ('tungro', 'leafsmut') or 'smut' in key.split() or key == 'smut':
                    print('  BLOCKED (never merge):', d)
                    continue
                if 'tungro' in key:
                    print('  BLOCKED tungro:', d)
                    continue
                canon = CLASS_NAME_MAP.get(key) or CLASS_NAME_MAP.get(compact)
                if canon is None:
                    continue
                n = sum(1 for f in Path(d).rglob('*') if f.suffix.lower() in IMAGE_EXTENSIONS)
                if n:
                    counts[canon] = counts.get(canon, 0) + n
        return counts

    audit = {
        'cls_root': CLS_ROOT,
        'cls_root_exists': os.path.isdir(CLS_ROOT),
        'cls_extra': CLS_EXTRA,
        'cls_extra_exists': os.path.isdir(CLS_EXTRA),
        'det_root': DET_ROOT,
        'det_root_exists': os.path.isdir(DET_ROOT),
        'optional_roots': {},
        'mehedi_train_counts': {},
        'extra_counts': {},
    }
    if os.path.isdir(CLS_ROOT):
        for split_name in ('Training data', 'train', 'Train'):
            p = os.path.join(CLS_ROOT, split_name)
            if os.path.isdir(p):
                for cls in CANONICAL_CLASSES:
                    d = os.path.join(p, cls)
                    if os.path.isdir(d):
                        audit['mehedi_train_counts'][cls] = sum(
                            1 for f in Path(d).rglob('*') if f.suffix.lower() in IMAGE_EXTENSIONS
                        )
                break
    audit['extra_counts'] = count_images_in_class_dirs(CLS_EXTRA) if os.path.isdir(CLS_EXTRA) else {}
    for root in OPTIONAL_IMAGE_ROOTS:
        exists = os.path.isdir(root)
        img_counts = count_images_in_class_dirs(root) if exists else {}
        has_images = sum(img_counts.values()) > 0
        audit['optional_roots'][root] = {
            'exists': exists, 'has_images': has_images, 'counts': img_counts, 'merge_ok': has_images,
        }
        print(f'OPTIONAL {root}: exists={exists} has_images={has_images} counts={img_counts}')
    audit['merge_recommendation'] = {
        'merge_extra': os.path.isdir(CLS_EXTRA),
        'optional_mergeable_roots': [r for r, info in audit['optional_roots'].items() if info.get('merge_ok')],
        'note': 'Never merge tungro/smut; set MERGE_OPTIONAL_NOTEBOOKS=True only after review',
    }
    with open(DATA_AUDIT_PATH, 'w', encoding='utf-8') as f:
        json.dump(audit, f, indent=2, ensure_ascii=False)
    training_report['data_audit_session0'] = audit
    print('Wrote', DATA_AUDIT_PATH)


---
# PHẦN A — YOLO Detection
## 3. Resolve YOLO data.yaml

In [ ]:
if not (RUN_YOLO or (IS_C and EXPORT_DET_TFLITE and _looks_like_yolo_root(DET_ROOT))):
    print(f'Skip YOLO yaml resolve (TRAINING_GROUP={TRAINING_GROUP}) — load checkpoint')
else:
    import re

    def resolve_det_yaml(root):
        search_roots = [root]
        parent = os.path.dirname(root)
        if parent and parent not in search_roots:
            search_roots.append(parent)

        candidates = []
        for search_root in search_roots:
            if not os.path.isdir(search_root):
                continue
            for dirpath, _, filenames in os.walk(search_root):
                for fn in filenames:
                    if fn.endswith(('.yaml', '.yml')):
                        candidates.append(os.path.join(dirpath, fn))

        for path in sorted(set(candidates), key=len):
            with open(path, encoding='utf-8') as f:
                cfg = yaml.safe_load(f)
            if not cfg or ('train' not in cfg and 'names' not in cfg):
                continue

            yaml_base = root
            if not os.path.isdir(os.path.join(yaml_base, 'images', 'train')):
                for search_root in search_roots:
                    if os.path.isdir(os.path.join(search_root, 'images', 'train')):
                        yaml_base = search_root
                        break

            patched = os.path.join(WORK_DIR, 'det_data.yaml')
            cfg = dict(cfg)
            cfg['path'] = os.path.abspath(yaml_base)

            for key in ('train', 'val', 'valid', 'test'):
                if key not in cfg or not cfg[key] or os.path.isabs(str(cfg[key])):
                    continue
                rel = str(cfg[key]).replace('\\', '/')
                if not os.path.isdir(os.path.join(yaml_base, rel)):
                    for alt in (rel.replace('rice/', ''), re.sub(r'^(\.\./)+', '', rel)):
                        if os.path.isdir(os.path.join(yaml_base, alt)):
                            cfg[key] = alt
                            break

            with open(patched, 'w', encoding='utf-8') as f:
                yaml.dump(cfg, f, default_flow_style=False)
            print('Using YOLO yaml:', path)
            print('Patched yaml:', patched)
            print('Dataset path:', cfg['path'])
            print('train:', cfg.get('train'), 'val:', cfg.get('val') or cfg.get('valid'))
            print('Classes:', cfg.get('names'))
            return patched, cfg

        img_dirs = {}
        for key, subs in (
            ('train', ('images/train', 'train/images')),
            ('val', ('images/val', 'images/valid', 'valid/images', 'val/images')),
        ):
            hit = next((s for s in subs if os.path.isdir(os.path.join(root, s))), None)
            if hit:
                img_dirs[key] = hit
        if set(img_dirs) != {'train', 'val'}:
            raise FileNotFoundError(f'No YOLO yaml and no train + val image dirs in {root} (found {img_dirs})')
        cfg = {'path': os.path.abspath(root), **img_dirs, 'names': list(DET_CLASS_MAP), 'nc': len(DET_CLASS_MAP)}
        patched = os.path.join(WORK_DIR, 'det_data.yaml')
        with open(patched, 'w', encoding='utf-8') as f:
            yaml.dump(cfg, f, default_flow_style=False)
        print(f'WARNING: no yaml in {root}; wrote {patched} with names in DET_CLASS_MAP order: {cfg["names"]}')
        print('train:', cfg['train'], 'val:', cfg['val'])
        return patched, cfg


    DET_YAML, det_cfg = resolve_det_yaml(DET_ROOT)
    det_class_names = det_cfg.get('names', [])
    if isinstance(det_class_names, dict):
        det_class_names = [det_class_names[k] for k in sorted(det_class_names, key=lambda x: int(x) if str(x).isdigit() else x)]
    print('Detection classes:', det_class_names)


## 4. Explore YOLO dataset

In [ ]:
if not RUN_YOLO:
    print(f'Skip YOLO explore (TRAINING_GROUP={TRAINING_GROUP})')
else:
    def parse_yolo_label(label_path, img_w, img_h):
        boxes = []
        if not os.path.exists(label_path):
            return boxes
        with open(label_path, encoding='utf-8') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 5:
                    continue
                cls_id = int(parts[0])
                cx, cy, bw, bh = map(float, parts[1:5])
                x1 = (cx - bw / 2) * img_w
                y1 = (cy - bh / 2) * img_h
                boxes.append((cls_id, x1, y1, bw * img_w, bh * img_h))
        return boxes


    val_img_dir = None
    for sub in ('images/val', 'images/valid', 'valid/images', 'val/images'):
        p = os.path.join(DET_ROOT, sub)
        if os.path.isdir(p):
            val_img_dir = p
            break
    if val_img_dir:
        samples = [f for f in os.listdir(val_img_dir) if Path(f).suffix.lower() in IMAGE_EXTENSIONS][:9]
        fig, axes = plt.subplots(3, 3, figsize=(12, 12))
        for ax, fn in zip(axes.flatten(), samples):
            img_path = os.path.join(val_img_dir, fn)
            lbl_path = img_path.replace('/images/', '/labels/').rsplit('.', 1)[0] + '.txt'
            pil_img = Image.open(img_path).convert('RGB')
            img_w, img_h = pil_img.size
            img = np.array(pil_img)
            ax.imshow(img)
            for cls_id, x, y, w, h in parse_yolo_label(lbl_path, img_w, img_h):
                label = det_class_names[cls_id] if cls_id < len(det_class_names) else str(cls_id)
                ax.add_patch(patches.Rectangle((x, y), w, h, linewidth=2, edgecolor='lime', facecolor='none'))
            ax.set_title(fn[:20], fontsize=8)
            ax.axis('off')
        plt.suptitle('YOLO val samples')
        plt.tight_layout()
        plt.show()
    else:
        print('Val image dir not found — skip visualization')


## 4.5 Detector label audit (0A + C) (`yusufmurtaza01/rice-leaf-diseases`)

Box area / image area per class on labels/train + labels/val. Disease boxes should sit around the lesion
(median area < `DET_LESION_MEDIAN_MAX`, share of whole-leaf boxes > `DET_WHOLE_LEAF_AREA` below
`DET_WHOLE_LEAF_MAX_SHARE`). `Rice__Healthy` boxes always cover the leaf and are reported separately.


In [ ]:
if not (IS_0A or (IS_C and EXPORT_DET_TFLITE)):
    print('Skip detector label audit — needs TRAINING_GROUP 0A, or C with EXPORT_DET_TFLITE')
elif 'DET_YAML' not in globals() or not os.path.isfile(DET_YAML):
    if DET_EVAL_REQUIRED:
        raise RuntimeError(
            'det_data.yaml missing (DET_EVAL_REQUIRED = True): Add Data yusufmurtaza01/rice-leaf-diseases, '
            'then re-run from the Config cell (DET_ROOT is resolved there).'
        )
    det_label_audit = None
    print('Skip detector label audit — no det_data.yaml')
else:
    import random

    with open(DET_YAML, encoding='utf-8') as f:
        _acfg = yaml.safe_load(f)
    _anames = _acfg['names']
    if isinstance(_anames, dict):
        _anames = [_anames[k] for k in sorted(_anames, key=int)]
    _img_seg, _lbl_seg = f'{os.sep}images{os.sep}', f'{os.sep}labels{os.sep}'

    def _label_path(img):
        return _lbl_seg.join(str(img).rsplit(_img_seg, 1)).rsplit('.', 1)[0] + '.txt'

    _areas = {n: [] for n in _anames}
    _imgs_with = {n: 0 for n in _anames}
    _samples = {n: [] for n in _anames}
    _split_counts = {}
    _poly_lines = 0
    for split in ('val', 'train'):
        rel = _acfg.get(split) or (_acfg.get('valid') if split == 'val' else None)
        if not rel:
            continue
        img_dir = str(rel) if os.path.isabs(str(rel)) else os.path.join(_acfg['path'], str(rel))
        imgs = sorted(p for p in Path(img_dir).rglob('*') if p.suffix.lower() in IMAGE_EXTENSIONS)
        random.Random(42).shuffle(imgs)
        n_lbl = 0
        for img in imgs:
            lp = _label_path(img)
            if not os.path.isfile(lp):
                continue
            n_lbl += 1
            per_img = {}
            with open(lp, encoding='utf-8') as f:
                for line in f:
                    parts = line.split()
                    if len(parts) < 5:
                        continue
                    if len(parts) > 5:
                        _poly_lines += 1
                        continue
                    c = int(float(parts[0]))
                    if not 0 <= c < len(_anames):
                        continue
                    cx, cy, w, h = (float(x) for x in parts[1:5])
                    _areas[_anames[c]].append(w * h)
                    per_img.setdefault(_anames[c], []).append((cx, cy, w, h))
            for name, bx in per_img.items():
                _imgs_with[name] += 1
                if len(_samples[name]) < 9:
                    _samples[name].append((str(img), bx))
        _split_counts[split] = {'images': len(imgs), 'label_files': n_lbl}

    per_class = {}
    for n in _anames:
        a = np.asarray(_areas[n])
        per_class[n] = {
            'maps_to': DET_CLASS_MAP.get(n),
            'boxes': int(a.size),
            'images': _imgs_with[n],
            'boxes_per_image': round(a.size / _imgs_with[n], 2) if _imgs_with[n] else 0.0,
            'area_median': round(float(np.median(a)), 4) if a.size else None,
            'area_p90': round(float(np.quantile(a, 0.9)), 4) if a.size else None,
            'whole_leaf_share': round(float((a > DET_WHOLE_LEAF_AREA).mean()), 4) if a.size else None,
        }
    disease = [n for n in _anames if DET_CLASS_MAP.get(n) != 'Healthy Rice Leaf' and per_class[n]['boxes']]
    violations = [
        n for n in disease
        if per_class[n]['area_median'] >= DET_LESION_MEDIAN_MAX
        or per_class[n]['whole_leaf_share'] >= DET_WHOLE_LEAF_MAX_SHARE
    ]
    _all = np.concatenate([np.asarray(_areas[n]) for n in disease]) if disease else np.zeros(0)
    verdict = 'lesion_boxes' if disease and not violations else 'mixed_or_whole_leaf'

    print('Detector label audit (box area / image area):', _split_counts)
    print(f'  {"class":<28} {"maps_to":<24} {"boxes":>6} {"imgs":>6} {"box/img":>8} {"median":>7} {"p90":>7} {"whole":>7}')
    for n, s in per_class.items():
        med = f'{s["area_median"]:.3f}' if s['boxes'] else 'n/a'
        p90 = f'{s["area_p90"]:.3f}' if s['boxes'] else 'n/a'
        whole = f'{s["whole_leaf_share"]:.1%}' if s['boxes'] else 'n/a'
        print(f'  {n:<28} {str(s["maps_to"]):<24} {s["boxes"]:>6} {s["images"]:>6} {s["boxes_per_image"]:>8.2f} '
              f'{med:>7} {p90:>7} {whole:>7}')
    if _poly_lines:
        print(f'  WARNING: {_poly_lines} polygon label lines skipped (not bbox format)')
    if _all.size:
        print(f'  Disease classes overall: median {np.median(_all):.3f}, '
              f'whole-leaf share {(_all > DET_WHOLE_LEAF_AREA).mean():.1%}')
    if verdict == 'lesion_boxes':
        print(f'Label audit verdict: lesion_boxes (median < {DET_LESION_MEDIAN_MAX}, '
              f'whole-leaf share < {DET_WHOLE_LEAF_MAX_SHARE:.0%} for every disease class)')
    else:
        print(f'WARNING: label audit verdict {verdict}: classes {violations} look like whole-leaf boxes '
              f'(median >= {DET_LESION_MEDIAN_MAX} or whole-leaf share >= {DET_WHOLE_LEAF_MAX_SHARE:.0%}). '
              'Export continues; report this before retraining the detector.')

    for n in _anames:
        if not _samples[n]:
            continue
        fig, axes = plt.subplots(3, 3, figsize=(9, 9))
        for ax in axes.flatten():
            ax.axis('off')
        for ax, (img_path, bx) in zip(axes.flatten(), _samples[n]):
            pil = Image.open(img_path).convert('RGB')
            iw, ih = pil.size
            ax.imshow(pil)
            for cx, cy, w, h in bx:
                ax.add_patch(patches.Rectangle(
                    ((cx - w / 2) * iw, (cy - h / 2) * ih), w * iw, h * ih,
                    linewidth=2, edgecolor='lime', facecolor='none',
                ))
            ax.set_title(f'{os.path.basename(img_path)[:18]} ({len(bx)})', fontsize=7)
        fig.suptitle(f'{n} -> {DET_CLASS_MAP.get(n)}')
        plt.tight_layout()
        plt.show()
        plt.close(fig)

    det_label_audit = {
        'dataset_yaml': DET_YAML,
        'splits': _split_counts,
        'whole_leaf_area': DET_WHOLE_LEAF_AREA,
        'lesion_median_max': DET_LESION_MEDIAN_MAX,
        'whole_leaf_max_share': DET_WHOLE_LEAF_MAX_SHARE,
        'per_class': per_class,
        'disease_median': round(float(np.median(_all)), 4) if _all.size else None,
        'disease_whole_leaf_share': round(float((_all > DET_WHOLE_LEAF_AREA).mean()), 4) if _all.size else None,
        'polygon_lines_skipped': _poly_lines,
        'violations': violations,
        'verdict': verdict,
    }
    training_report['detector_label_audit'] = det_label_audit


## 5. Train YOLO11s (chỉ khi `TRAINING_GROUP = '0A'`)

`B_B2` / `C`: cell này **SKIP** — dùng `yolo_best.pt` từ checkpoint. Nếu thấy Ultralytics train → Config vẫn là `'0A'`, **Stop** ngay.

In [ ]:
if not RUN_YOLO:
    print(f'Skip YOLO train (TRAINING_GROUP={TRAINING_GROUP}, RUN_YOLO=False) — load checkpoint Session A')
else:
    print(f'*** TRAIN YOLO NOW (TRAINING_GROUP={TRAINING_GROUP}). '
          f'Neu muon Session B: Stop → TRAINING_GROUP=\"B_B2\" → chay lai Config ***')
    det_start = time.time()
    if DET_RESUME:
        import glob
        import ultralytics

        def _load_ckpt(path):
            return torch.load(str(path), map_location='cpu', weights_only=False)

        def _epoch_rank(ck):
            e = ck.get('epoch', -1)
            return float('inf') if e == -1 else e

        resume_src = next((p for p in DET_RESUME_CANDIDATES if os.path.isfile(p)), None)
        if resume_src is None:
            hits = sorted(glob.glob(DET_RESUME_GLOB, recursive=True))
            if len(hits) > 1:
                raise RuntimeError(f'Several last.pt match {DET_RESUME_GLOB}: {hits}. Keep one Add Data.')
            resume_src = hits[0] if hits else None
        if resume_src is None:
            raise FileNotFoundError(
                f'DET_RESUME = True but no last.pt in {DET_RESUME_CANDIDATES} or {DET_RESUME_GLOB}. '
                f'Mounted /kaggle/input: {list_kaggle_inputs()}. Add Data {KAGGLE_USER}/{DET_RESUME_SLUG}, '
                f'or set DET_RESUME = False to train from {YOLO_MODEL}.'
            )

        ck = _load_ckpt(resume_src)
        targs = ck.get('train_args') or {}
        ck_model = ck.get('ema') or ck.get('model')
        ck_names = getattr(ck_model, 'names', None)
        if isinstance(ck_names, dict):
            ck_names = [ck_names[k] for k in sorted(ck_names)]
        problems = []
        if Path(str(targs.get('model', ''))).name != YOLO_MODEL:
            problems.append(f"model {targs.get('model')} != YOLO_MODEL {YOLO_MODEL}")
        if int(targs.get('epochs', -1)) != DET_EPOCHS:
            problems.append(f"epochs {targs.get('epochs')} != DET_EPOCHS {DET_EPOCHS}")
        if targs.get('imgsz') != DET_IMG_SIZE:
            problems.append(f"imgsz {targs.get('imgsz')} != DET_IMG_SIZE {DET_IMG_SIZE}")
        if os.path.abspath(str(targs.get('data', ''))) != os.path.abspath(DET_YAML):
            problems.append(f"data {targs.get('data')} != DET_YAML {DET_YAML}")
        if list(ck_names or []) != list(det_class_names):
            problems.append(f'class names {ck_names} != det_data.yaml {det_class_names}')
        if problems:
            raise ValueError(f'Cannot resume from {resume_src}: ' + '; '.join(problems))
        if ck.get('version') != ultralytics.__version__:
            print(f"WARNING: last.pt saved by ultralytics {ck.get('version')}, installed {ultralytics.__version__}")

        run_dir = Path(WORK_DIR) / 'yolo_rice_detect'
        local_last = run_dir / 'weights' / 'last.pt'
        run_ck = ck
        if local_last.is_file():
            local_ck = _load_ckpt(local_last)
            if _epoch_rank(local_ck) >= _epoch_rank(ck):
                run_ck = local_ck
                print(f"Keep local {local_last} (epoch {local_ck.get('epoch')}) >= input (epoch {ck.get('epoch')})")
            del local_ck
        if run_ck is ck:
            src_dir = Path(resume_src).parent.parent
            local_last.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(resume_src, local_last)
            for rel in ('weights/best.pt', 'results.csv', 'args.yaml'):
                src = src_dir / rel
                if src.is_file():
                    shutil.copy2(src, run_dir / rel)
                else:
                    print(f'WARNING: {src} missing in resume dataset')
            print(f'Copied {src_dir} -> {run_dir}')
        if not os.path.isfile(YOLO_BEST_PATH):
            raise FileNotFoundError(
                f'{YOLO_BEST_PATH} missing: upload weights/best.pt next to last.pt '
                '(Ultralytics only rewrites best.pt when fitness improves after resume).'
            )

        last_epoch = run_ck.get('epoch', -1)
        training_report['detection_resume'] = {
            'from': resume_src,
            'last_completed_epoch': None if last_epoch == -1 else last_epoch + 1,
            'epochs': DET_EPOCHS,
            'ultralytics_ckpt': ck.get('version'),
            'ultralytics_installed': ultralytics.__version__,
            'note': 'EarlyStopping patience counts from the resume epoch',
        }
        del ck, run_ck, ck_model
        if last_epoch == -1:
            print(f'{local_last} is a finished run (epoch -1): skip training, use {YOLO_BEST_PATH}')
        else:
            print(f'RESUME YOLO: completed {last_epoch + 1}/{DET_EPOCHS}, continue at epoch {last_epoch + 2} from {resume_src}')
            YOLO(str(local_last)).train(resume=True, device=0)
    else:
        yolo_model = YOLO(YOLO_MODEL)
        yolo_model.train(
            data=DET_YAML,
            imgsz=DET_IMG_SIZE,
            epochs=DET_EPOCHS,
            batch=DET_BATCH,
            patience=DET_PATIENCE,
            project=WORK_DIR,
            name='yolo_rice_detect',
            **DET_AUGMENT,
            device=0,
            exist_ok=True,
        )
    print(f'YOLO training: {(time.time()-det_start)/60:.1f} min')
    print('Best weights:', YOLO_BEST_PATH)


## 6. Evaluate YOLO

In [ ]:
if not RUN_YOLO:
    print(f'Skip YOLO eval (TRAINING_GROUP={TRAINING_GROUP}) — load checkpoint')
else:
    yolo_best = YOLO(YOLO_BEST_PATH)
    det_metrics = yolo_best.val(data=DET_YAML, imgsz=DET_IMG_SIZE, device=0)
    det_map50 = float(det_metrics.box.map50)
    det_map5095 = float(det_metrics.box.map)
    det_precision = float(det_metrics.box.mp)
    det_recall = float(det_metrics.box.mr)
    print(f'mAP50={det_map50:.4f} mAP50-95={det_map5095:.4f} P={det_precision:.4f} R={det_recall:.4f}')
    training_report['detection'] = {
        'map50': det_map50, 'map50_95': det_map5095,
        'precision': det_precision, 'recall': det_recall,
        'model': YOLO_MODEL, 'imgsz': DET_IMG_SIZE, 'augment': DET_AUGMENT,
    }
    if det_map50 < 0.70:
        print('WARNING: mAP50 < 0.70 — pipeline vẫn chạy nhưng crop có thể kém')


## 6.5 Export Group 0A → `rice-v3-session-a-checkpoint`


In [ ]:
if not IS_0A:
    print('Skip Session A export — TRAINING_SESSION != A')
else:
    import zipfile

    assert os.path.isfile(YOLO_BEST_PATH), f'Missing YOLO weights: {YOLO_BEST_PATH}'

    if os.path.isdir(SESSION_A_EXPORT_DIR):
        shutil.rmtree(SESSION_A_EXPORT_DIR)
    os.makedirs(SESSION_A_EXPORT_DIR, exist_ok=True)

    shutil.copy2(YOLO_BEST_PATH, os.path.join(SESSION_A_EXPORT_DIR, 'yolo_best.pt'))
    if os.path.isfile(DET_YAML):
        shutil.copy2(DET_YAML, os.path.join(SESSION_A_EXPORT_DIR, 'det_data.yaml'))

    session_a_state = {
        'det_map50': det_map50,
        'det_map5095': det_map5095,
        'det_precision': det_precision,
        'det_recall': det_recall,
        'yolo_model': YOLO_MODEL,
        'det_img_size': DET_IMG_SIZE,
        'yolo_best_path': YOLO_BEST_PATH,
        'detection': training_report.get('detection', {}),
        'det_resume': training_report.get('detection_resume'),
    }
    state_path = os.path.join(SESSION_A_EXPORT_DIR, 'session_a_state.json')
    with open(state_path, 'w', encoding='utf-8') as f:
        json.dump(session_a_state, f, indent=2, ensure_ascii=False)

    partial_report = os.path.join(SESSION_A_EXPORT_DIR, 'training_report_partial.json')
    with open(partial_report, 'w', encoding='utf-8') as f:
        json.dump(training_report, f, indent=2, ensure_ascii=False)

    if os.path.isfile(SESSION_A_ZIP_PATH):
        os.remove(SESSION_A_ZIP_PATH)
    if os.path.isfile(DATA_AUDIT_PATH):
        shutil.copy2(DATA_AUDIT_PATH, os.path.join(SESSION_A_EXPORT_DIR, 'data_audit.json'))
    print(f'=== UPLOAD: {SESSION_A_UPLOAD_SLUG} ===')
    with zipfile.ZipFile(SESSION_A_ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zf:
        for fn in os.listdir(SESSION_A_EXPORT_DIR):
            zf.write(os.path.join(SESSION_A_EXPORT_DIR, fn), fn)

    print('Session A checkpoint exported:')
    for fn in sorted(os.listdir(SESSION_A_EXPORT_DIR)):
        p = os.path.join(SESSION_A_EXPORT_DIR, fn)
        print(f'  {fn} ({os.path.getsize(p)/1024/1024:.1f} MB)')
    print(f'Zip: {SESSION_A_ZIP_PATH} ({os.path.getsize(SESSION_A_ZIP_PATH)/1024/1024:.1f} MB)')
    print()
    print('=' * 60)
    print('DOWNLOAD CHECKPOINT — làm theo thứ tự:')
    print('  1. Save Version (Save & Run All hoặc Quick Save)')
    print('  2. Output tab → download session_a_checkpoint.zip')
    print('  3. Tạo Kaggle Dataset (private) → upload zip')
    print(f'  4. Đóng session. Tiếp: TRAINING_GROUP="B_B2" + Add Data {SESSION_A_UPLOAD_SLUG}')
    print('=' * 60)


---
## STOP — Kết thúc Group 0A

1. Export `session_a_checkpoint.zip` (yolo_best.pt + session_a_state.json + data_audit.json)
2. **Save Version** → Output download (+ backup local)
3. Upload Kaggle Dataset **`anhnguynthvan/session-train-0a-v5`**
4. **Đóng session**

Tiếp: `TRAINING_GROUP = 'B_B2'` + Add Data checkpoint.


## 6.6 Load Session A checkpoint

In [ ]:
if not IS_BB2:
    print('Skip Session B load — TRAINING_GROUP != B_B2')
else:
    import zipfile

    def find_checkpoint_root():
        """Find dir containing yolo_best.pt + session_a_state.json (nested OK)."""
        def walk_for_pair(root, max_depth=4):
            root = os.path.abspath(root)
            for dirpath, dirnames, filenames in os.walk(root):
                rel = os.path.relpath(dirpath, root)
                depth = 0 if rel == '.' else rel.count(os.sep) + 1
                if depth > max_depth:
                    dirnames[:] = []
                    continue
                names = set(filenames)
                if 'session_a_state.json' in names and 'yolo_best.pt' in names:
                    return dirpath
            return None

        def walk_for_state_and_best(root, max_depth=4):
            """Fallback: state + best.pt (no yolo_best.pt)."""
            root = os.path.abspath(root)
            state_dir = None
            best_path = None
            for dirpath, dirnames, filenames in os.walk(root):
                rel = os.path.relpath(dirpath, root)
                depth = 0 if rel == '.' else rel.count(os.sep) + 1
                if depth > max_depth:
                    dirnames[:] = []
                    continue
                if 'session_a_state.json' in filenames and state_dir is None:
                    state_dir = dirpath
                if 'best.pt' in filenames and best_path is None:
                    # prefer non-yolov8s best.pt
                    best_path = os.path.join(dirpath, 'best.pt')
            if state_dir and best_path and os.path.isfile(best_path):
                return state_dir, best_path
            return None, None

        for root in SESSION_A_INPUT_CANDIDATES:
            if not os.path.isdir(root):
                continue
            # Direct hit
            yolo = os.path.join(root, 'yolo_best.pt')
            state = os.path.join(root, 'session_a_state.json')
            if os.path.isfile(yolo) and os.path.isfile(state):
                zip_path = os.path.join(root, SESSION_A_ZIP_NAME)
                return root, zip_path if os.path.isfile(zip_path) else None, None
            # Nested (e.g. session_a_checkpoint (1)/)
            nested = walk_for_pair(root)
            if nested:
                zip_path = os.path.join(root, SESSION_A_ZIP_NAME)
                if not os.path.isfile(zip_path):
                    zip_path = os.path.join(nested, SESSION_A_ZIP_NAME)
                return nested, zip_path if os.path.isfile(zip_path) else None, None
            # Fallback best.pt + state
            state_dir, best_pt = walk_for_state_and_best(root)
            if state_dir and best_pt:
                return state_dir, None, best_pt
        return None, None, None

    restore_dir = os.path.join(WORK_DIR, 'session_a_restore')
    if os.path.isdir(restore_dir):
        shutil.rmtree(restore_dir)
    os.makedirs(restore_dir, exist_ok=True)

    input_root, zip_path, best_fallback = find_checkpoint_root()
    assert input_root is not None, (
        'Không tìm thấy Session A checkpoint (cần yolo_best.pt + session_a_state.json). '
        f'Add Data {SESSION_A_UPLOAD_SLUG}. Tried: {SESSION_A_INPUT_CANDIDATES}'
    )
    print(f'Using checkpoint from: {input_root}')

    if zip_path and os.path.isfile(zip_path):
        print(f'Unzipping {zip_path} ...')
        with zipfile.ZipFile(zip_path, 'r') as zf:
            zf.extractall(restore_dir)
    else:
        print(f'Copying files from {input_root} ...')
        for item in os.listdir(input_root):
            src = os.path.join(input_root, item)
            if os.path.isfile(src):
                shutil.copy2(src, os.path.join(restore_dir, item))

    yolo_src = os.path.join(restore_dir, 'yolo_best.pt')
    if not os.path.isfile(yolo_src) and best_fallback and os.path.isfile(best_fallback):
        print(f'Fallback: copy {best_fallback} -> yolo_best.pt')
        shutil.copy2(best_fallback, yolo_src)
    assert os.path.isfile(yolo_src), f'Missing yolo_best.pt in checkpoint: {restore_dir}'

    os.makedirs(os.path.dirname(YOLO_BEST_PATH), exist_ok=True)
    shutil.copy2(yolo_src, YOLO_BEST_PATH)

    det_yaml_src = os.path.join(restore_dir, 'det_data.yaml')
    if os.path.isfile(det_yaml_src):
        shutil.copy2(det_yaml_src, os.path.join(WORK_DIR, 'det_data.yaml'))

    state_path = os.path.join(restore_dir, 'session_a_state.json')
    assert os.path.isfile(state_path), f'Missing session_a_state.json in {restore_dir}'
    with open(state_path, encoding='utf-8') as f:
        session_a_state = json.load(f)

    _a_model = session_a_state.get('yolo_model')
    if _a_model != YOLO_MODEL:
        raise RuntimeError(
            f'Session A checkpoint {input_root} was trained with {_a_model}, Config YOLO_MODEL = {YOLO_MODEL}. '
            f'Add Data {SESSION_A_UPLOAD_SLUG} (v5 0A output) or fix YOLO_MODEL.'
        )
    det_map50 = float(session_a_state['det_map50'])
    det_map5095 = float(session_a_state['det_map5095'])
    det_precision = float(session_a_state['det_precision'])
    det_recall = float(session_a_state['det_recall'])
    training_report['detection'] = session_a_state.get('detection', {
        'map50': det_map50, 'map50_95': det_map5095,
        'precision': det_precision, 'recall': det_recall,
        'model': session_a_state.get('yolo_model', YOLO_MODEL),
        'imgsz': session_a_state.get('det_img_size', DET_IMG_SIZE),
    })
    training_report['session_a_checkpoint_dir'] = input_root

    yolo_best = YOLO(YOLO_BEST_PATH)
    print(f'Loaded Session A checkpoint: mAP50={det_map50:.4f} from {input_root}')


## 6.6b Validate Session A checkpoint

In [ ]:
if not IS_BB2:
    print('Skip checkpoint validation — TRAINING_SESSION != B')
else:
    assert os.path.isfile(YOLO_BEST_PATH), f'Missing YOLO weights: {YOLO_BEST_PATH}'
    yolo_size = os.path.getsize(YOLO_BEST_PATH)
    assert yolo_size > 10_000_000, f'yolo_best.pt suspiciously small ({yolo_size} bytes)'
    assert det_map50 > 0, 'det_map50 not loaded from session_a_state.json'
    print(f'Checkpoint OK — YOLO ready ({yolo_size / 1024 / 1024:.1f} MB), mAP50={det_map50:.2%}')

    smoke_img = None
    if 'mehedi_test' in dir() and os.path.isdir(mehedi_test):
        for cls in os.listdir(mehedi_test):
            cp = os.path.join(mehedi_test, cls)
            if not os.path.isdir(cp):
                continue
            for fn in os.listdir(cp):
                if Path(fn).suffix.lower() in IMAGE_EXTENSIONS:
                    smoke_img = os.path.join(cp, fn)
                    break
            if smoke_img:
                break
    if smoke_img:
        _ = yolo_best.predict(smoke_img, conf=0.20, verbose=False)
        print(f'Smoke test OK — yolo.predict on {Path(smoke_img).name}')
    else:
        print('Smoke test skipped — no test image found')


## 6.7 Load Group B checkpoint (Group C)


In [ ]:
if not IS_C:
    print('Skip Group C load — TRAINING_GROUP != C')
else:
    import zipfile

    def find_b_root():
        """Find dir with classifier_best.keras (flat or nested)."""
        def walk_for_keras(root, max_depth=4):
            root = os.path.abspath(root)
            for dirpath, dirnames, filenames in os.walk(root):
                rel = os.path.relpath(dirpath, root)
                depth = 0 if rel == '.' else rel.count(os.sep) + 1
                if depth > max_depth:
                    dirnames[:] = []
                    continue
                if 'classifier_best.keras' in filenames:
                    return dirpath
            return None

        for root in SESSION_B_INPUT_CANDIDATES:
            if not os.path.isdir(root):
                continue
            if os.path.isfile(os.path.join(root, 'classifier_best.keras')):
                return root
            nested = walk_for_keras(root)
            if nested:
                return nested
            for fn in os.listdir(root):
                if fn.endswith('.zip'):
                    out = os.path.join(WORK_DIR, 'session_b_unpacked')
                    os.makedirs(out, exist_ok=True)
                    with zipfile.ZipFile(os.path.join(root, fn), 'r') as zf:
                        zf.extractall(out)
                    if os.path.isfile(os.path.join(out, 'classifier_best.keras')):
                        return out
                    found = walk_for_keras(out)
                    if found:
                        return found
        return None

    b_root = find_b_root()
    assert b_root, (
        'Missing Session B checkpoint (cần classifier_best.keras). '
        f'Add Data {KAGGLE_USER}/session-train-b. Tried: {SESSION_B_INPUT_CANDIDATES}'
    )
    print('Loaded Session B from', b_root)
    cls_src = os.path.join(b_root, 'classifier_best.keras')
    assert os.path.isfile(cls_src), cls_src
    shutil.copy2(cls_src, CLASSIFIER_BEST_PATH)
    yolo_src = os.path.join(b_root, 'yolo_best.pt')
    if os.path.isfile(yolo_src):
        os.makedirs(os.path.dirname(YOLO_BEST_PATH), exist_ok=True)
        shutil.copy2(yolo_src, YOLO_BEST_PATH)
    ci = os.path.join(b_root, 'class_indices.json')
    if os.path.isfile(ci):
        shutil.copy2(ci, os.path.join(PIPELINE_DIR, 'class_indices.json'))
        with open(ci, encoding='utf-8') as f:
            class_indices = json.load(f)
    draft = os.path.join(b_root, 'pipeline_config_draft.json')
    if os.path.isfile(draft):
        shutil.copy2(draft, os.path.join(PIPELINE_DIR, 'pipeline_config_draft.json'))
    partial = os.path.join(b_root, 'training_report_partial.json')
    if os.path.isfile(partial):
        with open(partial, encoding='utf-8') as f:
            prev = json.load(f)
            if isinstance(prev, dict):
                training_report.update(prev)
    training_report['session_b_checkpoint_dir'] = b_root
    _b_det_model = (training_report.get('detection') or {}).get('model')
    if _b_det_model != YOLO_MODEL:
        raise RuntimeError(
            f'Session B checkpoint {b_root} carries detector {_b_det_model}, Config YOLO_MODEL = {YOLO_MODEL}. '
            f'Add Data {SESSION_B_UPLOAD_SLUG} (v5 B_B2 output).'
        )

    _b_gate = training_report.get('gate') or {}
    if not training_report.get('b_ready_for_c'):
        if not ALLOW_C_WITHOUT_GATE:
            raise RuntimeError(
                'Session B checkpoint did not pass GATE (b_ready_for_c is not true). '
                'Retrain B, or set ALLOW_C_WITHOUT_GATE = True for a trial build only.'
            )
        print('*** TRIAL BUILD: B GATE FAIL — output is NOT a release ***')
        print('    stored gate:', {
            k: _b_gate.get(k) for k in ('gate_acc', 'gate_macro_f1', 'gate_use_tta', 'pipeline_acc')
        })
    _draft_path = os.path.join(b_root, 'pipeline_config_draft.json')
    if os.path.isfile(_draft_path):
        with open(_draft_path, encoding='utf-8') as f:
            _draft = json.load(f)
        _mismatch = {}
        if _draft.get('backbone') and _draft['backbone'] != CLS_BACKBONES[0]:
            _mismatch['backbone'] = (_draft['backbone'], CLS_BACKBONES[0])
        if _draft.get('img_size') and list(_draft['img_size']) != list(CLS_IMG_SIZE):
            _mismatch['img_size'] = (_draft['img_size'], list(CLS_IMG_SIZE))
        if _mismatch:
            raise ValueError(f'Checkpoint does not match Config (checkpoint, config): {_mismatch}')

    # Restore scalars skipped in Group C (YOLO/classifier train cells not run).
    # Values come from training_report_partial.json inside session_b_checkpoint — do not invent.
    _det = training_report.get('detection') if isinstance(training_report.get('detection'), dict) else {}
    if 'det_map50' not in globals():
        if 'map50' not in _det:
            raise KeyError(
                'detection.map50 missing in training_report — '
                'Session B zip phải có training_report_partial.json với detection.map50'
            )
        det_map50 = float(_det['map50'])
        det_map5095 = float(_det.get('map50_95') or 0)
        det_precision = float(_det.get('precision') or 0)
        det_recall = float(_det.get('recall') or 0)
    if float(det_map50) <= 0:
        raise ValueError('det_map50 invalid after restore: %r' % (det_map50,))
    if 'cls_test_acc' not in globals():
        cls_test_acc = float(
            training_report.get('cls_test_acc')
            or training_report.get('classifier_test_acc')
            or 0
        )
        if cls_test_acc <= 0 and training_report.get('classifier_leaderboard'):
            cls_test_acc = float(training_report['classifier_leaderboard'][0].get('test_acc', 0) or 0)
    if 'cls_val_acc' not in globals():
        _lb = training_report.get('classifier_leaderboard') or []
        if _lb and isinstance(_lb[0], dict) and _lb[0].get('val_acc') is not None:
            cls_val_acc = float(_lb[0]['val_acc'])
        else:
            cls_val_acc = float(training_report.get('classifier_val_acc') or 0)
    if 'BEST_CONF' not in globals():
        _bt = (training_report.get('pipeline') or {}).get('best_threshold') or {}
        BEST_CONF = float(_bt.get('conf') if _bt.get('conf') is not None else 0.20)
        BEST_PADDING = float(_bt.get('padding') if _bt.get('padding') is not None else 0.15)
        pipeline_acc = float(
            _bt.get('acc')
            if _bt.get('acc') is not None
            else (training_report.get('pipeline') or {}).get('acc') or 0
        )
    print(
        'Restored checkpoint metrics: det_map50=%.4f cls_test_acc=%.4f cls_val_acc=%.4f'
        % (float(det_map50), float(cls_test_acc), float(cls_val_acc))
    )

    print('Group C artifacts ready')

    # Make models + test paths available for PHẦN C (Group C skips B_B2 merge/train)
    cls_model = tf.keras.models.load_model(CLASSIFIER_BEST_PATH, compile=False)
    _in_shape = tuple(cls_model.input_shape[1:])
    if _in_shape != (*CLS_IMG_SIZE, 3):
        raise ValueError(f'Classifier input {_in_shape} != Config {(*CLS_IMG_SIZE, 3)}')
    print('Checkpoint matches Config:', CLS_BACKBONES[0], _in_shape)
    preprocess_fn = tf.keras.applications.efficientnet.preprocess_input
    best_classifier = cls_model
    classifier_preprocess = preprocess_fn
    if os.path.isfile(YOLO_BEST_PATH):
        yolo_best = YOLO(YOLO_BEST_PATH)
    else:
        raise FileNotFoundError(
            f'Missing yolo_best.pt for Group C pipeline: {YOLO_BEST_PATH}. '
            'Checkpoint session-train-b phải có yolo_best.pt'
        )
    winner = {'name': CLS_BACKBONES[0], 'preprocess': preprocess_fn, 'model': cls_model,
              'preprocess_fn': preprocess_fn}
    if 'class_indices' in globals() and isinstance(class_indices, dict) and class_indices:
        class_names = [n for n, _ in sorted(class_indices.items(), key=lambda x: x[1])]
    else:
        class_names = list(CANONICAL_CLASSES)
        class_indices = {n: i for i, n in enumerate(class_names)}
        print('WARNING: class_indices.json missing — using CANONICAL_CLASSES')
    # test_dir normally set in §7 merge (B_B2 only); Group C uses Mehedi test from audit
    if 'mehedi_test' not in globals() or not mehedi_test or not os.path.isdir(mehedi_test):
        raise RuntimeError(
            'mehedi_test missing — chạy cell data audit (resolve_mehedi_splits) trước pipeline'
        )
    test_dir = mehedi_test
    val_dir = mehedi_val if ('mehedi_val' in globals() and mehedi_val) else mehedi_test
    train_dir = mehedi_train if ('mehedi_train' in globals() and mehedi_train) else mehedi_test
    print('Group C models loaded into memory')
    print('test_dir (Group C):', test_dir)
    print('class_names:', class_names)


In [ ]:
if IS_0A:
    raise RuntimeError(
        'Group 0A xong. Save Version → download session_a_checkpoint.zip → '
        f'upload {SESSION_A_UPLOAD_SLUG} → đóng session. '
        'Tiếp: TRAINING_GROUP="B_B2".'
    )
print('Continue TRAINING_GROUP=', TRAINING_GROUP)


---
# PHẦN B — Classifier 8 lớp
## 7. Merge Mehedi + extra train

In [ ]:
if not (IS_BB2):
    print('Skip — TRAINING_GROUP != B_B2')
else:
    import hashlib
    import imagehash
    import yaml

    def phash_bits(path):
        try:
            with Image.open(path) as im:
                return imagehash.phash(im.convert('RGB')).hash.flatten()
        except Exception:
            return None


    def phash_index(root):
        paths, bits = [], []
        if root and os.path.isdir(root):
            for cls_name in sorted(os.listdir(root)):
                cp = os.path.join(root, cls_name)
                if not os.path.isdir(cp):
                    continue
                for fn in sorted(os.listdir(cp)):
                    if Path(fn).suffix.lower() not in IMAGE_EXTENSIONS:
                        continue
                    b = phash_bits(os.path.join(cp, fn))
                    if b is not None:
                        paths.append(os.path.join(cp, fn))
                        bits.append(b)
        bank = np.stack(bits) if bits else np.zeros((0, 64), dtype=bool)
        return paths, bank


    def min_hamming(bits, bank):
        if bank.shape[0] == 0:
            return 64
        return int((bank != bits).sum(axis=1).min())


    def init_merged_train(base_train, merged_dir):
        if os.path.isdir(merged_dir):
            shutil.rmtree(merged_dir)
        shutil.copytree(base_train, merged_dir)
        _, bank = phash_index(merged_dir)
        return bank


    BLOCKED_CLASS_WORDS = ('tungro', 'smut')

    def canonical_from_dir(name):
        key = name.strip().lower().replace('-', ' ').replace('_', ' ')
        if any(w in key for w in BLOCKED_CLASS_WORDS):
            return None
        canonical = normalize_class_name(name)
        if canonical in CANONICAL_CLASSES:
            return canonical
        words = key.split()
        if len(words) > 1 and words[0] == 'rice':
            canonical = normalize_class_name(' '.join(words[1:]))
            if canonical in CANONICAL_CLASSES:
                return canonical
        return None


    def list_images(d):
        return sorted(
            str(p) for p in Path(d).rglob('*')
            if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
        )


    def collect_class_folders(root, max_depth=5):
        by_cls = {}
        root_depth = root.rstrip(os.sep).count(os.sep)
        for dirpath, dirnames, _ in os.walk(root):
            dirnames.sort()
            keep = []
            for d in dirnames:
                canonical = canonical_from_dir(d)
                if canonical:
                    by_cls.setdefault(canonical, []).extend(list_images(os.path.join(dirpath, d)))
                else:
                    keep.append(d)
            dirnames[:] = keep if dirpath.count(os.sep) - root_depth < max_depth else []
        return {c: fs for c, fs in by_cls.items() if fs}


    def yolo_label_path(img_path):
        parts = list(Path(img_path).parts)
        idx = max(i for i, p in enumerate(parts) if p == 'images')
        parts[idx] = 'labels'
        return str(Path(*parts).with_suffix('.txt'))


    def collect_yolo(root):
        by_cls, mixed = {}, 0
        for yml in sorted(Path(root).rglob('*.y*ml')):
            try:
                with open(yml) as f:
                    cfg = yaml.safe_load(f)
            except Exception:
                continue
            names = cfg.get('names') if isinstance(cfg, dict) else None
            if isinstance(names, dict):
                names = [names[k] for k in sorted(names)]
            if not names:
                continue
            idx_to_cls = {i: canonical_from_dir(str(n)) for i, n in enumerate(names)}
            for img_dir in sorted(p for p in yml.parent.rglob('images') if p.is_dir()):
                for img in list_images(img_dir):
                    lbl = yolo_label_path(img)
                    if not os.path.isfile(lbl):
                        continue
                    with open(lbl) as f:
                        ids = {int(float(line.split()[0])) for line in f if line.strip()}
                    if len(ids) != 1:
                        mixed += len(ids) > 1
                        continue
                    canonical = idx_to_cls.get(ids.pop())
                    if canonical:
                        by_cls.setdefault(canonical, []).append(img)
        by_cls = {c: sorted(set(fs)) for c, fs in by_cls.items()}
        return by_cls, mixed


    def print_tree(root, max_depth=3, max_lines=40):
        root_depth = root.rstrip(os.sep).count(os.sep)
        lines = 0
        for dirpath, dirnames, filenames in os.walk(root):
            dirnames.sort()
            depth = dirpath.count(os.sep) - root_depth
            if depth >= max_depth:
                dirnames[:] = []
            n_img = sum(1 for fn in filenames if Path(fn).suffix.lower() in IMAGE_EXTENSIONS)
            print(f'    {"  " * depth}{os.path.basename(dirpath) or dirpath}/ ({n_img} images)')
            lines += 1
            if lines >= max_lines:
                print('    ...')
                return


    def collect_source(src_root, source_tag):
        if not src_root or not os.path.isdir(src_root):
            print(f'  [{source_tag}] missing: {src_root}')
            return {}
        by_cls = collect_class_folders(src_root)
        layout = 'class_folders'
        if not by_cls:
            by_cls, mixed = collect_yolo(src_root)
            layout = 'yolo'
            if mixed:
                print(f'  [{source_tag}] yolo: skipped {mixed} mixed-class images')
        if not by_cls:
            layout = 'empty'
        counts = {c: len(fs) for c, fs in sorted(by_cls.items())}
        print(f'  [{source_tag}] layout={layout}: {counts}')
        if layout == 'empty':
            print_tree(src_root)
        return by_cls


    def merge_source(
        files_by_class, source_tag, merged_dir, holdout_bank, train_bank,
        allowed_classes=None, cap_by_class=None,
    ):
        added, seen = {}, {}
        removed = {'leak_val_test': {}, 'dup_train': {}}
        for canonical in sorted(files_by_class):
            if allowed_classes is not None and canonical not in allowed_classes:
                continue
            cap = None if cap_by_class is None else int(cap_by_class.get(canonical, 0))
            dst = os.path.join(merged_dir, canonical)
            os.makedirs(dst, exist_ok=True)
            files = list(files_by_class[canonical])
            random.Random(f'{SEED}-{source_tag}-{canonical}').shuffle(files)
            n_added = 0
            for src_f in files:
                if cap is not None and n_added >= cap:
                    break
                seen[canonical] = seen.get(canonical, 0) + 1
                tag = hashlib.md5(src_f.encode('utf-8')).hexdigest()[:8]
                dst_f = os.path.join(dst, f'{source_tag}_{tag}_{os.path.basename(src_f)}')
                if os.path.exists(dst_f):
                    continue
                bits = phash_bits(src_f)
                if bits is None:
                    continue
                if min_hamming(bits, holdout_bank) <= DEDUP_HAMMING:
                    removed['leak_val_test'][canonical] = removed['leak_val_test'].get(canonical, 0) + 1
                    continue
                if min_hamming(bits, train_bank) <= DEDUP_HAMMING:
                    removed['dup_train'][canonical] = removed['dup_train'].get(canonical, 0) + 1
                    continue
                shutil.copy2(src_f, dst_f)
                train_bank = np.vstack([train_bank, bits[None, :]])
                n_added += 1
            if n_added:
                added[canonical] = n_added
        print(f'  [{source_tag}] merged {sum(added.values())} images: {dict(sorted(added.items()))}')
        for kind, by_cls in removed.items():
            if by_cls:
                print(f'  [{source_tag}] dedup removed ({kind}): {by_cls}')
        return train_bank, added, removed, seen


    def resolve_extra_v1_root(root):
        if not root or not os.path.isdir(root):
            return None
        if any(os.path.isdir(os.path.join(root, s)) for s in EXTRA_V1_SOURCES):
            return root
        for d in sorted(os.listdir(root)):
            nested = os.path.join(root, d)
            if os.path.isdir(nested) and any(
                os.path.isdir(os.path.join(nested, s)) for s in EXTRA_V1_SOURCES
            ):
                print(f'  Auto-resolve CLS_EXTRA_V1 → {nested}')
                return nested
        return None


    mehedi_train_counts = count_split(mehedi_train)
    CLS_EXTRA = resolve_extra_train_root(CLS_EXTRA)
    extra_v1_root = resolve_extra_v1_root(CLS_EXTRA_V1) if USE_EXTRA_V1 else None
    if USE_EXTRA_V1 and extra_v1_root is None:
        raise RuntimeError(
            f'USE_EXTRA_V1=True but rice-extra-v1 not found. Add Data {KAGGLE_USER}/rice-extra-v1 '
            f'(tried {CLS_EXTRA_V1_ROOTS}) or set USE_EXTRA_V1=False. Mounted: {list_kaggle_inputs()}'
        )

    weak_set = set(MERGE_WEAK_CLASSES) if MERGE_WEAK_CLASSES_ONLY else None
    print('Collecting extra train sources...')
    merge_plan = []
    extra_inventory = {}
    if MERGE_EXTRA_INTO_TRAIN and os.path.isdir(CLS_EXTRA):
        files = collect_source(CLS_EXTRA, 'raihan')
        extra_inventory['raihan'] = {c: len(fs) for c, fs in files.items()}
        merge_plan.append(('raihan', files, None))
    if extra_v1_root:
        for src_name in EXTRA_V1_SOURCES:
            files = collect_source(os.path.join(extra_v1_root, src_name), src_name)
            extra_inventory[src_name] = {c: len(fs) for c, fs in files.items()}
            merge_plan.append((src_name, files, 'fill'))
        v1_weak_total = sum(
            n for s in EXTRA_V1_SOURCES for c, n in extra_inventory.get(s, {}).items()
            if weak_set is None or c in weak_set
        )
        if v1_weak_total == 0:
            raise RuntimeError(
                f'rice-extra-v1 has 0 usable weak-class images in {extra_v1_root}. '
                f'Inventory: {extra_inventory}. See the folder trees printed above.'
            )

    print(f'Building pHash index (DEDUP_HAMMING={DEDUP_HAMMING})...')
    _, val_bank = phash_index(mehedi_val)
    _, test_bank = phash_index(mehedi_test)
    holdout_bank = np.vstack([val_bank, test_bank])
    _, mehedi_train_bank = phash_index(mehedi_train)
    train_test_near_dup = sum(
        1 for b in test_bank if min_hamming(b, mehedi_train_bank) <= DEDUP_HAMMING
    )
    print(f'Mehedi test near-duplicates of Mehedi train: {train_test_near_dup}/{len(test_bank)}')
    if train_test_near_dup:
        print('  WARNING: official split has train/test near-duplicates — test acc may be optimistic')

    merge_added_by_source, dedup_removed_by_source, leak_ratio_by_source = {}, {}, {}
    merge_added = {}
    dedup_removed = {'leak_val_test': {}, 'dup_train': {}}
    if merge_plan:
        mode = 'weak-classes-only' if weak_set else 'all-classes'
        print(f'Merging {len(merge_plan)} sources ({mode}) into {MERGED_TRAIN_DIR}')
        train_bank = init_merged_train(mehedi_train, MERGED_TRAIN_DIR)
        for tag, files, cap in merge_plan:
            if cap == 'fill':
                _now = count_split(MERGED_TRAIN_DIR)
                cap = {c: max(0, WEAK_CLASS_TARGET - _now.get(c, 0)) for c in CANONICAL_CLASSES}
            train_bank, added, removed, seen = merge_source(
                files, tag, MERGED_TRAIN_DIR, holdout_bank, train_bank,
                allowed_classes=weak_set, cap_by_class=cap,
            )
            merge_added_by_source[tag] = added
            dedup_removed_by_source[tag] = removed
            n_seen = sum(seen.values())
            n_leak = sum(removed['leak_val_test'].values())
            leak_ratio_by_source[tag] = round(n_leak / n_seen, 4) if n_seen else 0.0
            if n_seen and leak_ratio_by_source[tag] > LEAK_WARN_RATIO:
                print(
                    f'  WARNING [{tag}]: {leak_ratio_by_source[tag]:.0%} of examined images are near-dups '
                    f'of Mehedi val/test — source likely contains Mehedi images'
                )
            for c, n in added.items():
                merge_added[c] = merge_added.get(c, 0) + n
            for kind in dedup_removed:
                for c, n in removed[kind].items():
                    dedup_removed[kind][c] = dedup_removed[kind].get(c, 0) + n
        train_dir = MERGED_TRAIN_DIR
    else:
        train_dir = mehedi_train
    val_dir = mehedi_val
    test_dir = mehedi_test
    merged_train_counts = count_split(train_dir)

    print('\nTrain images per class (Mehedi -> merged):')
    for c in CANONICAL_CLASSES:
        print(f'  {c:<24} {mehedi_train_counts.get(c, 0):>5} -> {merged_train_counts.get(c, 0):>5}')
    if extra_v1_root:
        for c in MERGE_WEAK_CLASSES:
            if not any((merge_added_by_source.get(s) or {}).get(c) for s in EXTRA_V1_SOURCES):
                print(f'  WARNING: weak class {c} got 0 images from rice-extra-v1')
        short = {
            c: merged_train_counts.get(c, 0) for c in MERGE_WEAK_CLASSES
            if merged_train_counts.get(c, 0) < MIN_WEAK_CLASS_TRAIN
        }
        assert not short, f'Weak classes below {MIN_WEAK_CLASS_TRAIN} train images after merge: {short}'

    _audit = training_report['data_audit']
    _audit['mehedi_train_counts'] = mehedi_train_counts
    _audit['merged_train_counts'] = merged_train_counts
    _audit['merge_added_by_class'] = merge_added
    _audit['merge_total_added'] = sum(merge_added.values())
    _audit['merge_added_by_source'] = merge_added_by_source
    _audit['dedup_removed_by_source'] = dedup_removed_by_source
    _audit['leak_ratio_by_source'] = leak_ratio_by_source
    _audit['extra_v1_root'] = extra_v1_root
    _audit['extra_inventory'] = extra_inventory
    _audit['extra_v1_sources'] = list(EXTRA_V1_SOURCES) if extra_v1_root else []
    _audit['weak_class_target'] = WEAK_CLASS_TARGET
    _audit['dedup_ran'] = True
    _audit['dedup_hamming'] = DEDUP_HAMMING
    _audit['dedup_removed_by_class'] = dedup_removed
    _audit['train_test_near_dup'] = train_test_near_dup
    print('train_dir:', train_dir)
    print('val_dir:', val_dir)
    print('test_dir:', test_dir)
    print(f'Train images: {sum(merged_train_counts.values())} (Mehedi base: {sum(mehedi_train_counts.values())})')


## 8. Augmentation + generators

In [ ]:
if not (IS_BB2):
    print('Skip — TRAINING_GROUP != B_B2')
else:
    def build_albumentations():
        transforms = [
            A.RandomResizedCrop(size=CLS_IMG_SIZE, scale=(0.7, 1.0), ratio=(0.8, 1.25), p=0.5),
            A.Affine(rotate=(-30, 30), scale=(0.85, 1.15), border_mode=cv2.BORDER_REFLECT_101, p=0.5),
            A.OneOf([
                A.GaussianBlur(blur_limit=(3, 5), p=1.0),
                A.MotionBlur(blur_limit=5, p=1.0),
            ], p=0.15),
            A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.6),
            A.HueSaturationValue(hue_shift_limit=8, sat_shift_limit=25, val_shift_limit=20, p=0.4),
            A.RandomShadow(p=0.2),
            A.OneOf([
                A.GaussNoise(std_range=(0.02, 0.06), p=1.0),
                A.ISONoise(color_shift=(0.01, 0.05), intensity=(0.1, 0.4), p=1.0),
            ], p=0.2),
            A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=0.1),
            A.ImageCompression(quality_range=(70, 95), p=0.15),
            # Small holes only: a large hole can erase the only lesion on the leaf.
            A.CoarseDropout(
                num_holes_range=(1, 4), hole_height_range=(0.05, 0.12), hole_width_range=(0.05, 0.12), p=0.25,
            ),
        ]
        return A.Compose(transforms)


    def crop_with_padding_rgb(image_rgb, x1, y1, x2, y2, padding=0.15):
        h, w = image_rgb.shape[:2]
        bw, bh = x2 - x1, y2 - y1
        px, py = bw * padding, bh * padding
        nx1, ny1 = max(0, int(x1 - px)), max(0, int(y1 - py))
        nx2, ny2 = min(w, int(x2 + px)), min(h, int(y2 + py))
        if (nx2 - nx1) < MIN_CROP_SIZE or (ny2 - ny1) < MIN_CROP_SIZE:
            return None
        return image_rgb[ny1:ny2, nx1:nx2]


    def build_yolo_crop_cache(yolo, train_root, conf=0.15, padding=0.15):
        cache = {}
        total = 0
        for cls_name in sorted(os.listdir(train_root)):
            cp = os.path.join(train_root, cls_name)
            if not os.path.isdir(cp):
                continue
            for fn in os.listdir(cp):
                if Path(fn).suffix.lower() not in IMAGE_EXTENSIONS:
                    continue
                path = os.path.join(cp, fn)
                total += 1
                bgr = cv2.imread(path)
                if bgr is None:
                    cache[path] = None
                    continue
                rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
                results = yolo.predict(path, conf=conf, verbose=False)[0]
                crop = None
                if results.boxes is not None and len(results.boxes) > 0:
                    best_box = max(results.boxes, key=lambda b: float(b.conf[0]))
                    x1, y1, x2, y2 = best_box.xyxy[0].cpu().numpy()
                    crop = crop_with_padding_rgb(rgb, x1, y1, x2, y2, padding)
                    if crop is not None and (
                        crop.shape[0] * crop.shape[1]
                        < CROP_AWARE_MIN_AREA * rgb.shape[0] * rgb.shape[1]
                    ):
                        crop = None
                cache[path] = crop
        n_crops = sum(1 for v in cache.values() if v is not None)
        print(f'YOLO crop cache: {n_crops}/{total} train images have valid crops')
        return cache


    class AlbumentationsSequence(Sequence):
        def __init__(self, generator, transform=None, preprocess_fn=None):
            self.generator = generator
            self.transform = transform
            self.preprocess_fn = preprocess_fn

        def __len__(self):
            return len(self.generator)

        def __getitem__(self, index):
            batch_x, batch_y = self.generator[index]
            processed = np.zeros_like(batch_x, dtype=np.float32)
            for i in range(len(batch_x)):
                img = batch_x[i].astype(np.uint8)
                if self.transform:
                    img = self.transform(image=img)['image']
                processed[i] = self.preprocess_fn(img)
            return processed, batch_y


    class OversampledSequence(Sequence):
        # Duplicate weak-class samples; optional YOLO crop mixing for pipeline alignment.
        def __init__(
            self, flow, transform, preprocess_fn, weak_class_factors, class_indices,
            datagen=None, crop_cache=None, crop_ratio=0.0,
        ):
            self.flow = flow
            self.transform = transform
            self.preprocess_fn = preprocess_fn
            self.datagen = datagen
            self.id_to_name = {v: k for k, v in class_indices.items()}
            self.weak_class_factors = weak_class_factors
            self.default_factor = OVERSAMPLE_FACTOR
            self.crop_cache = crop_cache or {}
            self.crop_ratio = crop_ratio
            self._build_index()

        def _factor_for_class(self, cls_id):
            name = self.id_to_name.get(cls_id)
            if name and name in self.weak_class_factors:
                return self.weak_class_factors[name]
            if name and name in WEAK_CLASSES_OVERSAMPLE:
                return self.default_factor
            return 1

        def _build_index(self):
            self.indices = list(range(len(self.flow.classes)))
            extra = []
            for i, cls_id in enumerate(self.flow.classes):
                factor = self._factor_for_class(cls_id)
                if factor > 1:
                    extra.extend([i] * (factor - 1))
            self.indices.extend(extra)
            random.shuffle(self.indices)

        def __len__(self):
            return math.ceil(len(self.indices) / self.flow.batch_size)

        def on_epoch_end(self):
            self._build_index()

        def _load_sample(self, path, cls_id):
            use_crop = (
                self.crop_cache
                and self.crop_ratio > 0
                and random.random() < self.crop_ratio
            )
            crop_rgb = self.crop_cache.get(path) if use_crop else None
            if crop_rgb is not None:
                pil = Image.fromarray(crop_rgb).resize(CLS_IMG_SIZE)
                arr = tf.keras.utils.img_to_array(pil)
            else:
                img = tf.keras.utils.load_img(path, target_size=CLS_IMG_SIZE)
                arr = tf.keras.utils.img_to_array(img)
            if self.datagen:
                arr = self.datagen.random_transform(arr)
            if self.transform:
                arr = self.transform(image=arr.astype(np.uint8))['image']
            return self.preprocess_fn(arr.astype(np.float32))

        def __getitem__(self, idx):
            batch_x, batch_y = [], []
            start = idx * self.flow.batch_size
            end = min(start + self.flow.batch_size, len(self.indices))
            for i in range(start, end):
                img_idx = self.indices[i]
                path = self.flow.filepaths[img_idx]
                cls_id = self.flow.classes[img_idx]
                batch_x.append(self._load_sample(path, cls_id))
                oh = np.zeros(self.flow.num_classes, dtype=np.float32)
                oh[cls_id] = 1.0
                batch_y.append(oh)
            return np.array(batch_x), np.array(batch_y)


    # Rotation and zoom come from A.Affine / A.RandomResizedCrop; doing them here as well would stack.
    train_datagen = ImageDataGenerator(
        rotation_range=0, width_shift_range=0.1, height_shift_range=0.1,
        zoom_range=0.0, horizontal_flip=True, vertical_flip=True,
        brightness_range=[0.8, 1.2], fill_mode='reflect',
    )
    eval_datagen = ImageDataGenerator()
    alb_transform = build_albumentations()
    verify_albumentations(alb_transform)
    training_report['cls_augment'] = {
        'version': CLS_AUGMENT_VERSION,
        'albumentations': [type(t).__name__ for t in alb_transform.transforms],
    }
    print('Classifier augment', CLS_AUGMENT_VERSION, training_report['cls_augment']['albumentations'])

    _grid_src = []
    for _cls in sorted(os.listdir(train_dir)):
        _cp = os.path.join(train_dir, _cls)
        if not os.path.isdir(_cp):
            continue
        _fs = sorted(f for f in os.listdir(_cp) if Path(f).suffix.lower() in IMAGE_EXTENSIONS)
        if _fs:
            _grid_src.append((_cls, os.path.join(_cp, random.Random(SEED).choice(_fs))))
    _grid_src = _grid_src[:4]
    if _grid_src:
        fig, axes = plt.subplots(len(_grid_src), 5, figsize=(15, 3 * len(_grid_src)), squeeze=False)
        for row, (_cls, _p) in zip(axes, _grid_src):
            _base = tf.keras.utils.img_to_array(tf.keras.utils.load_img(_p, target_size=CLS_IMG_SIZE))
            row[0].imshow(_base.astype(np.uint8))
            row[0].set_title(f'{_cls[:22]} (orig)', fontsize=8)
            for ax in row[1:]:
                ax.imshow(alb_transform(image=train_datagen.random_transform(_base).astype(np.uint8))['image'])
            for ax in row:
                ax.axis('off')
        fig.suptitle(f'Classifier augmentation {CLS_AUGMENT_VERSION}')
        plt.tight_layout()
        fig.savefig(os.path.join(WORK_DIR, 'augment_grid.png'), dpi=80)
        plt.show()
        plt.close(fig)

    # Build crop cache once (after yolo_best loaded in Session B / YOLO eval)
    yolo_crop_cache = {}
    if USE_CROP_AWARE_TRAIN and globals().get('yolo_best') is not None:
        print('Building crop-aware training cache...')
        yolo_crop_cache = build_yolo_crop_cache(
            yolo_best, train_dir, conf=CROP_AWARE_CONF, padding=CROP_AWARE_PADDING,
        )
        n_valid = sum(1 for v in yolo_crop_cache.values() if v is not None)
        est_mb = n_valid * CLS_IMG_SIZE[0] * CLS_IMG_SIZE[1] * 3 / (1024 * 1024)
        print(f'Crop cache RAM estimate: {n_valid} crops ~{est_mb:.0f} MB')
        if est_mb > 500:
            print('WARNING: crop cache lớn — cân nhắc USE_CROP_AWARE_TRAIN=False nếu OOM')
    elif USE_CROP_AWARE_TRAIN:
        print('WARNING: USE_CROP_AWARE_TRAIN=True but yolo_best missing — using full images only')


## 8.5 SESSION_B_PREFLIGHT

In [ ]:
if IS_BB2:
    _train_counts = count_split(train_dir) if 'train_dir' in dir() else {}
    _train_total = sum(_train_counts.values())
    checks = {
        'TRAINING_GROUP_BB2': IS_BB2,
        'CLS_BATCH_safe': CLS_BATCH <= 16,
        'CLS_RESUME_off': not CLS_RESUME,
        'single_backbone': len(CLS_BACKBONES) == 1,
        'cls_root': os.path.isdir(CLS_ROOT),
        'yolo_ckpt': os.path.isfile(YOLO_BEST_PATH),
        'merge_extra': (not MERGE_EXTRA_INTO_TRAIN) or os.path.isdir(CLS_EXTRA),
        'train_images_ok': _train_total >= 1300,
        'no_extra_leak_into_test': bool(training_report['data_audit'].get('dedup_ran')),
        'extra_v1_merged': (not USE_EXTRA_V1) or bool(training_report['data_audit'].get('extra_v1_root')),
    }
    print('=== SESSION B TRAIN PREFLIGHT ===')
    for k, v in checks.items():
        print(f'  {k}: {"PASS" if v else "FAIL"}')
    print(f'  train_images: {_train_total} (Mehedi base ~1322; with rice-extra-v1 expect ~4000-6000)')
    assert all(checks.values()), 'Preflight FAIL — fix issues above before cell 9'
    print('Preflight PASS — safe to run classifier training')
else:
    print('Skip Session B preflight — TRAINING_SESSION != B')


## 9. Train classifier — auto-select best backbone

In [ ]:
if not (IS_BB2):
    print('Skip — TRAINING_GROUP != B_B2')
else:
    BACKBONE_CONFIG = {
        'mobilenetv2': {
            'builder': lambda: MobileNetV2(weights='imagenet', include_top=False, input_shape=(*CLS_IMG_SIZE, 3)),
            'preprocess': lambda img: mobilenet_preprocess_input(img.astype(np.float32)),
            'unfreeze': -30,
        },
        'efficientnetb0': {
            'builder': lambda: EfficientNetB0(weights='imagenet', include_top=False, input_shape=(*CLS_IMG_SIZE, 3)),
            'preprocess': lambda img: efficientnet_preprocess_input(img.astype(np.float32)),
            'unfreeze': CLS_UNFREEZE_EFFNET,
        },
        'efficientnetb3': {
            'builder': lambda: EfficientNetB3(weights='imagenet', include_top=False, input_shape=(*CLS_IMG_SIZE, 3)),
            'preprocess': lambda img: efficientnet_preprocess_input(img.astype(np.float32)),
            'unfreeze': CLS_UNFREEZE_EFFNET,
        },
        'densenet201': {
            'builder': lambda: DenseNet201(weights='imagenet', include_top=False, input_shape=(*CLS_IMG_SIZE, 3)),
            'preprocess': lambda img: densenet_preprocess_input(img.astype(np.float32)),
            'unfreeze': 150,
        },
    }


    class CategoricalFocalLoss(tf.keras.losses.Loss):
        def __init__(self, gamma=2.0, label_smoothing=0.0, **kwargs):
            super().__init__(**kwargs)
            self.gamma = gamma
            self.label_smoothing = label_smoothing

        def call(self, y_true, y_pred):
            y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
            if self.label_smoothing > 0:
                num_classes = tf.shape(y_true)[-1]
                y_true = y_true * (1.0 - self.label_smoothing) + self.label_smoothing / tf.cast(num_classes, tf.float32)
            ce = -y_true * tf.math.log(y_pred)
            modulating = tf.pow(1.0 - y_pred, self.gamma)
            return tf.reduce_sum(modulating * ce, axis=-1)


    def build_loss_fn():
        if USE_FOCAL_LOSS:
            return CategoricalFocalLoss(gamma=FOCAL_GAMMA, label_smoothing=CLS_LABEL_SMOOTHING)
        return tf.keras.losses.CategoricalCrossentropy(label_smoothing=CLS_LABEL_SMOOTHING)


    def build_classifier(backbone_name, num_classes):
        cfg = BACKBONE_CONFIG[backbone_name]
        backbone = cfg['builder']()
        x = backbone.output
        x = GlobalAveragePooling2D()(x)
        x = Dropout(CLS_DROPOUT)(x)
        out = Dense(num_classes, activation='softmax')(x)
        return Model(backbone.input, out), backbone, cfg['preprocess']


    def get_backbone_layer(model, backbone_name):
        hints = {
            'densenet201': ('densenet',),
            'mobilenetv2': ('mobilenet',),
            'efficientnetb0': ('efficientnet',),
            'efficientnetb3': ('efficientnet',),
        }.get(backbone_name, (backbone_name,))
        for layer in model.layers:
            if hasattr(layer, 'layers') and any(h in layer.name.lower() for h in hints):
                return layer
        for layer in model.layers:
            if hasattr(layer, 'layers') and len(layer.layers) > 20:
                return layer
        layer_info = [(l.name, type(l).__name__) for l in model.layers]
        raise ValueError(f'Cannot locate backbone "{backbone_name}" in loaded model: {layer_info}')


    def freeze_batchnorm(model):
        # BN stats must stay frozen: batch 16 is too small to re-estimate them.
        for layer in model.layers:
            if isinstance(layer, tf.keras.layers.BatchNormalization):
                layer.trainable = False
            elif hasattr(layer, 'layers'):
                freeze_batchnorm(layer)


    def set_trainable(backbone, unfreeze):
        if not hasattr(backbone, 'layers'):
            raise TypeError(
                f'set_trainable expected nested backbone model, got {type(backbone).__name__}'
            )
        if unfreeze == 'allFalse':
            for layer in backbone.layers:
                layer.trainable = False
        elif isinstance(unfreeze, int) and unfreeze >= 0:
            for layer in backbone.layers[:unfreeze]:
                layer.trainable = False
            for layer in backbone.layers[unfreeze:]:
                layer.trainable = True
        else:
            for layer in backbone.layers[:unfreeze]:
                layer.trainable = False
            for layer in backbone.layers[unfreeze:]:
                layer.trainable = True
        freeze_batchnorm(backbone)


    def compute_balanced_factors(flow, max_factor=None):
        max_factor = max_factor or BALANCED_SAMPLER_MAX_FACTOR
        counts = Counter(flow.classes)
        top = max(counts.values())
        id_to_name = {v: k for k, v in flow.class_indices.items()}
        return {
            id_to_name[c]: int(min(max_factor, max(1, round(math.sqrt(top / n)))))
            for c, n in counts.items()
        }


    def make_generators(preprocess_fn):
        tg = train_datagen.flow_from_directory(
            train_dir, target_size=CLS_IMG_SIZE, batch_size=CLS_BATCH,
            class_mode='categorical', shuffle=True,
        )
        vg = eval_datagen.flow_from_directory(
            val_dir, target_size=CLS_IMG_SIZE, batch_size=CLS_BATCH,
            class_mode='categorical', shuffle=False,
        )
        eg = eval_datagen.flow_from_directory(
            test_dir, target_size=CLS_IMG_SIZE, batch_size=CLS_BATCH,
            class_mode='categorical', shuffle=False,
        )
        crop_ratio = CROP_AWARE_RATIO if yolo_crop_cache else 0.0
        balanced_factors = compute_balanced_factors(tg)
        print(f'Balanced sampler factors (sqrt, max {BALANCED_SAMPLER_MAX_FACTOR}x): {balanced_factors}')
        train_seq = OversampledSequence(
            tg, alb_transform, preprocess_fn,
            balanced_factors, tg.class_indices, train_datagen,
            crop_cache=yolo_crop_cache, crop_ratio=crop_ratio,
        )
        val_seq = AlbumentationsSequence(vg, None, preprocess_fn)
        test_seq = AlbumentationsSequence(eg, None, preprocess_fn)
        return tg, vg, eg, train_seq, val_seq, test_seq


    def warmup_cosine_lr(epochs, base_lr, warmup=None, min_lr=1e-7):
        warmup = CLS_WARMUP_EPOCHS_P2 if warmup is None else warmup

        def schedule(epoch, lr=None):
            if epoch < warmup:
                return float(base_lr * (epoch + 1) / (warmup + 1))
            progress = (epoch - warmup) / max(1, epochs - warmup)
            return float(min_lr + 0.5 * (base_lr - min_lr) * (1 + math.cos(math.pi * progress)))
        return schedule


    class TimeBudgetCallback(tf.keras.callbacks.Callback):
        def __init__(self, phase):
            super().__init__()
            self.phase = phase
            self.deadline = SESSION_T0 + CLS_TIME_BUDGET_H * 3600
            self.epoch = 0
            self.epoch_start = time.time()

        def _stop(self, why):
            self.model.stop_training = True
            elapsed = session_elapsed_h()
            training_report['time_guard']['stops'].append(
                {'phase': self.phase, 'why': why, 'elapsed_h': round(elapsed, 2)}
            )
            print(f'\nTIME GUARD: stop {self.phase} ({why}) at {elapsed:.2f} h, budget {CLS_TIME_BUDGET_H} h')

        def on_epoch_begin(self, epoch, logs=None):
            self.epoch = epoch
            self.epoch_start = time.time()

        def on_train_batch_end(self, batch, logs=None):
            if not self.model.stop_training and time.time() > self.deadline:
                self._stop(f'budget passed during epoch {self.epoch + 1}')

        def on_epoch_end(self, epoch, logs=None):
            epoch_s = time.time() - self.epoch_start
            total = int(self.params.get('epochs') or 0)
            if epoch == 0:
                proj = session_elapsed_h() + epoch_s * max(0, total - 1) / 3600
                training_report['time_guard']['projections'][self.phase] = round(proj, 2)
                print(f'\nPROJECTION: {self.phase} ~{proj:.1f} h since session start if all {total} epochs run '
                      f'({epoch_s / 60:.1f} min/epoch, budget {CLS_TIME_BUDGET_H} h)')
            if not self.model.stop_training and time.time() + epoch_s > self.deadline:
                self._stop(f'next epoch would end past the budget, after epoch {epoch + 1}/{total}')


    def build_train_callbacks(
        val_seq, val_steps, num_classes, val_f1_every, ckpt_acc, ckpt_f1, lr_schedule=None,
    ):
        lr_cb = (
            tf.keras.callbacks.LearningRateScheduler(lr_schedule, verbose=0)
            if lr_schedule is not None
            else ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=5, min_lr=1e-7, verbose=1)
        )
        return [
            TrainingProgressCallback(),
            ValMacroF1Callback(val_seq, val_steps, num_classes, every_n_epochs=val_f1_every),
            ModelCheckpoint(ckpt_acc, monitor='val_accuracy', save_best_only=True, mode='max', verbose=1),
            ModelCheckpoint(ckpt_f1, monitor='val_macro_f1', save_best_only=True, mode='max', verbose=1),
            EarlyStopping(
                monitor=CLS_CHECKPOINT_METRIC, mode='max',
                patience=CLS_EARLY_STOP, restore_best_weights=False, verbose=1,
            ),
            lr_cb,
            TimeBudgetCallback('P2' if lr_schedule is not None else 'P1'),
        ]


    def free_ram_before_phase2(backbone_name, ckpt_primary, ckpt_acc, focal_custom, num_classes):
        import gc
        gc.collect()
        tf.keras.backend.clear_session()
        path = ckpt_primary if os.path.isfile(ckpt_primary) else ckpt_acc
        if path and os.path.isfile(path):
            print(f'Reloading model for Phase 2 from {path}')
            loaded = tf.keras.models.load_model(path, custom_objects=focal_custom)
            model, backbone, _ = build_classifier(backbone_name, num_classes)
            model.set_weights(loaded.get_weights())
            del loaded
            gc.collect()
            set_trainable(backbone, BACKBONE_CONFIG[backbone_name]['unfreeze'])
            return model, backbone
        print('WARNING: no Phase 1 checkpoint — rebuilding model for Phase 2')
        model, backbone, _ = build_classifier(backbone_name, num_classes)
        set_trainable(backbone, BACKBONE_CONFIG[backbone_name]['unfreeze'])
        return model, backbone


    def fit_phase2_with_oom_fallback(
        model, backbone_name, preprocess_fn, num_classes, focal_custom,
        vg, train_seq, val_seq, steps, val_steps, epochs_p2, val_f1_every,
        ckpt_acc, ckpt_f1, loss_fn, fit_kwargs,
    ):
        global CLS_BATCH
        batch_attempt = CLS_BATCH
        while batch_attempt >= 4:
            try:
                cbs = build_train_callbacks(
                    val_seq, val_steps, num_classes, val_f1_every, ckpt_acc, ckpt_f1,
                    lr_schedule=warmup_cosine_lr(epochs_p2, CLS_LR_P2),
                )
                model.compile(
                    optimizer=tf.keras.optimizers.AdamW(CLS_LR_P2, weight_decay=CLS_WEIGHT_DECAY),
                    loss=loss_fn, metrics=['accuracy'],
                )
                model.fit(
                    train_seq, steps_per_epoch=steps, epochs=epochs_p2,
                    validation_data=val_seq, validation_steps=val_steps,
                    callbacks=cbs, verbose=1, **fit_kwargs,
                )
                return model, train_seq, val_seq, steps, val_steps
            except (tf.errors.ResourceExhaustedError, RuntimeError, MemoryError) as exc:
                msg = str(exc).lower()
                if 'memory' not in msg and 'resource exhausted' not in msg and 'oom' not in msg:
                    raise
                if batch_attempt <= 4:
                    raise
                batch_attempt = batch_attempt // 2
                CLS_BATCH = batch_attempt
                print(f'OOM Phase 2 — retry with CLS_BATCH={CLS_BATCH}')
                del model
                import gc
                gc.collect()
                tf.keras.backend.clear_session()
                ckpt_primary = ckpt_f1 if CLS_CHECKPOINT_METRIC == 'val_macro_f1' else ckpt_acc
                model, _ = free_ram_before_phase2(
                    backbone_name, ckpt_primary, ckpt_acc, focal_custom, num_classes,
                )
                _, vg, _, train_seq, val_seq, _ = make_generators(preprocess_fn)
                steps = len(train_seq)
                val_steps = math.ceil(vg.samples / CLS_BATCH)
        return model, train_seq, val_seq, steps, val_steps


    class TrainingProgressCallback(tf.keras.callbacks.Callback):
        def on_train_batch_end(self, batch, logs=None):
            logs = logs or {}
            steps = self.params.get('steps')
            if batch == 0:
                print('Batch 1 xong — XLA compile hoan tat, training binh thuong')
            elif batch > 0 and batch % 20 == 0:
                step_info = f'{batch}/{steps}' if steps else str(batch)
                print(f'  step {step_info} loss={logs.get("loss", 0):.4f}')


    class ValMacroF1Callback(tf.keras.callbacks.Callback):
        def __init__(self, val_seq, val_steps, num_classes, every_n_epochs=1):
            super().__init__()
            self.val_seq = val_seq
            self.val_steps = val_steps
            self.num_classes = num_classes
            self.every_n_epochs = max(1, every_n_epochs)
            self.best_macro_f1 = 0.0
            self.last_macro_f1 = 0.0

        def on_epoch_end(self, epoch, logs=None):
            logs = logs or {}
            if (epoch + 1) % self.every_n_epochs != 0:
                logs['val_macro_f1'] = self.last_macro_f1
                return
            y_true, y_pred = [], []
            for i in range(self.val_steps):
                bx, by = self.val_seq[i]
                preds = self.model.predict(bx, verbose=0)
                y_true.extend(np.argmax(by, axis=1))
                y_pred.extend(np.argmax(preds, axis=1))
            macro_f1 = float(f1_score(y_true, y_pred, average='macro', zero_division=0))
            self.last_macro_f1 = macro_f1
            logs['val_macro_f1'] = macro_f1
            if macro_f1 >= self.best_macro_f1:
                self.best_macro_f1 = macro_f1
                print(f'\nEpoch {epoch + 1}: val_macro_f1 improved to {macro_f1:.3f}')
            else:
                print(f'\nEpoch {epoch + 1}: val_macro_f1 = {macro_f1:.3f} (best {self.best_macro_f1:.3f})')


    def train_backbone(backbone_name):
        print(f'\n{"="*60}\nTraining backbone: {backbone_name}\n{"="*60}')
        cfg = BACKBONE_CONFIG[backbone_name]
        preprocess_fn = cfg['preprocess']
        focal_custom = {'CategoricalFocalLoss': CategoricalFocalLoss}

        epochs_p1 = 3 if QUICK_TRAIN else CLS_EPOCHS_P1
        epochs_p2 = 5 if QUICK_TRAIN else CLS_EPOCHS_P2
        val_f1_every = 3 if QUICK_TRAIN else VAL_F1_EVERY_N_EPOCHS
        if QUICK_TRAIN:
            print('QUICK_TRAIN=True — smoke test only (3+5 epochs)')

        tg, vg, eg, train_seq, val_seq, test_seq = make_generators(preprocess_fn)
        model, backbone, _ = build_classifier(backbone_name, tg.num_classes)
        set_trainable(backbone, 'allFalse')
        class_weight = None
        if USE_CLASS_WEIGHT:
            cw = compute_class_weight('balanced', classes=np.arange(tg.num_classes), y=tg.classes)
            class_weight = {i: float(w) for i, w in enumerate(cw)}

        steps = len(train_seq)
        val_steps = math.ceil(vg.samples / CLS_BATCH)

        ckpt_acc = os.path.join(WORK_DIR, f'classifier_{backbone_name}.keras')
        ckpt_f1 = os.path.join(WORK_DIR, f'classifier_{backbone_name}_best_f1.keras')
        ckpt_primary = ckpt_f1 if CLS_CHECKPOINT_METRIC == 'val_macro_f1' else ckpt_acc

        if CLS_RESUME and os.path.isfile(ckpt_primary):
            print(f'Resuming weights from {ckpt_primary}')
            resumed = tf.keras.models.load_model(ckpt_primary, custom_objects=focal_custom)
            model.set_weights(resumed.get_weights())
            del resumed

        print(f'Train samples (oversampled index): {len(train_seq.indices)}')
        print(f'Steps/epoch: {steps} | Val steps: {val_steps} | batch={CLS_BATCH}')
        print(f'Est. ~{steps * 2}s/step -> ~{steps * 2 / 60:.0f} min/epoch (Phase 1)')
        print('Batch 1 dang XLA compile — co the mat 10-20 phut, KHONG bam Stop')

        cbs = build_train_callbacks(
            val_seq, val_steps, tg.num_classes, val_f1_every, ckpt_acc, ckpt_f1,
        )
        loss_fn = build_loss_fn()
        fit_kwargs = {'class_weight': class_weight} if class_weight else {}

        model.compile(optimizer=tf.keras.optimizers.AdamW(CLS_LR_P1, weight_decay=CLS_WEIGHT_DECAY), loss=loss_fn, metrics=['accuracy'])

        model.fit(train_seq, steps_per_epoch=steps, epochs=epochs_p1,
                  validation_data=val_seq, validation_steps=val_steps,
                  callbacks=cbs, verbose=1, **fit_kwargs)

        if FREE_RAM_BEFORE_PHASE2:
            del model
            import gc
            gc.collect()
            model, backbone = free_ram_before_phase2(
                backbone_name, ckpt_primary, ckpt_acc, focal_custom, tg.num_classes,
            )
        else:
            set_trainable(backbone, BACKBONE_CONFIG[backbone_name]['unfreeze'])

        model, train_seq, val_seq, steps, val_steps = fit_phase2_with_oom_fallback(
            model, backbone_name, preprocess_fn, tg.num_classes, focal_custom,
            vg, train_seq, val_seq, steps, val_steps, epochs_p2, val_f1_every,
            ckpt_acc, ckpt_f1, loss_fn, fit_kwargs,
        )

        if not os.path.isfile(ckpt_primary):
            raise FileNotFoundError(f'Missing checkpoint: {ckpt_primary}')
        best = tf.keras.models.load_model(ckpt_primary, custom_objects=focal_custom)
        val_loss, val_acc = best.evaluate(val_seq, steps=val_steps, verbose=0)
        test_loss, test_acc = best.evaluate(test_seq, steps=math.ceil(eg.samples / CLS_BATCH), verbose=0)

        y_true, y_pred = [], []
        eg.reset()
        for i in range(len(test_seq)):
            bx, by = test_seq[i]
            preds = best.predict(bx, verbose=0)
            y_true.extend(np.argmax(by, axis=1))
            y_pred.extend(np.argmax(preds, axis=1))
        macro_f1 = f1_score(y_true, y_pred, average='macro')

        # val macro F1 của best checkpoint
        yv_true, yv_pred = [], []
        for i in range(val_steps):
            bx, by = val_seq[i]
            preds = best.predict(bx, verbose=0)
            yv_true.extend(np.argmax(by, axis=1))
            yv_pred.extend(np.argmax(preds, axis=1))
        val_macro_f1 = f1_score(yv_true, yv_pred, average='macro')

        return {
            'name': backbone_name, 'model': best, 'path': ckpt_primary,
            'preprocess_fn': preprocess_fn, 'val_acc': float(val_acc),
            'val_macro_f1': float(val_macro_f1),
            'test_acc': float(test_acc), 'macro_f1': float(macro_f1),
            'test_seq': test_seq, 'test_gen': eg, 'tg': tg,
            'train_seq': train_seq, 'val_seq': val_seq, 'vg': vg,
        }


    leaderboard = []
    for bb in CLS_BACKBONES:
        result = train_backbone(bb)
        leaderboard.append(result)
        training_report['classifier_leaderboard'].append({
            'backbone': result['name'], 'val_acc': result['val_acc'],
            'val_macro_f1': result['val_macro_f1'],
            'test_acc': result['test_acc'], 'macro_f1': result['macro_f1'],
        })

    winner_key = 'val_macro_f1' if CLS_CHECKPOINT_METRIC == 'val_macro_f1' else 'val_acc'
    winner = max(leaderboard, key=lambda r: r[winner_key])
    best_classifier = winner['model']
    classifier_preprocess = winner['preprocess_fn']
    class_indices = winner['tg'].class_indices
    class_names = [n for n, _ in sorted(class_indices.items(), key=lambda x: x[1])]
    cls_test_acc = winner['test_acc']
    cls_val_acc = winner['val_acc']

    shutil.copy2(winner['path'], CLASSIFIER_BEST_PATH)
    train_seq = winner['train_seq']
    val_seq = winner['val_seq']
    print(f'\nWinner: {winner["name"]} | val={cls_val_acc:.2%} val_f1={winner["val_macro_f1"]:.3f} test={cls_test_acc:.2%}')


## 10. Classifier evaluation + MODEL_LEADERBOARD

In [ ]:
if not (IS_BB2):
    print('Skip — TRAINING_GROUP != B_B2')
else:
    print('\n=== MODEL LEADERBOARD ===')
    print(f'{"Backbone":<16} {"Val Acc":>8} {"Val F1":>8} {"Test Acc":>9} {"Macro F1":>9}')
    for r in leaderboard:
        print(f'{r["name"]:<16} {r["val_acc"]:>8.2%} {r["val_macro_f1"]:>8.3f} {r["test_acc"]:>9.2%} {r["macro_f1"]:>9.3f}')
    print(f'\n→ Winner: {winner["name"]} (by {CLS_CHECKPOINT_METRIC})')

    def predict_tta(model, seq, views=None):
        views = tuple(views or TTA_VIEWS)
        y_true, probs = [], []
        for i in range(len(seq)):
            bx, by = seq[i]
            acc = np.zeros((bx.shape[0], by.shape[1]), dtype=np.float64)
            for v in views:
                if v == 'orig':
                    xb = bx
                elif v == 'hflip':
                    xb = bx[:, :, ::-1, :]
                elif v == 'vflip':
                    xb = bx[:, ::-1, :, :]
                else:
                    raise ValueError(f'Unknown TTA view: {v}')
                acc += model.predict(np.ascontiguousarray(xb), verbose=0)
            probs.append(acc / len(views))
            y_true.extend(np.argmax(by, axis=1))
        return np.array(y_true), np.concatenate(probs, axis=0)


    def tta_metrics(model, seq):
        yt, pr = predict_tta(model, seq)
        yp = np.argmax(pr, axis=1)
        rep = classification_report(
            yt, yp, target_names=class_names, digits=3, output_dict=True, zero_division=0,
        )
        return {
            'acc': float(np.mean(yt == yp)),
            'macro_f1': float(rep['macro avg']['f1-score']),
            'per_class': {n: rep[n] for n in class_names if n in rep},
        }

    y_true, y_pred = [], []
    winner['test_gen'].reset()
    for i in range(len(winner['test_seq'])):
        bx, by = winner['test_seq'][i]
        preds = best_classifier.predict(bx, verbose=0)
        y_true.extend(np.argmax(by, axis=1))
        y_pred.extend(np.argmax(preds, axis=1))

    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(10, 8))
    sn.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
    plt.title(f'Confusion Matrix — {winner["name"]} Test Acc: {cls_test_acc:.1%}')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
    cls_report = classification_report(y_true, y_pred, target_names=class_names, digits=3, output_dict=True)
    print(classification_report(y_true, y_pred, target_names=class_names, digits=3))

    # Top confusion pairs (off-diagonal)
    confusion_pairs = []
    for i, true_name in enumerate(class_names):
        for j, pred_name in enumerate(class_names):
            if i != j and cm[i, j] > 0:
                confusion_pairs.append({
                    'true': true_name, 'pred': pred_name, 'count': int(cm[i, j]),
                })
    confusion_pairs.sort(key=lambda x: x['count'], reverse=True)
    print('\nTop confusion pairs:')
    for pair in confusion_pairs[:5]:
        print(f"  {pair['true']} → {pair['pred']}: {pair['count']}")

    training_report['classifier_winner'] = winner['name']
    training_report['classifier_test_acc'] = cls_test_acc
    training_report['classifier_val_macro_f1'] = winner['val_macro_f1']
    training_report['classifier_per_class'] = {
        name: cls_report[name] for name in class_names if name in cls_report
    }
    training_report['classifier_macro_f1'] = float(cls_report['macro avg']['f1-score'])
    training_report['classifier_top_confusions'] = confusion_pairs[:10]

    if USE_TTA_EVAL:
        _tta_pre = tta_metrics(best_classifier, winner['test_seq'])
        training_report['classifier_tta_pre_b2'] = {
            'acc': _tta_pre['acc'], 'macro_f1': _tta_pre['macro_f1'], 'views': list(TTA_VIEWS),
        }
        print(
            f'TTA {TTA_VIEWS} (pre-B2): test_acc={_tta_pre["acc"]:.3f} '
            f'macro_f1={_tta_pre["macro_f1"]:.3f} | no-TTA test_acc={cls_test_acc:.3f}'
        )


## 10.5 SESSION B2 — Hard-pair fine-tune (optional)


In [ ]:
if not IS_BB2:
    print('Skip B2 — TRAINING_GROUP != B_B2')
elif not RUN_B2_FINETUNE:
    print('Skip B2 — RUN_B2_FINETUNE=False')
else:
    cls_acc = float(training_report.get('cls_test_acc') or training_report.get('classifier_test_acc') or 0)
    if cls_acc <= 0 and training_report.get('classifier_leaderboard'):
        cls_acc = float(training_report['classifier_leaderboard'][0].get('test_acc', 0) or 0)
    need_b2 = cls_acc < (TARGET_ACCURACY + 0.03)
    print(f'B2 check: cls_acc={cls_acc:.3f} need_b2={need_b2}')
    if not need_b2:
        print('PASS đủ rõ — skip B2 fine-tune')
    elif session_elapsed_h() >= CLS_TIME_BUDGET_H:
        print(f'Skip B2 — time guard: {session_elapsed_h():.2f} h >= budget {CLS_TIME_BUDGET_H} h; '
              'keeping the pre-B2 classifier')
        training_report['b2_finetune'] = {'skipped': 'time_guard', 'elapsed_h': round(session_elapsed_h(), 2)}
    elif not os.path.isfile(CLASSIFIER_BEST_PATH):
        raise RuntimeError('No classifier_best.keras — cannot run B2')
    else:
        print(f'B2 fine-tune up to {B2_EPOCHS} epochs @ LR={B2_LR}')
        focal_custom = {'CategoricalFocalLoss': CategoricalFocalLoss} if USE_FOCAL_LOSS else {}
        model_b2 = tf.keras.models.load_model(
            CLASSIFIER_BEST_PATH, compile=False, custom_objects=focal_custom or None,
        )
        freeze_batchnorm(model_b2)
        if USE_FOCAL_LOSS:
            loss_fn = CategoricalFocalLoss(gamma=FOCAL_GAMMA, label_smoothing=CLS_LABEL_SMOOTHING)
        else:
            loss_fn = tf.keras.losses.CategoricalCrossentropy(label_smoothing=CLS_LABEL_SMOOTHING)
        model_b2.compile(
            optimizer=tf.keras.optimizers.AdamW(learning_rate=B2_LR, weight_decay=CLS_WEIGHT_DECAY),
            loss=loss_fn,
            metrics=['accuracy'],
        )

        # Prefer winner sequences; rebuild hard-pair oversampled train seq when possible
        base_train = (
            (winner.get('train_seq') if 'winner' in dir() else None)
            or globals().get('train_seq')
        )
        val_gen = (
            (winner.get('val_seq') if 'winner' in dir() else None)
            or globals().get('val_seq')
            or (winner.get('vg') if 'winner' in dir() else None)
        )
        tg_flow = (winner.get('tg') if 'winner' in dir() else None) or globals().get('tg')
        if base_train is None and tg_flow is None:
            raise RuntimeError(
                'B2: no train_seq/tg in scope — train_backbone must return train_seq'
            )

        crop_cache = globals().get('yolo_crop_cache') or {}
        crop_ratio = CROP_AWARE_RATIO if crop_cache else 0.0
        preprocess_fn = (
            (winner.get('preprocess_fn') if 'winner' in dir() else None)
            or globals().get('classifier_preprocess')
        )
        alb = globals().get('alb_transform')
        tdg = globals().get('train_datagen')

        if tg_flow is not None and preprocess_fn is not None and 'OversampledSequence' in dir():
            b2_train_seq = OversampledSequence(
                tg_flow, alb, preprocess_fn,
                HARD_PAIR_CLASS_FACTORS, tg_flow.class_indices, tdg,
                crop_cache=crop_cache, crop_ratio=crop_ratio,
            )
            print(f'B2 hard-pair train samples: {len(b2_train_seq.indices)}')
        elif base_train is not None:
            b2_train_seq = base_train
            print('B2 using winner train_seq (no hard-pair rebuild)')
        else:
            raise RuntimeError('B2: unable to build train sequence')

        if val_gen is None:
            raise RuntimeError('B2: no val_seq/vg in scope')

        test_seq = winner['test_seq'] if 'winner' in dir() else globals().get('test_seq')
        if test_seq is None:
            raise RuntimeError('B2: no test_seq for re-eval')

        B2_CANDIDATE_PATH = os.path.join(WORK_DIR, 'classifier_b2_candidate.keras')
        PRE_B2_PATH = os.path.join(WORK_DIR, 'classifier_pre_b2.keras')
        shutil.copy2(CLASSIFIER_BEST_PATH, PRE_B2_PATH)
        pre_acc = float(training_report.get('classifier_test_acc') or cls_acc)
        pre_f1 = float(training_report.get('classifier_macro_f1') or 0)
        print(f'B2 baseline (pre-B2): test_acc={pre_acc:.3f} macro_f1={pre_f1:.3f} -> backup {PRE_B2_PATH}')

        cb = [
            ValMacroF1Callback(val_gen, len(val_gen), len(class_names), every_n_epochs=1),
            tf.keras.callbacks.ModelCheckpoint(
                B2_CANDIDATE_PATH, monitor='val_macro_f1', mode='max', save_best_only=True, verbose=1
            ),
            tf.keras.callbacks.EarlyStopping(
                monitor='val_macro_f1', mode='max', patience=8, restore_best_weights=True
            ),
            TimeBudgetCallback('B2'),
        ]
        model_b2.fit(
            b2_train_seq,
            validation_data=val_gen,
            epochs=B2_EPOCHS,
            callbacks=cb,
            verbose=1,
        )
        # EarlyStopping only restores best weights when it actually stops early,
        # so the ModelCheckpoint file (best val_macro_f1) is the candidate.
        if not os.path.isfile(B2_CANDIDATE_PATH):
            model_b2.save(B2_CANDIDATE_PATH)
        print('B2 candidate (best val_macro_f1):', B2_CANDIDATE_PATH)
        del model_b2

        b2_model = tf.keras.models.load_model(
            B2_CANDIDATE_PATH, compile=False, custom_objects=focal_custom or None,
        )
        y_true, y_pred = [], []
        for i in range(len(test_seq)):
            bx, by = test_seq[i]
            preds = b2_model.predict(bx, verbose=0)
            y_true.extend(np.argmax(by, axis=1))
            y_pred.extend(np.argmax(preds, axis=1))
        b2_acc = float(np.mean(np.array(y_true) == np.array(y_pred)))
        b2_f1 = float(f1_score(y_true, y_pred, average='macro'))
        b2_report = classification_report(
            y_true, y_pred, target_names=class_names, digits=3, output_dict=True,
        )
        print(f'B2 candidate test_acc={b2_acc:.3f} macro_f1={b2_f1:.3f}')
        print(classification_report(y_true, y_pred, target_names=class_names, digits=3))

        keep_b2 = b2_acc >= pre_acc and b2_f1 >= pre_f1 - 0.005
        if keep_b2:
            shutil.copy2(B2_CANDIDATE_PATH, CLASSIFIER_BEST_PATH)
            cls_test_acc = b2_acc
            training_report['classifier_test_acc'] = b2_acc
            training_report['cls_test_acc'] = b2_acc
            training_report['classifier_macro_f1'] = b2_f1
            training_report['classifier_per_class'] = {
                name: b2_report[name] for name in class_names if name in b2_report
            }
            print(f'B2 KEPT: {pre_acc:.3f} -> {b2_acc:.3f}')
        else:
            shutil.copy2(PRE_B2_PATH, CLASSIFIER_BEST_PATH)
            cls_test_acc = pre_acc
            training_report['classifier_test_acc'] = pre_acc
            training_report['cls_test_acc'] = pre_acc
            print(f'B2 REJECTED (worse than pre-B2): restored {PRE_B2_PATH} -> {CLASSIFIER_BEST_PATH}')
        del b2_model

        best_classifier = tf.keras.models.load_model(
            CLASSIFIER_BEST_PATH, compile=False, custom_objects=focal_custom or None,
        )
        training_report['b2_finetune'] = {
            'epochs': B2_EPOCHS, 'lr': B2_LR, 'hard_pair': True,
            'hard_pair_factors': dict(HARD_PAIR_CLASS_FACTORS),
            'crop_aware_ratio': crop_ratio,
            'pre_acc': pre_acc, 'pre_f1': pre_f1,
            'b2_acc': b2_acc, 'b2_f1': b2_f1,
            'kept': bool(keep_b2),
        }

if IS_BB2 and USE_TTA_EVAL:
    _tta_final = tta_metrics(best_classifier, winner['test_seq'])
    training_report['classifier_test_acc_tta'] = _tta_final['acc']
    training_report['classifier_macro_f1_tta'] = _tta_final['macro_f1']
    training_report['classifier_per_class_tta'] = _tta_final['per_class']
    training_report['tta_views'] = list(TTA_VIEWS)
    print(
        f'Final classifier — no-TTA: acc={float(training_report.get("classifier_test_acc") or 0):.3f} '
        f'f1={float(training_report.get("classifier_macro_f1") or 0):.3f} | '
        f'TTA {TTA_VIEWS}: acc={_tta_final["acc"]:.3f} f1={_tta_final["macro_f1"]:.3f}'
    )

_gate_sfx = '_tta' if (IS_BB2 and GATE_USE_TTA and 'classifier_test_acc_tta' in training_report) else ''
final_cls_acc = float(training_report.get(f'classifier_test_acc{_gate_sfx}') or 0) if IS_BB2 else 0.0
if IS_BB2:
    training_report['gate_metric'] = 'tta' if _gate_sfx else 'no_tta'
    final_cls_f1 = float(training_report.get(f'classifier_macro_f1{_gate_sfx}') or 0)
    _per_class = training_report.get(f'classifier_per_class{_gate_sfx}') or {}
    gate_checks = {
        f'test_acc {final_cls_acc:.3f} >= {TARGET_ACCURACY}': final_cls_acc >= TARGET_ACCURACY,
        f'macro_f1 {final_cls_f1:.3f} >= {TARGET_MACRO_F1}': final_cls_f1 >= TARGET_MACRO_F1,
    }
    for _cls in ('Bacterial Leaf Blight', 'Leaf Blast', 'Brown Spot'):
        _rec = float((_per_class.get(_cls) or {}).get('recall', 0) or 0)
        gate_checks[f'recall {_cls} {_rec:.3f} >= 0.75'] = _rec >= 0.75
    training_report['b_gate_checks'] = {k: bool(v) for k, v in gate_checks.items()}
    training_report['b_ready_for_c'] = all(gate_checks.values())
    print(f'=== B GATE ({training_report["gate_metric"]}) ===')
    for _name, _ok in gate_checks.items():
        print(f'  {_name}: {"PASS" if _ok else "FAIL"}')
    if training_report['b_ready_for_c']:
        print('B READY FOR C')
    else:
        print('B GATE FAIL — DO NOT run Group C')


---
# PHẦN C — Pipeline Detect → Crop → Classify
## 11. Pipeline functions

In [ ]:
if not (IS_BB2 or IS_C):
    print('Skip — only B_B2 or C')
else:
    def classify_array(cls_model, image_rgb, preprocess_fn, names):
        pil = Image.fromarray(image_rgb).resize(CLS_IMG_SIZE)
        arr = np.array(pil, dtype=np.float32)
        batch = np.expand_dims(preprocess_fn(arr), axis=0)
        probs = cls_model.predict(batch, verbose=0)[0]
        idx = int(np.argmax(probs))
        return names[idx], float(probs[idx]), probs


    def crop_with_padding(image_rgb, x1, y1, x2, y2, padding=0.15):
        h, w = image_rgb.shape[:2]
        bw, bh = x2 - x1, y2 - y1
        px, py = bw * padding, bh * padding
        nx1, ny1 = max(0, int(x1 - px)), max(0, int(y1 - py))
        nx2, ny2 = min(w, int(x2 + px)), min(h, int(y2 + py))
        if (nx2 - nx1) < MIN_CROP_SIZE or (ny2 - ny1) < MIN_CROP_SIZE:
            return None
        return image_rgb[ny1:ny2, nx1:nx2]


    def detect_crop_classify(image_path, yolo, cls_model, names, preprocess_fn, conf=0.20, padding=0.15):
        bgr = cv2.imread(str(image_path))
        if bgr is None:
            raise FileNotFoundError(image_path)
        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        results = yolo.predict(image_path, conf=conf, verbose=False)[0]
        regions = []
        if results.boxes is not None and len(results.boxes) > 0:
            for box in results.boxes:
                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                det_conf = float(box.conf[0])
                crop = crop_with_padding(rgb, x1, y1, x2, y2, padding)
                if crop is None:
                    continue
                disease, cls_conf, probs = classify_array(cls_model, crop, preprocess_fn, names)
                regions.append({
                    'bbox': (float(x1), float(y1), float(x2), float(y2)),
                    'det_conf': det_conf, 'disease': disease, 'cls_conf': cls_conf,
                    'score': det_conf * cls_conf, 'source': 'detection',
                })
        if not regions:
            disease, cls_conf, probs = classify_array(cls_model, rgb, preprocess_fn, names)
            regions.append({
                'bbox': (0, 0, rgb.shape[1], rgb.shape[0]),
                'det_conf': 0.0, 'disease': disease, 'cls_conf': cls_conf,
                'score': cls_conf, 'source': 'fallback_full_image',
            })
        best = max(regions, key=lambda r: r['score'])
        return {'image_rgb': rgb, 'regions': regions, 'prediction': best['disease'],
                'confidence': best['cls_conf'], 'used_fallback': best['source'] == 'fallback_full_image'}


    def classify_only(image_path, cls_model, names, preprocess_fn):
        bgr = cv2.imread(str(image_path))
        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        disease, conf, _ = classify_array(cls_model, rgb, preprocess_fn, names)
        return disease, conf


    def iter_test_images(test_root):
        for cls in sorted(os.listdir(test_root)):
            cp = os.path.join(test_root, cls)
            if not os.path.isdir(cp):
                continue
            true_label = normalize_class_name(cls)
            for fn in os.listdir(cp):
                if Path(fn).suffix.lower() in IMAGE_EXTENSIONS:
                    yield os.path.join(cp, fn), true_label


## 12. Full test-set E2E evaluation

In [ ]:
if not (IS_BB2 or IS_C):
    print('Skip — only B_B2 or C')
else:
    # Ensure runtime globals (Group C skips §7 merge + classifier train)
    if 'test_dir' not in globals() or not test_dir or not os.path.isdir(test_dir):
        if 'mehedi_test' in globals() and mehedi_test and os.path.isdir(mehedi_test):
            test_dir = mehedi_test
            print('Resolved test_dir <- mehedi_test:', test_dir)
        else:
            raise RuntimeError(
                'test_dir undefined. Group C: chạy audit + cell 6.7 Load Session B trước.'
            )
    if 'best_classifier' not in globals() or best_classifier is None:
        if 'cls_model' in globals() and cls_model is not None:
            best_classifier = cls_model
        elif os.path.isfile(CLASSIFIER_BEST_PATH):
            best_classifier = tf.keras.models.load_model(CLASSIFIER_BEST_PATH, compile=False)
        else:
            raise RuntimeError('best_classifier missing — load Session B checkpoint trước')
    if 'classifier_preprocess' not in globals() or classifier_preprocess is None:
        classifier_preprocess = (
            globals().get('preprocess_fn')
            or tf.keras.applications.efficientnet.preprocess_input
        )
    if 'class_names' not in globals() or not class_names:
        if 'class_indices' in globals() and isinstance(class_indices, dict) and class_indices:
            class_names = [n for n, _ in sorted(class_indices.items(), key=lambda x: x[1])]
        else:
            class_names = list(CANONICAL_CLASSES)
    if 'yolo_best' not in globals() or yolo_best is None:
        if os.path.isfile(YOLO_BEST_PATH):
            yolo_best = YOLO(YOLO_BEST_PATH)
        else:
            raise FileNotFoundError(f'Missing yolo_best for pipeline: {YOLO_BEST_PATH}')

    def eval_pipeline_on_test(conf=0.20, padding=0.15, max_images=None):
        correct_pipe, correct_cls, total = 0, 0, 0
        for i, (path, true_label) in enumerate(iter_test_images(test_dir)):
            if max_images and i >= max_images:
                break
            pipe = detect_crop_classify(path, yolo_best, best_classifier, class_names, classifier_preprocess, conf, padding)
            cls_pred, _ = classify_only(path, best_classifier, class_names, classifier_preprocess)
            total += 1
            if pipe['prediction'] == true_label:
                correct_pipe += 1
            if cls_pred == true_label:
                correct_cls += 1
        return {
            'pipeline_acc': correct_pipe / total if total else 0,
            'classifier_acc': correct_cls / total if total else 0,
            'total': total,
        }

    baseline = eval_pipeline_on_test(conf=0.20, padding=0.15)
    print(f"Classifier-only: {baseline['classifier_acc']:.2%}")
    print(f"Pipeline (conf=0.20, pad=0.15): {baseline['pipeline_acc']:.2%}")
    training_report['pipeline']['baseline'] = baseline

    def tta_probs(model, image_rgb, preprocess_fn, views):
        arr = preprocess_fn(np.array(Image.fromarray(image_rgb).resize(CLS_IMG_SIZE), dtype=np.float32))
        by_view = {'orig': arr, 'hflip': arr[:, ::-1, :], 'vflip': arr[::-1, :, :]}
        return model.predict(np.stack([by_view[v] for v in views]), verbose=0).mean(axis=0)

    def apply_output_rule(probs, names):
        probs = np.asarray(probs, dtype=np.float32)
        top = int(np.argmax(probs))
        percents = {n: 0.0 for n in names}
        percents[names[top]] = round(float(probs[top]) * 100, 1)
        return {
            'label': names[top],
            'percent': percents[names[top]],
            'is_healthy': names[top] == OUTPUT_RULE['healthy_class'],
            'percents': percents,
        }

    _demo_views = (list(training_report.get('tta_views') or TTA_VIEWS) if USE_TTA_EVAL else []) or ['orig']
    _demo, _seen = [], set()
    for _p, _true in iter_test_images(test_dir):
        if _true not in _seen:
            _seen.add(_true)
            _demo.append((_p, _true))
    _demo = _demo[:8]
    _det_names = yolo_best.names
    demo_rows = []
    fig, axes = plt.subplots(2, 4, figsize=(20, 10))
    for ax, (_p, _true) in zip(axes.flatten(), _demo):
        _rgb = cv2.cvtColor(cv2.imread(_p), cv2.COLOR_BGR2RGB)
        _out = apply_output_rule(tta_probs(best_classifier, _rgb, classifier_preprocess, _demo_views), class_names)
        ax.imshow(_rgb)
        _boxes = []
        if not (_out['is_healthy'] and OUTPUT_RULE['hide_boxes_when_healthy']):
            _r = yolo_best.predict(_p, conf=DET_DEMO_CONF, iou=DET_NMS_IOU, verbose=False)[0]
            if _r.boxes is not None and len(_r.boxes):
                for _xyxy, _c, _k in zip(
                    _r.boxes.xyxy.cpu().numpy(), _r.boxes.conf.cpu().numpy(), _r.boxes.cls.cpu().numpy()
                ):
                    _raw = _det_names[int(_k)]
                    _name = DET_DISPLAY_NAMES.get(_raw, _raw)
                    if _name is None:
                        continue
                    x1, y1, x2, y2 = (float(v) for v in _xyxy)
                    ax.add_patch(patches.Rectangle(
                        (x1, y1), x2 - x1, y2 - y1, linewidth=2, edgecolor='red', facecolor='none',
                    ))
                    ax.text(x1, max(0.0, y1 - 4), f'{_name} {float(_c):.0%}', color='white', fontsize=8,
                            bbox={'facecolor': 'red', 'alpha': 0.7, 'pad': 1})
                    _boxes.append({'label': _name, 'conf': round(float(_c), 3)})
        ax.set_title(f"true: {_true}\n-> {_out['label']} {_out['percent']:.1f}%", fontsize=9)
        demo_rows.append({
            'image': os.path.basename(_p), 'true': _true,
            'label': _out['label'], 'percent': _out['percent'], 'boxes': _boxes,
        })
    for ax in axes.flatten():
        ax.axis('off')
    fig.suptitle(f'Output rule {OUTPUT_RULE["mode"]} (TTA {_demo_views}) + detector boxes (conf >= {DET_DEMO_CONF})')
    plt.tight_layout()
    fig.savefig(os.path.join(WORK_DIR, 'output_rule_demo.png'), dpi=80)
    plt.show()
    plt.close(fig)
    for _row in demo_rows:
        print(f"  {_row['image'][:28]:<28} true={_row['true']:<24} -> {_row['label']} {_row['percent']:.1f}% "
              f"| boxes {[b['label'] for b in _row['boxes']]}")
    training_report['output_rule_demo'] = demo_rows


## 12.5 THRESHOLD_SEARCH

In [ ]:
if not (IS_BB2 or IS_C):
    print('Skip — only B_B2 or C')
else:
    best_thresh = {'acc': 0, 'conf': 0.20, 'padding': 0.15}
    for conf in THRESHOLD_GRID['conf']:
        for pad in THRESHOLD_GRID['padding']:
            res = eval_pipeline_on_test(conf=conf, padding=pad)
            print(f'conf={conf:.2f} pad={pad:.2f} → pipeline={res["pipeline_acc"]:.2%}')
            if res['pipeline_acc'] > best_thresh['acc']:
                best_thresh = {'acc': res['pipeline_acc'], 'conf': conf, 'padding': pad, 'classifier_acc': res['classifier_acc']}

    print(f'\nBest threshold: conf={best_thresh["conf"]} padding={best_thresh["padding"]} pipeline={best_thresh["acc"]:.2%}')
    BEST_CONF = best_thresh['conf']
    BEST_PADDING = best_thresh['padding']
    pipeline_acc = best_thresh['acc']
    training_report['pipeline']['best_threshold'] = best_thresh
    training_report['pipeline']['best_acc'] = best_thresh['acc']
    training_report['pipeline']['acc'] = best_thresh['acc']


---
# PHẦN D — Export + GATE_CHECK
## 13. Export artifacts

In [ ]:
if not (IS_BB2 or IS_C):
    print('Skip — only B_B2 or C')
else:
    # Restore scalars skipped in Group C (YOLO/classifier train cells not run).
    # Values come from training_report_partial.json inside session_b_checkpoint — do not invent.
    _det = training_report.get('detection') if isinstance(training_report.get('detection'), dict) else {}
    if 'det_map50' not in globals():
        if 'map50' not in _det:
            raise KeyError(
                'detection.map50 missing in training_report — '
                'Session B zip phải có training_report_partial.json với detection.map50'
            )
        det_map50 = float(_det['map50'])
        det_map5095 = float(_det.get('map50_95') or 0)
        det_precision = float(_det.get('precision') or 0)
        det_recall = float(_det.get('recall') or 0)
    if float(det_map50) <= 0:
        raise ValueError('det_map50 invalid after restore: %r' % (det_map50,))
    if 'cls_test_acc' not in globals():
        cls_test_acc = float(
            training_report.get('cls_test_acc')
            or training_report.get('classifier_test_acc')
            or 0
        )
        if cls_test_acc <= 0 and training_report.get('classifier_leaderboard'):
            cls_test_acc = float(training_report['classifier_leaderboard'][0].get('test_acc', 0) or 0)
    if 'cls_val_acc' not in globals():
        _lb = training_report.get('classifier_leaderboard') or []
        if _lb and isinstance(_lb[0], dict) and _lb[0].get('val_acc') is not None:
            cls_val_acc = float(_lb[0]['val_acc'])
        else:
            cls_val_acc = float(training_report.get('classifier_val_acc') or 0)
    if 'BEST_CONF' not in globals():
        _bt = (training_report.get('pipeline') or {}).get('best_threshold') or {}
        BEST_CONF = float(_bt.get('conf') if _bt.get('conf') is not None else 0.20)
        BEST_PADDING = float(_bt.get('padding') if _bt.get('padding') is not None else 0.15)
        pipeline_acc = float(
            _bt.get('acc')
            if _bt.get('acc') is not None
            else (training_report.get('pipeline') or {}).get('acc') or 0
        )
    print(
        'Restored checkpoint metrics: det_map50=%.4f cls_test_acc=%.4f cls_val_acc=%.4f'
        % (float(det_map50), float(cls_test_acc), float(cls_val_acc))
    )

    export_yolo = os.path.join(PIPELINE_DIR, 'yolo_best.pt')
    export_cls = os.path.join(PIPELINE_DIR, 'classifier_best.keras')
    shutil.copy2(YOLO_BEST_PATH, export_yolo)
    shutil.copy2(CLASSIFIER_BEST_PATH, export_cls)

    with open(os.path.join(PIPELINE_DIR, 'class_indices.json'), 'w', encoding='utf-8') as f:
        json.dump(class_indices, f, indent=2, ensure_ascii=False)

    pipeline_config = {
        'version': 'v2',
        'detection': {
            'framework': f'ultralytics-{DET_ARCH}', 'model_file': 'yolo_best.pt',
            'base_model': YOLO_MODEL, 'imgsz': DET_IMG_SIZE,
            'conf_thresh': BEST_CONF, 'padding': BEST_PADDING,
            'map50': det_map50,
        },
        'classification': {
            'framework': 'tensorflow-keras', 'model_file': 'classifier_best.keras',
            'backbone': winner['name'], 'input_size': list(CLS_IMG_SIZE),
            'preprocessing': winner['name'],
            'test_accuracy': cls_test_acc, 'val_accuracy': cls_val_acc,
        },
        'class_names': class_names,
        'num_classes': len(class_names),
        'pipeline_test_accuracy': pipeline_acc,
        'target_accuracy': TARGET_ACCURACY,
    }

    with open(os.path.join(PIPELINE_DIR, 'pipeline_config.json'), 'w', encoding='utf-8') as f:
        json.dump(pipeline_config, f, indent=2, ensure_ascii=False)

    training_report['gate'] = {
        'classifier_test_acc': cls_test_acc,
        'pipeline_test_acc': pipeline_acc,
        'target': TARGET_ACCURACY,
        'passed': cls_test_acc >= TARGET_ACCURACY,
    }
    with open(REPORT_PATH, 'w', encoding='utf-8') as f:
        json.dump(training_report, f, indent=2, ensure_ascii=False)
    shutil.copy2(REPORT_PATH, os.path.join(PIPELINE_DIR, 'training_report.json'))

    print('Exported to', PIPELINE_DIR)
    for p in sorted(Path(PIPELINE_DIR).rglob('*')):
        if p.is_file():
            print(f'  {p.name} ({p.stat().st_size/1024/1024:.1f} MB)')


## 14. GATE_CHECK

In [ ]:
PER_CLASS_THRESHOLDS = {
    'Bacterial Leaf Blight': {'recall': 0.75},
    'Leaf Blast': {'recall': 0.75},
    'Brown Spot': {'recall': 0.75},
    'Healthy Rice Leaf': {'precision': 0.80},
    'Narrow Brown Leaf Spot': {'precision': 0.40},
    'Leaf scald': {'recall': 0.50},
}

if IS_0A:
    print('Skip GATE_CHECK — Group 0A')
else:
    cls_test_acc = float(
        training_report.get('cls_test_acc')
        or training_report.get('classifier_test_acc')
        or 0
    )
    if cls_test_acc <= 0 and training_report.get('classifier_leaderboard'):
        cls_test_acc = float(training_report['classifier_leaderboard'][0].get('test_acc', 0) or 0)
    macro_f1 = float(training_report.get('classifier_macro_f1') or training_report.get('macro_f1') or 0)
    pipe = training_report.get('pipeline') or {}
    bt = pipe.get('best_threshold') or {}
    pipe_acc = float(
        bt.get('acc') or pipe.get('best_acc') or pipe.get('acc') or 0
    )
    gate_use_tta = bool(GATE_USE_TTA and training_report.get('classifier_test_acc_tta') is not None)
    gate_acc = float(training_report['classifier_test_acc_tta']) if gate_use_tta else cls_test_acc
    gate_f1 = float(training_report.get('classifier_macro_f1_tta') or 0) if gate_use_tta else macro_f1
    passed = gate_acc >= TARGET_ACCURACY
    print('=== GATE_CHECK v3 ===', '(TTA)' if gate_use_tta else '(no TTA)')
    print(f'Classifier test acc: {gate_acc:.3f} (need >= {TARGET_ACCURACY}) ->', 'PASS' if passed else 'FAIL')
    if gate_use_tta:
        print(f'  no-TTA reference: test acc {cls_test_acc:.3f} | macro-F1 {macro_f1:.3f}')
    print(f'Macro-F1: {gate_f1:.3f} (need >= {TARGET_MACRO_F1})')
    if gate_f1 and gate_f1 < TARGET_MACRO_F1:
        passed = False
    print(f'Pipeline E2E: {pipe_acc:.3f} (target >= {TARGET_ACCURACY})')
    per_class = (
        (training_report.get('classifier_per_class_tta') if gate_use_tta else None)
        or training_report.get('classifier_per_class') or {}
    )
    for cls_name, thr in PER_CLASS_THRESHOLDS.items():
        metrics = per_class.get(cls_name) or {}
        for metric, need in thr.items():
            got = float(metrics.get(metric, 0) or 0)
            ok = got >= need
            print(f'  {cls_name} {metric}: {got:.3f} (need >= {need}) ->', 'OK' if ok else 'WARN')
            if cls_name in ('Bacterial Leaf Blight', 'Leaf Blast', 'Brown Spot', 'Healthy Rice Leaf') and not ok:
                passed = False
    training_report['gate'] = {
        'passed': bool(passed),
        'target_accuracy': TARGET_ACCURACY,
        'cls_test_acc': cls_test_acc,
        'macro_f1': macro_f1,
        'pipeline_acc': pipe_acc,
        'gate_use_tta': gate_use_tta,
        'gate_acc': gate_acc,
        'gate_macro_f1': gate_f1,
    }
    with open(REPORT_PATH, 'w', encoding='utf-8') as f:
        json.dump(training_report, f, indent=2, ensure_ascii=False)
    print('GATE overall:', 'PASS' if passed else 'FAIL')


## 15. Export Group B_B2 → `rice-v3-session-b-checkpoint`


In [ ]:
if not IS_BB2:
    print('Skip Group B_B2 export')
else:
    import zipfile
    os.makedirs(SESSION_B_EXPORT_DIR, exist_ok=True)
    for src, name in [(CLASSIFIER_BEST_PATH, 'classifier_best.keras'), (YOLO_BEST_PATH, 'yolo_best.pt')]:
        if os.path.isfile(src):
            shutil.copy2(src, os.path.join(SESSION_B_EXPORT_DIR, name))
    ci_path = os.path.join(PIPELINE_DIR, 'class_indices.json')
    if os.path.isfile(ci_path):
        shutil.copy2(ci_path, os.path.join(SESSION_B_EXPORT_DIR, 'class_indices.json'))
    elif 'class_indices' in globals():
        with open(os.path.join(SESSION_B_EXPORT_DIR, 'class_indices.json'), 'w', encoding='utf-8') as f:
            json.dump(class_indices, f, indent=2, ensure_ascii=False)
    draft = {
        'backbone': CLS_BACKBONES[0],
        'img_size': list(CLS_IMG_SIZE),
        'reject_threshold': REJECT_THRESHOLD,
        'temperature': TEMPERATURE,
        'best_conf': globals().get('BEST_CONF', 0.20),
        'best_padding': globals().get('BEST_PADDING', 0.15),
        'preprocess': 'efficientnet',
        'tta_views': list(TTA_VIEWS) if USE_TTA_EVAL else [],
    }
    with open(os.path.join(SESSION_B_EXPORT_DIR, 'pipeline_config_draft.json'), 'w', encoding='utf-8') as f:
        json.dump(draft, f, indent=2)
    with open(os.path.join(SESSION_B_EXPORT_DIR, 'training_report_partial.json'), 'w', encoding='utf-8') as f:
        json.dump(training_report, f, indent=2, ensure_ascii=False)
    with zipfile.ZipFile(SESSION_B_ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zf:
        for fn in os.listdir(SESSION_B_EXPORT_DIR):
            zf.write(os.path.join(SESSION_B_EXPORT_DIR, fn), arcname=fn)
    print('Wrote', SESSION_B_ZIP_PATH)
    print('=== UPLOAD CHECKLIST (Group B_B2) ===')
    print(f'1. Save Version → download zip → upload {SESSION_B_UPLOAD_SLUG}')
    if training_report.get('b_ready_for_c'):
        print(f'2. B GATE PASS: Add Data {SESSION_B_UPLOAD_SLUG} (already top of SESSION_B_INPUT_CANDIDATES), TRAINING_GROUP=\"C\"')
    else:
        print('2. B GATE FAIL: DO NOT run Group C — send training_report_partial.json for analysis')


## 16a. Group C — detector TFLite (float32 + INT8)


In [ ]:
if not (IS_C and EXPORT_DET_TFLITE):
    print('Skip detector TFLite export — needs TRAINING_GROUP C and EXPORT_DET_TFLITE')
else:
    import glob
    import re
    import socket
    from importlib.metadata import PackageNotFoundError, version as pkg_version

    def _pkg(name):
        try:
            return pkg_version(name)
        except PackageNotFoundError:
            return 'not installed'

    det_versions = {n: _pkg(n) for n in ('ultralytics', 'tensorflow', 'onnx2tf', 'tf_keras')}
    print('Detector export versions:', det_versions)
    if tuple(int(x) for x in re.findall(r'\d+', det_versions['ultralytics'])[:3]) >= (8, 4, 83):
        raise RuntimeError(
            f"ultralytics {det_versions['ultralytics']} replaced format='tflite' with LiteRT (other file names, "
            'other input layout). Restart the session so the pinned pip cell (ultralytics<8.4.83) is imported, '
            'then Run All.'
        )
    import importlib
    import importlib.util
    import subprocess
    import sys

    # Ultralytics' own auto-install adds the NGC index, which Kaggle cannot resolve, so install from plain PyPI.
    # Never tf_keras / tensorflow here: Kaggle's TF 2.20 must stay.
    DET_EXPORT_DEPS = {
        'onnx2tf': 'onnx2tf>=1.26.3,<1.29.0',
        'sng4onnx': 'sng4onnx>=1.0.1',
        'onnx_graphsurgeon': 'onnx-graphsurgeon>=0.3.26',
        'ai_edge_litert': 'ai-edge-litert>=1.2.0',
        'onnxslim': 'onnxslim>=0.1.82',
        'onnxruntime': 'onnxruntime',
    }
    _missing = [m for m in DET_EXPORT_DEPS if importlib.util.find_spec(m) is None]
    try:
        socket.create_connection(('pypi.org', 443), timeout=5).close()
        _online = True
    except OSError:
        _online = False
    if _online:
        _pip = subprocess.run(
            [sys.executable, '-m', 'pip', 'install', '-q', '-c', PIP_CONSTRAINTS, *DET_EXPORT_DEPS.values()],
            capture_output=True, text=True,
        )
        if _pip.returncode != 0:
            print(_pip.stdout[-3000:], _pip.stderr[-3000:])
            raise RuntimeError('pip install of the detector export deps failed (output above)')
    elif _missing:
        raise RuntimeError(f'No Internet and missing {_missing}: Settings -> Internet ON, then re-run this cell.')
    importlib.invalidate_caches()
    for _m in ('onnx_graphsurgeon', 'sng4onnx'):
        try:
            importlib.import_module(_m)
        except Exception as e:
            raise RuntimeError(f'{_m} still not importable after pip install: {e}') from e
    try:
        importlib.import_module('onnx2tf')
    except Exception as e:
        print(f'WARNING: onnx2tf 1.x import fails ({e}); the Ultralytics export will fall back to onnx2tf 2.3.x')
    det_versions.update({
        n: _pkg(n) for n in ('onnx2tf', 'onnx-graphsurgeon', 'sng4onnx', 'ai-edge-litert', 'onnxslim', 'onnxruntime')
    })
    print('Detector export deps:', det_versions)

    det_eval = 'DET_YAML' in globals() and os.path.isfile(DET_YAML)
    if not det_eval:
        if DET_EVAL_REQUIRED:
            raise RuntimeError(
                'det_data.yaml missing (DET_EVAL_REQUIRED = True): Add Data yusufmurtaza01/rice-leaf-diseases, '
                'then re-run from the Config cell (DET_ROOT is resolved there).'
            )
        print('WARNING: no det_data.yaml (YOLO dataset not mounted): float32 export + smoke test only, '
              'mAP50 / parity skipped. Add Data yusufmurtaza01/rice-leaf-diseases to measure them.')
    assert os.path.isfile(YOLO_BEST_PATH), YOLO_BEST_PATH
    det_dir = os.path.join(WORK_DIR, 'det_export')
    det_out = os.path.join(det_dir, 'out')
    os.makedirs(det_out, exist_ok=True)
    det_pt = os.path.join(det_dir, 'yolo_best.pt')
    shutil.copy2(YOLO_BEST_PATH, det_pt)

    def _tree(root):
        for p in sorted(Path(root).rglob('*')):
            if p.is_file():
                print(f'  {p.relative_to(root)} ({p.stat().st_size / 1024**2:.1f} MB)')

    def _pick(paths, kind):
        names = {p: os.path.basename(p).lower() for p in paths}
        if kind == 'int8':
            return [p for p in paths if 'int8' in names[p]]
        return [p for p in paths if 'float32' in names[p]] or [
            p for p in paths if not any(t in names[p] for t in ('int8', 'float16', 'w8a'))
        ]

    def export_det(kind):
        # Each export rebuilds yolo_best_saved_model/, so the result is copied out before the next one.
        shutil.rmtree(os.path.join(det_dir, 'yolo_best_saved_model'), ignore_errors=True)
        kwargs = {'int8': True, 'data': det_calib_yaml} if kind == 'int8' else {'half': False, 'int8': False}
        t0 = time.time()
        ret = YOLO(det_pt).export(format='tflite', imgsz=DET_IMG_SIZE, **kwargs)
        ret = str(ret) if ret else ''
        if ret.endswith('.tflite') and os.path.isfile(ret):
            found = [ret]
        elif ret and os.path.isdir(ret):
            found = glob.glob(os.path.join(ret, '**', '*.tflite'), recursive=True)
        else:
            found = []
        if not found:
            found = [
                h for root in (det_dir, WORK_DIR)
                for h in glob.glob(os.path.join(root, '**', '*.tflite'), recursive=True)
                if os.path.getmtime(h) >= t0
            ]
        out_prefix = os.path.abspath(det_out) + os.sep
        found = [h for h in dict.fromkeys(os.path.abspath(h) for h in found) if not h.startswith(out_prefix)]
        hits = _pick(found, kind)
        if not hits:
            print(f'export() returned: {ret!r}; new .tflite files: {found}')
            print('Versions:', det_versions)
            print(f'{det_dir} tree:')
            _tree(det_dir)
            raise FileNotFoundError(
                f'No {kind} .tflite after export in {det_dir}. Look for "export failure" in the '
                '"TensorFlow SavedModel:" / "TensorFlow Lite:" log lines above '
                '(usually onnx2tf / tf_keras vs the Kaggle TensorFlow version).'
            )
        dst = os.path.join(det_out, f'rice_det_{DET_ARCH}_{kind}.tflite')
        shutil.copy2(hits[0], dst)
        print(f'Detector {kind}: {dst} <- {hits[0]} ({os.path.getsize(dst) / 1024**2:.1f} MB)')
        return dst

    def export_det_onnx2tf():
        onnx_path = os.path.join(det_dir, 'yolo_best.onnx')
        if not os.path.isfile(onnx_path):
            onnx_path = str(YOLO(det_pt).export(format='onnx', imgsz=DET_IMG_SIZE, simplify=True))
        fb_dir = os.path.join(det_dir, 'onnx2tf_out')
        shutil.rmtree(fb_dir, ignore_errors=True)
        # Subprocess: the onnx2tf 1.x already imported in this kernel must not mix with the 2.x files.
        steps = [
            [sys.executable, '-m', 'pip', 'install', '-q', '-c', PIP_CONSTRAINTS, 'onnx2tf>=2.3.0,<2.3.16'],
            [
                sys.executable, '-c',
                'import sys; from onnx2tf import convert; '
                'convert(input_onnx_file_path=sys.argv[1], output_folder_path=sys.argv[2], non_verbose=True)',
                onnx_path, fb_dir,
            ],
        ]
        logs = []
        for cmd in steps:
            r = subprocess.run(cmd, capture_output=True, text=True)
            logs.append(f'$ {" ".join(cmd[:5])[:200]} -> exit {r.returncode}\n{r.stdout[-2000:]}\n{r.stderr[-4000:]}')
            if r.returncode != 0:
                break
        det_versions['onnx2tf_fallback'] = _pkg('onnx2tf')
        hits = sorted(glob.glob(os.path.join(fb_dir, '*_float32.tflite')))
        if not hits:
            print('\n'.join(logs))
            print('Versions:', det_versions)
            print(f'{det_dir} tree:')
            _tree(det_dir)
            raise FileNotFoundError(f'onnx2tf fallback produced no *_float32.tflite in {fb_dir}')
        dst = os.path.join(det_out, f'rice_det_{DET_ARCH}_float32.tflite')
        shutil.copy2(hits[0], dst)
        print(f'Detector float32 (onnx2tf {det_versions["onnx2tf_fallback"]}): {dst} <- {hits[0]} '
              f'({os.path.getsize(dst) / 1024**2:.1f} MB)')
        return dst

    det_export_path = 'ultralytics'
    _f32 = os.path.join(det_out, f'rice_det_{DET_ARCH}_float32.tflite')
    if os.path.isfile(_f32) and os.path.getmtime(_f32) >= os.path.getmtime(YOLO_BEST_PATH):
        det_files = {'float32': _f32}
        print('Reusing float32 detector from an earlier run in this session:', _f32)
    else:
        try:
            det_files = {'float32': export_det('float32')}
        except Exception as e:
            print(f'Ultralytics float32 export failed ({type(e).__name__}: {e}); '
                  'falling back to onnx2tf 2.3.x on the ONNX file')
            det_export_path = 'onnx2tf_direct'
            det_files = {'float32': export_det_onnx2tf()}
    print('Detector export path:', det_export_path)
    _itp = tf.lite.Interpreter(model_path=det_files['float32'])
    try:
        _itp.allocate_tensors()
    except Exception as e:
        raise RuntimeError(f'float32 detector TFLite fails allocate_tensors(): {e}') from e
    _shape = [int(x) for x in _itp.get_input_details()[0]['shape']]
    if len(_shape) != 4 or _shape[3] != 3:
        raise RuntimeError(f'Detector TFLite input must be NHWC [1, H, W, 3] (app contract); got {_shape}')
    print('float32 detector smoke OK, input', _shape)
    del _itp
    import gc
    import random

    det_int8_error = None
    det_calib_n = 0
    _i8 = os.path.join(det_out, f'rice_det_{DET_ARCH}_int8.tflite')
    if not EXPORT_DET_INT8:
        print('Skip detector INT8 export (EXPORT_DET_INT8 = False); float32 only')
    elif not det_eval:
        print('Skip detector INT8 export: calibration needs det_data.yaml; float32 only')
    elif os.path.isfile(_i8) and os.path.getmtime(_i8) >= os.path.getmtime(YOLO_BEST_PATH):
        det_files['int8'] = _i8
        print('Reusing int8 detector from an earlier run in this session:', _i8)
    else:
        with open(DET_YAML, encoding='utf-8') as f:
            _ccfg = yaml.safe_load(f)
        _cval = str(_ccfg.get('val') or _ccfg.get('valid'))
        _cval = _cval if os.path.isabs(_cval) else os.path.join(_ccfg['path'], _cval)
        _call = sorted(str(p) for p in Path(_cval).rglob('*') if p.suffix.lower() in IMAGE_EXTENSIONS)
        _cimgs = random.Random(42).sample(_call, min(DET_INT8_CALIB_IMAGES, len(_call)))
        det_calib_n = len(_cimgs)
        _clist = os.path.join(det_dir, 'calib_images.txt')
        with open(_clist, 'w', encoding='utf-8') as f:
            f.write('\n'.join(_cimgs) + '\n')
        _ccfg['val'] = _clist
        _ccfg.pop('valid', None)
        det_calib_yaml = os.path.join(det_dir, 'det_calib.yaml')
        with open(det_calib_yaml, 'w', encoding='utf-8') as f:
            yaml.dump(_ccfg, f, default_flow_style=False)
        print(f'INT8 calibration: {det_calib_n} of {len(_call)} val images (seed 42) -> {det_calib_yaml}')
        try:
            det_files['int8'] = export_det('int8')
        except Exception as e:
            det_int8_error = f'{type(e).__name__}: {e}'
            print('WARNING: INT8 detector export failed, float32 only:', det_int8_error)
        gc.collect()

    val_images = []
    if det_eval:
        with open(DET_YAML, encoding='utf-8') as f:
            _dcfg = yaml.safe_load(f)
        _val_rel = str(_dcfg.get('val') or _dcfg.get('valid'))
        _val_dir = _val_rel if os.path.isabs(_val_rel) else os.path.join(_dcfg['path'], _val_rel)
        val_images = sorted(str(p) for p in Path(_val_dir).rglob('*') if p.suffix.lower() in IMAGE_EXTENSIONS)
        assert val_images, f'No val images in {_val_dir}'
    probe_image = val_images[0] if val_images else next(iter_test_images(test_dir))[0]

    def tensor_io(path):
        itp = tf.lite.Interpreter(model_path=path)
        itp.allocate_tensors()

        def desc(d):
            return {
                'dtype': np.dtype(d['dtype']).name,
                'shape': [int(x) for x in d['shape']],
                'quantization': [float(d['quantization'][0]), int(d['quantization'][1])],
            }

        return itp, desc(itp.get_input_details()[0]), desc(itp.get_output_details()[0])

    def coords_normalized(itp):
        inp, out = itp.get_input_details()[0], itp.get_output_details()[0]
        h, w = int(inp['shape'][1]), int(inp['shape'][2])
        x = np.array(Image.open(probe_image).convert('RGB').resize((w, h)), dtype=np.float32)[None] / 255.0
        s, zp = inp['quantization']
        if np.issubdtype(inp['dtype'], np.integer) and s:
            info = np.iinfo(inp['dtype'])
            x = np.clip(np.round(x / s + zp), info.min, info.max)
        itp.set_tensor(inp['index'], x.astype(inp['dtype']))
        itp.invoke()
        y = itp.get_tensor(out['index'])[0].astype(np.float32)
        s, zp = out['quantization']
        if np.issubdtype(out['dtype'], np.integer) and s:
            y = (y - zp) * s
        if y.shape[0] > y.shape[1]:
            y = y.T
        best = int(np.argmax(y[4:].max(axis=0)))
        return bool(np.abs(y[:4, best]).max() <= 1.5)

    def pr_tune(box):
        px = np.asarray(box.px)
        present = [int(c) for c in np.asarray(box.ap_class_index).tolist()]
        rows = [i for i, c in enumerate(present) if DET_CLASS_MAP.get(det_names[c]) not in (None, 'Healthy Rice Leaf')]
        if not rows:
            raise RuntimeError(f'No mapped disease class in the val PR curves (present: {present})')
        p = np.asarray(box.p_curve)[rows].mean(0)
        r = np.asarray(box.r_curve)[rows].mean(0)
        f1 = np.asarray(box.f1_curve)[rows].mean(0)

        def at(j):
            return {
                'conf': round(float(px[j]), 3), 'precision': round(float(p[j]), 4),
                'recall': round(float(r[j]), 4), 'f1': round(float(f1[j]), 4),
            }

        reach = np.where(r >= DET_MIN_RECALL)[0]
        return {
            'classes': [det_names[present[i]] for i in rows],
            'f1': at(int(np.argmax(f1))),
            'recall_target': DET_MIN_RECALL,
            # Recall falls as conf rises: the last index still reaching the target is the highest such conf.
            'at_recall': at(int(reach.max())) if len(reach) else None,
        }

    def det_val(path):
        m = YOLO(path, task='detect').val(
            data=DET_YAML, imgsz=DET_IMG_SIZE, batch=1, split='val', plots=False, verbose=False,
        )
        return {'map50': float(m.box.map50), 'pr': pr_tune(m.box)}

    def predict_boxes(model, img):
        r = model.predict(img, imgsz=DET_IMG_SIZE, conf=det_parity_conf, iou=DET_NMS_IOU, verbose=False)[0]
        h, w = r.orig_shape
        if r.boxes is None or len(r.boxes) == 0:
            return np.zeros((0, 4)), np.zeros(0), float(h * w)
        return r.boxes.xyxy.cpu().numpy(), r.boxes.conf.cpu().numpy(), float(h * w)

    def box_iou(a, b):
        ix = max(0.0, min(a[2], b[2]) - max(a[0], b[0]))
        iy = max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
        inter = ix * iy
        union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
        return inter / union if union > 0 else 0.0

    pt_model = YOLO(det_pt)
    det_names = [pt_model.names[k] for k in sorted(pt_model.names)]
    print('Detection classes:', det_names)
    if set(det_names) != set(DET_CLASS_MAP):
        raise RuntimeError(f'Detector classes {det_names} != DET_CLASS_MAP keys {sorted(DET_CLASS_MAP)}; update DET_CLASS_MAP')
    _bad_targets = {v for v in DET_CLASS_MAP.values() if v is not None} - set(CANONICAL_CLASSES)
    if _bad_targets:
        raise RuntimeError(f'DET_CLASS_MAP targets not in CANONICAL_CLASSES: {sorted(_bad_targets)}')
    if set(det_names) != set(DET_DISPLAY_NAMES):
        raise RuntimeError(f'Detector classes {det_names} != DET_DISPLAY_NAMES keys {sorted(DET_DISPLAY_NAMES)}')
    print('Detector -> classifier class map:', {n: DET_CLASS_MAP[n] for n in det_names})
    print('Detector box labels (None = not drawn):', {n: DET_DISPLAY_NAMES[n] for n in det_names})
    val_pt = det_val(det_pt) if det_eval else None
    map_pt = val_pt['map50'] if det_eval else None
    det_parity_conf = val_pt['pr']['f1']['conf'] if det_eval else None
    parity_imgs = val_images[:DET_PARITY_IMAGES]
    pt_preds = [predict_boxes(pt_model, p) for p in parity_imgs]

    def parity(path):
        m = YOLO(path, task='detect')
        ref_n = hit_n = ref_small = hit_small = extra = 0
        conf_diff = []
        for img, (ref_xyxy, ref_conf, area) in zip(parity_imgs, pt_preds):
            xyxy, conf, _ = predict_boxes(m, img)
            used = set()
            for i in range(len(ref_xyxy)):
                bw, bh = ref_xyxy[i, 2] - ref_xyxy[i, 0], ref_xyxy[i, 3] - ref_xyxy[i, 1]
                small = bw * bh / area < DET_SMALL_BOX_AREA
                ref_n += 1
                ref_small += int(small)
                best_j, best_v = -1, 0.5
                for j in range(len(xyxy)):
                    if j in used:
                        continue
                    v = box_iou(ref_xyxy[i], xyxy[j])
                    if v >= best_v:
                        best_j, best_v = j, v
                if best_j >= 0:
                    used.add(best_j)
                    hit_n += 1
                    hit_small += int(small)
                    conf_diff.append(abs(float(conf[best_j]) - float(ref_conf[i])))
            extra += len(xyxy) - len(used)
        n = len(parity_imgs)
        return {
            'images': n,
            'boxes_pt': ref_n,
            'recall_vs_pt': hit_n / ref_n if ref_n else 1.0,
            'small_boxes_pt': ref_small,
            'small_recall_vs_pt': hit_small / ref_small if ref_small else 1.0,
            'extra_boxes_per_image': extra / n if n else 0.0,
            'mean_conf_diff': float(np.mean(conf_diff)) if conf_diff else 0.0,
        }

    det_results = {}
    for kind, path in det_files.items():
        itp, d_in, d_out = tensor_io(path)
        _v = det_val(path) if det_eval else None
        det_results[kind] = {
            'file': os.path.basename(path),
            'mb': round(os.path.getsize(path) / 1024**2, 2),
            'input': d_in,
            'output': d_out,
            'coords_normalized': coords_normalized(itp),
            'map50': _v['map50'] if _v else None,
            'pr': _v['pr'] if _v else None,
            'parity': parity(path) if det_eval else None,
        }
        del itp
        gc.collect()
        print(f'{kind}: in={d_in} out={d_out}')

    if det_eval:
        print(f'\nDetector TFLite vs .pt (val mAP50 .pt = {map_pt:.3f}; parity on {len(parity_imgs)} val images, '
              f'conf={det_parity_conf} (.pt max-F1) iou={DET_NMS_IOU}):')
        print(f'  {"kind":<8} {"MB":>6} {"mAP50":>7} {"drop":>7} {"recall":>7} {"small":>7} {"extra/img":>10} {"dconf":>7}')
        for kind, r in det_results.items():
            p = r['parity']
            print(f'  {kind:<8} {r["mb"]:>6.1f} {r["map50"]:>7.3f} {map_pt - r["map50"]:>+7.3f} '
                  f'{p["recall_vs_pt"]:>7.3f} {p["small_recall_vs_pt"]:>7.3f} '
                  f'{p["extra_boxes_per_image"]:>10.2f} {p["mean_conf_diff"]:>7.3f}')
        print(f'\nPR curve, mean over {val_pt["pr"]["classes"]}:')
        for kind, pr in [('pt', val_pt['pr'])] + [(k, r['pr']) for k, r in det_results.items()]:
            f1, ar = pr['f1'], pr['at_recall']
            ar_s = f'conf={ar["conf"]:.3f} P={ar["precision"]:.3f} F1={ar["f1"]:.3f}' if ar else 'not reached'
            print(f'  {kind:<8} max-F1 conf={f1["conf"]:.3f} P={f1["precision"]:.3f} R={f1["recall"]:.3f} '
                  f'F1={f1["f1"]:.3f} | recall>={DET_MIN_RECALL}: {ar_s}')
    else:
        print('\nSkip detector mAP50/parity (no YOLO dataset):')
        for kind, r in det_results.items():
            print(f'  {kind:<8} {r["mb"]:>6.1f} MB  mAP50 n/a  parity n/a  coords_normalized={r["coords_normalized"]}')
    int8_ok = bool(
        det_eval
        and 'int8' in det_results
        and map_pt - det_results['int8']['map50'] <= DET_INT8_MAX_MAP50_DROP
        and det_results['int8']['parity']['small_recall_vs_pt'] >= 0.95
    )
    print(f'INT8 recommended: {"yes" if int8_ok else "no"} '
          f'(needs mAP50 drop <= {DET_INT8_MAX_MAP50_DROP} and small-box recall vs .pt >= 0.95)')

    det_ship_kind = ('int8' if int8_ok else 'float32') if DET_TFLITE_SHIP == 'auto' else DET_TFLITE_SHIP
    if det_ship_kind == 'float16' or det_ship_kind not in det_results:
        raise ValueError(f'DET_TFLITE_SHIP={DET_TFLITE_SHIP!r} -> {det_ship_kind!r} not shippable; have {list(det_results)}')
    ship = det_results[det_ship_kind]
    shutil.copy2(det_files[det_ship_kind], os.path.join(PIPELINE_DIR, ship['file']))
    print(f'Shipping detector: {ship["file"]} ({det_ship_kind}, DET_TFLITE_SHIP={DET_TFLITE_SHIP!r}); '
          f'others kept in {det_out}')
    if DET_CONF_SOURCE == 'pr_f1' and ship['pr']:
        det_conf_source, det_pr_point = 'pr_f1', ship['pr']['f1']
        det_conf_thresh = det_pr_point['conf']
    else:
        det_conf_source, det_pr_point, det_conf_thresh = 'pipeline_grid', None, BEST_CONF
    print(f'Detector conf_thresh = {det_conf_thresh} ({det_conf_source}); pipeline crop conf (cell 12.5) = {BEST_CONF}')

    det_tflite_info = {
        'ship_file': ship['file'],
        'config': {
            'model_file': ship['file'],
            'quantization': det_ship_kind,
            'input': ship['input'],
            'output': ship['output'],
            'imgsz': DET_IMG_SIZE,
            'letterbox': True,
            'input_range': [0, 1],
            'coords_normalized': ship['coords_normalized'],
            'output_layout': 'xywh + class scores, no NMS',
            'conf_thresh': det_conf_thresh,
            'conf_source': det_conf_source,
            'pr_point': det_pr_point,
            'pipeline_crop_conf': BEST_CONF,
            'nms_iou': DET_NMS_IOU,
            'crop_padding': BEST_PADDING,
            'class_names': det_names,
            'class_map': {n: DET_CLASS_MAP[n] for n in det_names},
            'out_of_scope_classes': [n for n in det_names if DET_CLASS_MAP[n] is None],
            'classifier_classes_without_detector_label': [
                c for c in CANONICAL_CLASSES if c not in set(DET_CLASS_MAP.values())
            ],
            'labels_informational_only': False,
            'box_label_source': 'detector_class',
            'display_names': {n: DET_DISPLAY_NAMES[n] for n in det_names},
            'hidden_classes': [n for n in det_names if DET_DISPLAY_NAMES[n] is None],
            'hide_boxes_when_healthy': OUTPUT_RULE['hide_boxes_when_healthy'],
            'map50_pt': map_pt,
            'map50_tflite': ship['map50'],
            'export_path': det_export_path,
            'export_versions': det_versions,
            'det_eval': det_eval,
            'int8_recommended': int8_ok,
            'label_audit_verdict': (globals().get('det_label_audit') or {}).get('verdict'),
        },
    }
    training_report['detector_tflite'] = {
        'map50_pt': map_pt,
        'results': det_results,
        'int8_recommended': int8_ok,
        'shipped': det_ship_kind,
        'ship_policy': DET_TFLITE_SHIP,
        'parity_conf': det_parity_conf,
        'conf_thresh': det_conf_thresh,
        'conf_source': det_conf_source,
        'pr_pt': val_pt['pr'] if det_eval else None,
        'int8_calib_images': det_calib_n,
        'int8_error': det_int8_error,
        'nms_iou': DET_NMS_IOU,
        'small_box_area': DET_SMALL_BOX_AREA,
        'export_path': det_export_path,
        'export_versions': det_versions,
        'det_eval': det_eval,
    }
    import gc
    gc.collect()


## 16. Group C — TFLite INT8 + `rice-v3-mobile-release`


In [ ]:
if not IS_C:
    print('Skip Group C TFLite — TRAINING_GROUP != C')
else:
    import zipfile

    assert os.path.isfile(CLASSIFIER_BEST_PATH), CLASSIFIER_BEST_PATH
    cls_model = tf.keras.models.load_model(CLASSIFIER_BEST_PATH, compile=False)
    TEMPERATURE = 1.0
    print('Temperature:', TEMPERATURE, 'reject_threshold:', REJECT_THRESHOLD)

    CALIB_PER_CLASS = 25

    def rep_dataset():
        root = None
        if 'train_dir' in globals() and isinstance(train_dir, str) and os.path.isdir(train_dir):
            root = train_dir
        else:
            for split in ('Training data', 'train'):
                p = os.path.join(CLS_ROOT, split)
                if os.path.isdir(p):
                    root = p
                    break
        assert root, 'No INT8 calibration images: train_dir and CLS_ROOT train split missing'
        rng = random.Random(SEED)
        files = []
        for cls_dir in sorted(Path(root).iterdir()):
            if not cls_dir.is_dir():
                continue
            imgs = sorted(f for f in cls_dir.rglob('*') if f.suffix.lower() in IMAGE_EXTENSIONS)
            rng.shuffle(imgs)
            files.extend(imgs[:CALIB_PER_CLASS])
        rng.shuffle(files)
        print(f'INT8 calibration: {len(files)} images from {root} (<= {CALIB_PER_CLASS}/class)')
        for fp in files:
            try:
                img = Image.open(fp).convert('RGB').resize(CLS_IMG_SIZE)
            except Exception:
                continue
            arr = tf.keras.applications.efficientnet.preprocess_input(np.array(img, dtype=np.float32))
            yield [np.expand_dims(arr, 0)]

    quant_tag = 'int8'
    try:
        converter = tf.lite.TFLiteConverter.from_keras_model(cls_model)
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        converter.representative_dataset = rep_dataset
        converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
        converter.inference_input_type = tf.uint8
        converter.inference_output_type = tf.uint8
        tflite_model = converter.convert()
    except Exception as e:
        print('INT8 failed, dynamic-range fallback:', e)
        quant_tag = 'dynamic'
        converter = tf.lite.TFLiteConverter.from_keras_model(cls_model)
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        tflite_model = converter.convert()

    tflite_out = os.path.join(PIPELINE_DIR, 'rice_leaf_cls_int8.tflite')
    os.makedirs(PIPELINE_DIR, exist_ok=True)
    with open(tflite_out, 'wb') as f:
        f.write(tflite_model)
    shutil.copy2(tflite_out, TFLITE_PATH)
    print('TFLite:', tflite_out, 'bytes=', len(tflite_model), 'quant=', quant_tag)

    interpreter = tf.lite.Interpreter(model_path=tflite_out)
    interpreter.allocate_tensors()
    inp = interpreter.get_input_details()[0]
    dummy = np.zeros(inp['shape'], dtype=inp['dtype'])
    t0 = time.time()
    for _ in range(10):
        interpreter.set_tensor(inp['index'], dummy)
        interpreter.invoke()
    latency_ms = (time.time() - t0) / 10 * 1000
    print(f'TFLite latency smoke: {latency_ms:.1f} ms')

    out_det = interpreter.get_output_details()[0]
    in_scale, in_zp = inp['quantization']
    out_scale, out_zp = out_det['quantization']

    def tflite_probs(arr):
        x = arr[None]
        if np.issubdtype(inp['dtype'], np.integer) and in_scale:
            info = np.iinfo(inp['dtype'])
            x = np.clip(np.round(x / in_scale + in_zp), info.min, info.max)
        interpreter.set_tensor(inp['index'], x.astype(inp['dtype']))
        interpreter.invoke()
        y = interpreter.get_tensor(out_det['index'])[0].astype(np.float32)
        if np.issubdtype(out_det['dtype'], np.integer) and out_scale:
            y = (y - out_zp) * out_scale
        return y

    def view_of(arr, v):
        if v == 'orig':
            return arr
        if v == 'hflip':
            return np.ascontiguousarray(arr[:, ::-1, :])
        if v == 'vflip':
            return np.ascontiguousarray(arr[::-1, :, :])
        raise ValueError(f'Unknown TTA view: {v}')

    # Same inputs for TFLite and Keras, so the drop measures quantization only.
    tta_views = list(training_report.get('tta_views') or TTA_VIEWS)
    eval_views = ['orig'] + [v for v in tta_views if v != 'orig']
    name_to_idx = {n: i for i, n in enumerate(class_names)}
    hits = {'tflite': 0, 'tflite_tta': 0, 'keras': 0, 'keras_tta': 0}
    n_eval = 0
    for path, true_label in iter_test_images(test_dir):
        if true_label not in name_to_idx:
            continue
        arr = tf.keras.applications.efficientnet.preprocess_input(
            np.array(Image.open(path).convert('RGB').resize(CLS_IMG_SIZE), dtype=np.float32)
        )
        views = [view_of(arr, v) for v in eval_views]
        t_probs = np.stack([tflite_probs(v) for v in views])
        k_probs = cls_model.predict(np.stack(views), verbose=0)
        tta_idx = [eval_views.index(v) for v in tta_views] or [0]
        y = name_to_idx[true_label]
        hits['tflite'] += int(np.argmax(t_probs[0]) == y)
        hits['keras'] += int(np.argmax(k_probs[0]) == y)
        hits['tflite_tta'] += int(np.argmax(t_probs[tta_idx].mean(axis=0)) == y)
        hits['keras_tta'] += int(np.argmax(k_probs[tta_idx].mean(axis=0)) == y)
        n_eval += 1
    assert n_eval, f'No test images matched class_names in {test_dir}'
    tflite_eval = {k: v / n_eval for k, v in hits.items()}
    tflite_acc_drop = tflite_eval['keras_tta'] - tflite_eval['tflite_tta']
    print(f'TFLite vs Keras on {n_eval} Mehedi test images (TTA {tta_views}):')
    print(f"  no-TTA: TFLite {tflite_eval['tflite']:.3f} | Keras {tflite_eval['keras']:.3f}")
    print(f"  TTA:    TFLite {tflite_eval['tflite_tta']:.3f} | Keras {tflite_eval['keras_tta']:.3f} "
          f'| drop {tflite_acc_drop:+.3f}')
    if tflite_acc_drop > 0.02:
        print('WARNING: INT8 TFLite loses more than 0.02 accuracy vs Keras — check calibration')

    ci_path = os.path.join(PIPELINE_DIR, 'class_indices.json')
    if not os.path.isfile(ci_path) and 'class_indices' in globals():
        with open(ci_path, 'w', encoding='utf-8') as f:
            json.dump(class_indices, f, indent=2, ensure_ascii=False)

    pipeline_config = {
        'model_file': 'rice_leaf_cls_int8.tflite',
        'backbone': CLS_BACKBONES[0],
        'preprocess': 'efficientnet',
        'img_size': list(CLS_IMG_SIZE),
        'temperature': TEMPERATURE,
        'reject_threshold': REJECT_THRESHOLD,
        'quantization': quant_tag,
        'on_device': True,
        'yolo_on_device': bool(EXPORT_DET_TFLITE),
        'class_indices_file': 'class_indices.json',
        'latency_ms_kaggle_cpu_smoke': latency_ms,
        # App must average softmax over these views to match the gate metric.
        'tta_views': list(training_report.get('tta_views') or []),
        'gate_passed': bool(training_report.get('b_ready_for_c')),
        'release_channel': 'release' if training_report.get('b_ready_for_c') else 'trial',
        'source_checkpoint': training_report.get('session_b_checkpoint_dir'),
        'gate': training_report.get('gate') or {},
        'tflite_test_acc': tflite_eval['tflite'],
        'tflite_test_acc_tta': tflite_eval['tflite_tta'],
        'keras_test_acc_tta_same_input': tflite_eval['keras_tta'],
        'tflite_acc_drop': tflite_acc_drop,
    }
    pipeline_config['output_rule'] = dict(OUTPUT_RULE)
    if EXPORT_DET_TFLITE:
        pipeline_config['detector'] = det_tflite_info['config']
    with open(os.path.join(PIPELINE_DIR, 'pipeline_config.json'), 'w', encoding='utf-8') as f:
        json.dump(pipeline_config, f, indent=2)

    ext_report = {}
    if os.path.isdir(EXTERNAL_VAL_ROOT):
        counts = {}
        for name in os.listdir(EXTERNAL_VAL_ROOT):
            d = os.path.join(EXTERNAL_VAL_ROOT, name)
            if os.path.isdir(d):
                counts[name] = sum(1 for f in Path(d).rglob('*') if f.suffix.lower() in IMAGE_EXTENSIONS)
        ext_report = {'root': EXTERNAL_VAL_ROOT, 'counts': counts}
        print('External holdout counts:', counts)

    training_report['mobile'] = pipeline_config
    training_report['external_validation'] = ext_report
    with open(REPORT_PATH, 'w', encoding='utf-8') as f:
        json.dump(training_report, f, indent=2, ensure_ascii=False)
    shutil.copy2(REPORT_PATH, os.path.join(PIPELINE_DIR, 'training_report.json'))
    if os.path.isfile(CLASSIFIER_BEST_PATH):
        shutil.copy2(CLASSIFIER_BEST_PATH, os.path.join(PIPELINE_DIR, 'classifier_best.keras'))

    with zipfile.ZipFile(MOBILE_ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zf:
        _zip_files = [
            'rice_leaf_cls_int8.tflite', 'class_indices.json', 'pipeline_config.json',
            'training_report.json', 'classifier_best.keras',
        ]
        if EXPORT_DET_TFLITE:
            _zip_files.append(det_tflite_info['ship_file'])
        for fn in _zip_files:
            fp = os.path.join(PIPELINE_DIR, fn)
            if os.path.isfile(fp):
                zf.write(fp, arcname=fn)
    print('Wrote', MOBILE_ZIP_PATH)
    print('=== UPLOAD CHECKLIST (Group C) ===')
    print('1. Save Version')
    print('2. Download', MOBILE_ZIP_PATH)
    if pipeline_config['release_channel'] == 'trial':
        print(f'3. Upload Dataset {KAGGLE_USER}/rice-v3-mobile-trial (NOT rice-v3-mobile-release)')
    else:
        print(f'3. Upload Dataset {KAGGLE_USER}/rice-v3-mobile-release')
    print('4. Copy into Flutter assets/models/')
